# 服务器完整重算版：训练集与候选集 ESM/SaProt embedding 全部重新计算

这版 notebook 面向你的服务器环境：

- GPU：3090 Ti
- 项目目录：`/home/liuchang/PythonProject2`
- ESM2 模型：`/home/liuchang/esm2_t33_650M`
- SaProt 模型：`/home/liuchang/saprot_650M`
- SaProt 源码：`/home/liuchang/saprot/SaProt`
- foldseek：`~/anaconda3/envs/GFP/bin/foldseek`

核心策略：

1. 训练集使用“突变序列 + 对应 GFP family 的 WT 3Di”。
2. 候选集使用“候选序列 + sfGFP WT 3Di”。
3. 训练集和候选集都重新计算 ESM/SaProt embedding。
4. 物理特征使用 PyMOL 定义的发色团邻域残基重新计算。
5. 从零训练 XGBoost 和物理增强门控神经网络，不加载旧模型权重。
6. 候选 topN 重新 embedding 后，用新模型重评分并导出 top 6。

注意：这版会比较耗时，尤其是 14 万训练集的 ESM/SaProt embedding 重算。


In [1]:
# ==========================================
# 1. 配置区
# ==========================================

from pathlib import Path
from collections import defaultdict
import os
import sys
import re
import math
import random
import subprocess

import numpy as np
import pandas as pd

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# 服务器路径
PROJECT_DIR = Path("/home/liuchang/PythonProject2")
ESM_MODEL_PATH = Path("/home/liuchang/esm2_t33_650M")
SAPROT_MODEL_PATH = Path("/home/liuchang/saprot_650M")
SAPROT_CODE_PATH = Path("/home/liuchang/saprot/SaProt")
FOLDSEEK_BIN = Path("/home/liuchang/anaconda3/envs/GFP/bin/foldseek")

# 原始数据
GFP_XLSX = PROJECT_DIR / "GFP_data.xlsx"
REF_TXT = PROJECT_DIR / "AAseqs of 5 GFP proteins.txt"
EXCLUSION_CSV = PROJECT_DIR / "Exclusion_List.csv"

THERMOMPNN_CODE_PATH = Path("/data/liuchang/ThermoMPNN-main")
THERMOMPNN_STRUCTURE = PROJECT_DIR / "fold_sfgfp_model_0.pdb"

THERMOMPNN_COMMAND_TEMPLATE = (
    "conda run -n deepstab_env python "
    "{code_dir}/analysis/custom_inference.py "
    "--pdb {pdb} "
    "--out_dir {out_dir}"
)
FORCE_RERUN_THERMOMPNN = True


# WT 结构：统一使用 AF 预测 WT 结构生成 3Di
WT_STRUCTURE_FILES = {
    "avGFP": PROJECT_DIR / "fold_avgfp_model_0.cif",
    "amacGFP": PROJECT_DIR / "fold_amacgfp_model_0.cif",
    "cgreGFP": PROJECT_DIR / "fold_cgregfp_model_0.cif",
    "ppluGFP": PROJECT_DIR / "fold_pplugfp_model_0.cif",
    "sfGFP": PROJECT_DIR / "fold_sfgfp_model_0.cif",
}

# 是否强制重算 embedding。默认 False：只要缓存可用就直接读取，避免刷新 kernel 后重复跑半小时。
RECOMPUTE_TRAIN_EMBEDDINGS = False
RECOMPUTE_CANDIDATE_EMBEDDINGS = False
# 旧版已经生成的 .npy 没有 manifest；True 表示行数匹配时先复用一次，后续重新生成会写 manifest。
TRUST_LEGACY_EMBEDDING_CACHE = False
# 只对候选表排名前 TOP_N_REEMBED 条重新计算 ESM/SaProt embedding。
# 第一次建议用 200 测试流程；正式筛选可改为 1000、5000 或更多。
TOP_N_REEMBED = 1000

# embedding batch size。3090 Ti 可根据显存调大或调小。
ESM_BATCH_SIZE = 16
SAPROT_BATCH_SIZE = 16

# 输出文件夹：本 notebook 新生成的所有文件都放这里，避免和旧文件混在一起。
RUN_OUTPUT_DIR = PROJECT_DIR / "server_full_recompute_outputs"
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 输出文件
TRAIN_ESM_OUT = RUN_OUTPUT_DIR / "recomputed_train_esm2_embeddings.npy"
TRAIN_SAPROT_OUT = RUN_OUTPUT_DIR / "recomputed_train_saprot_embeddings.npy"
TRAIN_FULL_FEATURE_OUT = RUN_OUTPUT_DIR / "recomputed_train_full_features.npy"

# 候选序列由本 notebook 内部重新生成，不读取旧候选 CSV。
CANDIDATE_POOL_OUT = RUN_OUTPUT_DIR / "generated_candidate_pool_internal.csv"
CANDIDATE_ESM_OUT = RUN_OUTPUT_DIR / "recomputed_candidate_esm2_embeddings.npy"
CANDIDATE_SAPROT_OUT = RUN_OUTPUT_DIR / "recomputed_candidate_saprot_embeddings.npy"
CANDIDATE_SCORED_OUT = RUN_OUTPUT_DIR / "recomputed_candidate_scores.csv"
DETAILED_TOP6_OUT = RUN_OUTPUT_DIR / "recomputed_top6_detailed_report.csv"
SUBMISSION_TOP6_OUT = RUN_OUTPUT_DIR / "recomputed_submission_top6.csv"
SINGLE_MUTANT_SCAN_OUT = RUN_OUTPUT_DIR / "sfgfp_pocket_single_mutant_scan.csv"
THERMOMPNN_OUT_DIR = RUN_OUTPUT_DIR / "thermompnn_sfgfp_strict_aligned"
THERMOMPNN_RAW_OUT = THERMOMPNN_OUT_DIR / "thermompnn_single_mutation_scan.csv"
THERMO_STABILIZING_OUT = RUN_OUTPUT_DIR / "thermompnn_stabilizing_mutations_used_strict_aligned.csv"
XGB_RAW_MODEL_OUT = RUN_OUTPUT_DIR / "xgb_raw_brightness.json"
XGB_RANK_MODEL_OUT = RUN_OUTPUT_DIR / "xgb_family_rank_brightness.json"
GATING_RANK_MODEL_OUT = RUN_OUTPUT_DIR / "recomputed_gating_nn_family_rank.pth"
RANK_ENSEMBLE_MODEL_OUT = RUN_OUTPUT_DIR / "brightness_rank_ensemble_weight_model.joblib"
FINAL_BRIGHTNESS_META_MODEL_OUT = RUN_OUTPUT_DIR / "final_brightness_meta_model_data_driven.joblib"
POCKET_FEATURE_NAMES_OUT = RUN_OUTPUT_DIR / "pocket_physics_68_feature_names.csv"
SFGFP_ALIGNMENT_MAPPING_OUT = RUN_OUTPUT_DIR / "sfgfp_aligned_pocket_position_mapping.csv"
STRUCTURAL_ALIGNMENT_MAPPING_OUT = RUN_OUTPUT_DIR / "structure_aligned_pocket_mapping_to_sfgfp.csv"
ALIGNED_POCKET_FEATURE_NAMES_OUT = RUN_OUTPUT_DIR / "sfgfp_aligned_pocket_feature_names.csv"
STRUCT_ALIGNED_POCKET_FEATURE_NAMES_OUT = RUN_OUTPUT_DIR / "structure_aligned_pocket_feature_names.csv"
COMBINED_POCKET_FEATURE_NAMES_OUT = RUN_OUTPUT_DIR / "combined_pocket_feature_names.csv"
TRAIN_ALIGNED_POCKET_FEATURE_OUT = RUN_OUTPUT_DIR / "train_sfgfp_aligned_pocket_features.npy"
TRAIN_STRUCT_ALIGNED_POCKET_FEATURE_OUT = RUN_OUTPUT_DIR / "train_structure_aligned_pocket_features.npy"
TRAIN_COMBINED_POCKET_FEATURE_OUT = RUN_OUTPUT_DIR / "train_combined_pocket_features.npy"

# 亮度筛选主指标：raw brightness 用 sfGFP WT 作锚点；rank score 只作为多模型共识特征。
USE_RANK_BRIGHTNESS_MODEL = False
# 候选生成参数。
# TOP_N_REEMBED 控制最终进入 ESM2/SaProt 重算的候选数量，不是候选池总数。
MAX_CANDIDATE_MUTATIONS = 6
MAX_MUTATIONS_FROM_BRIGHTNESS = 40
MAX_MUTATIONS_FROM_THERMO = 60
MAX_GENERATED_CANDIDATES = 5000

# 是否优先围绕发光基团邻域残基设计突变。
# True: 候选池主要来自 sfGFP pocket 位点，同时保留少量非 pocket 稳定化突变作为折叠稳定性补偿。
# False: 不限制突变必须接近发光基团。
USE_POCKET_GUIDED_MUTATIONS = True
POCKET_MUTATION_ONLY = False
MAX_NONPOCKET_THERMO_MUTATIONS = 15

# 比赛导向优化：
# 1. 先生成 sfGFP pocket 单点饱和突变，避免只依赖跨家族迁移。
# 2. ThermoMPNN 作为 hard filter，明显不稳定的组合不进入最终 top6。
# 3. top6 按亮度型/稳定型/平衡型分策略输出，增加 Top-1 命中概率。
INCLUDE_POCKET_SATURATION = True
THERMO_DDG_HARD_MAX = 0.0
TOP6_MIN_HAMMING = 2
# 新策略：先用 ThermoMPNN 选稳定性 scaffold，再叠加发光基团邻域亮度突变。
STABILITY_FIRST_CANDIDATES = False
MAX_STABILITY_SCAFFOLD_MUTS = 2
MAX_BRIGHTNESS_AFTER_STABILITY_MUTS = 2
STABILITY_SCAFFOLD_DDG_MAX = -0.3
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# 只保护发色团核心和少数高风险关键位点。
# 注意：不再保护 61-64、68-69 等 pocket 环境残基，否则无法做发光基团邻域优化。
CHROMOPHORE_POSITIONS_0IDX = {64, 65, 66}  # sfGFP 65-67
PROTECTED_POSITIONS_0IDX = CHROMOPHORE_POSITIONS_0IDX | {95, 221}

# 亮度优先 GA 参数
GA_BRIGHTNESS_FIRST = True
GA_TOP_SINGLE_MUTATIONS = 60
GA_POP_SIZE = 220
GA_GENERATIONS = 10
GA_ELITE_SIZE = 40
GA_MAX_MUTATIONS = 6
GA_MUTATION_RATE = 0.35
GA_RANDOM_SEED = 42
GA_AVOID_CYS = True

# sfGFP WT anchored design constraints.
# 往年优秀序列通常是少量 pocket 调亮 + 多个 scaffold 稳定化，而不是 4 个 pocket 同时堆叠。
MIN_TOTAL_MUTATIONS_PER_CANDIDATE = 3
MAX_TOTAL_MUTATIONS_PER_CANDIDATE = 6
MIN_POCKET_MUTATIONS_PER_CANDIDATE = 1
MAX_POCKET_MUTATIONS_PER_CANDIDATE = 2
MIN_SCAFFOLD_MUTATIONS_PER_CANDIDATE = 1
MAX_SCAFFOLD_MUTATIONS_PER_CANDIDATE = 4
GA_TOP_SCAFFOLD_THERMO_MUTATIONS = 200

# 最终筛选时使用 sfGFP WT 作锚点。最终排序分数由验证集数据学习，不再手工指定 raw/rank/thermo 权重。
MIN_XGB_RAW_RELATIVE_TO_WT = 1.0
MIN_BRIGHTNESS_RESCORE_FOR_TOP = 0.45
MAX_RELATIVE_UNCERTAINTY_FOR_TOP = 0.25
FINAL_SELECTION_SCORE_COL = "Final_Learned_Score"

# 验证集用途分两层：
# 1. 主验证集用于训练 stacking/final score，所以默认使用每个 family 内部分层抽样。
# 2. leave-one-family-out 只作为风险审计，不再作为主验证集，否则会把 final score 学偏。
# 可选："within_family_stratified", "groupkfold", "leave_one_family_out"。
VALIDATION_SPLIT_MODE = "within_family_stratified"
VALIDATION_FRACTION = 0.15
VALIDATION_HOLDOUT_FAMILY = "avGFP"
RUN_LEAVE_ONE_FAMILY_OUT_DIAGNOSTIC = True
LOFO_XGB_ESTIMATORS = 600
XGB_EARLY_STOPPING_ROUNDS = 100
LOFO_DIAGNOSTIC_OUT = RUN_OUTPUT_DIR / "leave_one_family_out_diagnostic.csv"


# Delta feature switches: all brightness submodels use own-family WT anchors.
MODEL_FEATURE_USE_DELTA = True
GATING_USE_DELTA = True
LIGHTWEIGHT_USE_DELTA = True
DELTA_FEATURE_SUMMARY_OUT = RUN_OUTPUT_DIR / "all_model_delta_feature_summary.csv"


# 全模型跨 family 泛化审计与基于泛化能力的集成。
RUN_LOFO_SUBMODEL_AUDIT = True
RUN_LOFO_FULL = True
LOFO_GNN_EPOCHS = 5
LOFO_GNN_MAX_TRAIN_ROWS = 50000
LOFO_SUBMODEL_METRICS_OUT = RUN_OUTPUT_DIR / "lofo_submodel_metrics.csv"
LOFO_SUBMODEL_SUMMARY_OUT = RUN_OUTPUT_DIR / "lofo_submodel_summary.csv"
CROSSFAMILY_MODEL_WEIGHTS_OUT = RUN_OUTPUT_DIR / "crossfamily_model_weights.csv"
CROSSFAMILY_WEIGHT_MODE = "proportional_positive_score"
CROSSFAMILY_ENSEMBLE_SCORE_COL = "CrossFamily_EnsembleScore"
REFIT_XGB_ON_ALL_DATA_AFTER_VALIDATION = True
REFIT_GATING_ON_ALL_DATA_AFTER_VALIDATION = True
FINAL_GATING_REFIT_EPOCHS = 10
MODEL_TRUST_AUDIT_OUT = RUN_OUTPUT_DIR / "model_trust_audit_summary.csv"


# Stop-codon / nonsense mutations are truncated proteins, not full-length GFP variants.
EXCLUDE_STOP_CODON_MUTATIONS = True
STOP_CODON_MUTATION_ROWS_OUT = RUN_OUTPUT_DIR / "stop_codon_mutation_rows.csv"
STOP_CODON_MUTATION_SUMMARY_OUT = RUN_OUTPUT_DIR / "stop_codon_mutation_summary.csv"

# Family-subset comparison: keep the main workflow unchanged, but compare these training sets.
RUN_FAMILY_SUBSET_COMPARISON = True
FAMILY_SUBSET_COMPARISON_MODES = ["all", "exclude_pplu", "exclude_pplu_cgre"]
FAMILY_SUBSET_COMPARISON_OUT = RUN_OUTPUT_DIR / "family_subset_model_comparison.csv"
FAMILY_SUBSET_LOFO_OUT = RUN_OUTPUT_DIR / "family_subset_lofo_comparison.csv"
FAMILY_SUBSET_XGB_ESTIMATORS = 800


# Mutation numbering normalization.
# Internal coordinate system is the current reference FASTA/AAseqs coordinate:
# FASTA 1-based M=1, Python pos0=FASTA_position-1.
# Training aaMutations are known to be shifted by -1 relative to the reference,
# so we add +1 to convert them into the internal standard coordinate.
TRAIN_MUTATION_POSITION_OFFSET = 1
TRAIN_MUTATION_NUMBERING_REPORT_OUT = RUN_OUTPUT_DIR / "train_mutation_numbering_standardization_report.csv"
TRAIN_MUTATION_STANDARDIZED_OUT = RUN_OUTPUT_DIR / "train_mutations_standardized.csv"
MUTATION_NUMBERING_STANDARDIZATION_VERSION = "training_aaMutations_offset_plus_1_to_AAseqs_FASTA"

# Gating NN training stabilization
GATING_LR = 3e-4
GATING_MAX_EPOCHS = 30
GATING_WARMUP_EPOCHS = 3
GATING_EARLY_STOP_PATIENCE = 5
GATING_GRAD_CLIP_NORM = 1.0
GATING_DROPOUT = 0.30
GATING_RESIDUAL_SCALE = 0.20

# GA exploration: small probability to inject conservative scaffold mutations outside the closed pools.
GA_EXPLORATION_RATE = 0.08
CONSERVATIVE_AA_GROUPS = ["LIVM", "ST", "DE", "NQ", "KR", "FY", "AG"]

# 多模型漏斗与最终 top6 多样性约束
LIGHT_MODEL_WEIGHT = 0.25
HEAVY_MODEL_WEIGHT = 0.75
CONSENSUS_RAW_RELATIVE_MIN = 1.0
CONSENSUS_RANK_SCORE_MIN = 0.50
CONSENSUS_LIGHT_SCORE_MIN = 0.50
CONSENSUS_THERMO_DDG_MAX = 0.50
CONSENSUS_UNCERTAINTY_MAX = 0.25
MIN_MODEL_CONSENSUS_COUNT = 4
MAX_H148Y_IN_TOP6 = 3
MAX_SAME_POCKET_PATTERN_IN_TOP6 = 2

print("PROJECT_DIR:", PROJECT_DIR)
print("ESM_MODEL_PATH:", ESM_MODEL_PATH)
print("SAPROT_MODEL_PATH:", SAPROT_MODEL_PATH)
print("foldseek:", FOLDSEEK_BIN)


PROJECT_DIR: /home/liuchang/PythonProject2
ESM_MODEL_PATH: /home/liuchang/esm2_t33_650M
SAPROT_MODEL_PATH: /home/liuchang/saprot_650M
foldseek: /home/liuchang/anaconda3/envs/GFP/bin/foldseek


In [2]:
# ==========================================
# 2. 读取原始数据、参考序列、PyMOL pocket residues
# ==========================================

def parse_reference_sequences(path):
    refs = {}
    current = None
    for raw in Path(path).read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith(">"):
            current = line[1:].strip()
            refs[current] = ""
        elif current and re.fullmatch(r"[A-Z]+", line):
            if not refs[current]:
                refs[current] = line
    return refs

refs = parse_reference_sequences(REF_TXT)
SF_GFP = refs["sfGFP"]
print("reference lengths:", {k: len(v) for k, v in refs.items()})

brightness_df = pd.read_excel(GFP_XLSX, sheet_name="brightness")
brightness_df = brightness_df.rename(columns={
    "aaMutations": "mutations",
    "GFP type": "reference",
    "Brightness": "value",
})
brightness_df["mutations"] = brightness_df["mutations"].fillna("WT").astype(str)
brightness_df["reference"] = brightness_df["reference"].astype(str)
brightness_df["value"] = pd.to_numeric(brightness_df["value"], errors="coerce")
brightness_df = brightness_df.dropna(subset=["value"]).reset_index(drop=True)

# Stop-codon mutations contain '*'. They translate to truncated proteins, so they are
# excluded from the full-length GFP brightness/stability model and saved for audit.
brightness_df["has_stop_codon_mutation"] = brightness_df["mutations"].str.contains("*", regex=False, na=False)
stop_codon_df = brightness_df[brightness_df["has_stop_codon_mutation"]].copy()
stop_summary = (
    stop_codon_df.groupby("reference", dropna=False)
    .size()
    .reset_index(name="stop_codon_rows")
    if len(stop_codon_df)
    else pd.DataFrame(columns=["reference", "stop_codon_rows"])
)
stop_summary["total_stop_codon_rows"] = int(len(stop_codon_df))
stop_codon_df.to_csv(STOP_CODON_MUTATION_ROWS_OUT, index=False)
stop_summary.to_csv(STOP_CODON_MUTATION_SUMMARY_OUT, index=False)
print("stop-codon mutation rows:", len(stop_codon_df))
print("stop-codon audit saved:", STOP_CODON_MUTATION_ROWS_OUT)
display(stop_summary)

if EXCLUDE_STOP_CODON_MUTATIONS:
    before_stop_filter = len(brightness_df)
    brightness_df = brightness_df[~brightness_df["has_stop_codon_mutation"]].copy().reset_index(drop=True)
    print("excluded stop-codon mutations:", before_stop_filter - len(brightness_df), "remaining:", len(brightness_df))

before_top_df = pd.read_excel(GFP_XLSX, sheet_name="beforetopseqs")

AA3_TO_1 = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
}
AA_HYDRO = {"A":1.8,"C":2.5,"D":-3.5,"E":-3.5,"F":2.8,"G":-0.4,"H":-3.2,"I":4.5,"K":-3.9,"L":3.8,"M":1.9,"N":-3.5,"P":-1.6,"Q":-3.5,"R":-4.5,"S":-0.8,"T":-0.7,"V":4.2,"W":-0.9,"Y":-1.3}
AA_VOL = {"A":88.6,"C":108.5,"D":111.1,"E":138.4,"F":189.9,"G":60.1,"H":153.2,"I":166.7,"K":168.6,"L":166.7,"M":162.9,"N":114.1,"P":112.7,"Q":143.8,"R":173.4,"S":89.0,"T":116.1,"V":140.0,"W":227.8,"Y":193.6}
AA_CHARGE = {"D":-1,"E":-1,"K":1,"R":1,"H":0.1}
POS_AA = set("KRH")
NEG_AA = set("DE")
POLAR_AA = set("STNQYC")
AROMATIC_AA = set("FYW")

PYMOL_POCKET_RESIDUES_RAW = {
    "avGFP": [("LEU",42),("LEU",60),("VAL",61),("THR",62),("THR",63),("LEU",64),("VAL",68),("GLN",69),("VAL",112),("ASN",121),("ASN",146),("SER",147),("HIS",148),("ASN",149),("VAL",150),("THR",167),("THR",203),("GLN",204),("SER",205),("GLU",222)],
    "amacGFP": [("VAL",61),("THR",62),("THR",63),("LEU",64),("ILE",68),("LEU",69),("VAL",112),("ASN",121),("SER",147),("SER",147),("HIS",148),("ASN",149),("VAL",150),("CYS",203),("CYS",203),("GLN",204),("GLU",222)],
    "cgreGFP": [("THR",19),("GLU",20),("LEU",21),("ILE",32),("GLY",34),("TYR",49),("THR",62),("ILE",63),("LEU",64),("SER",65),("SER",66),("LEU",67),("VAL",71),("VAL",109),("ASN",122),("VAL",124),("VAL",126)],
    "ppluGFP": [("SER",53),("HIS",54),("VAL",55),("MET",56),("PHE",60),("TYR",61),("PHE",101),("GLY",112),("ALA",137),("THR",138),("ARG",156),("ARG",195),("ARG",196),("VAL",197)],
    "sfGFP": [("LEU",42),("LEU",44),("VAL",61),("THR",62),("THR",63),("LEU",64),("THR",65),("TYR",66),("GLY",67),("VAL",68),("GLN",69),("VAL",112),("ASN",121),("SER",147),("HIS",148),("VAL",150),("THR",204),("GLN",205),("SER",206),("LEU",221),("GLU",223)],
}
PYMOL_POCKET_RESIDUES_RAW["creGFP"] = PYMOL_POCKET_RESIDUES_RAW["cgreGFP"]

def normalize_family_name(family):
    return "cgreGFP" if str(family) == "creGFP" else str(family)

def deduplicate_residues(residue_list):
    seen = set()
    out = []
    for aa3, pos1 in residue_list:
        if pos1 in seen:
            continue
        seen.add(pos1)
        out.append((AA3_TO_1.get(aa3.upper(), "X"), int(pos1), aa3.upper()))
    return out

PYMOL_POCKET_RESIDUES = {fam: deduplicate_residues(v) for fam, v in PYMOL_POCKET_RESIDUES_RAW.items()}
print("pocket residues:", {k: len(v) for k, v in PYMOL_POCKET_RESIDUES.items()})

# sfGFP 发光基团邻域位点，来自上面 PyMOL 提取结果。
# 65-67 是发色团本身，已经在 PROTECTED_POSITIONS_0IDX 里保护，不参与突变。
SF_GFP_POCKET_POSITIONS_0IDX = {
    pos1 - 1
    for _aa, pos1, _aa3 in PYMOL_POCKET_RESIDUES["sfGFP"]
}
print("sfGFP pocket positions:", sorted(p + 1 for p in SF_GFP_POCKET_POSITIONS_0IDX))
print("data:", brightness_df.shape)


reference lengths: {'sfGFP': 238, 'avGFP': 238, 'amacGFP': 238, 'cgreGFP': 235, 'ppluGFP': 222}
stop-codon mutation rows: 428
stop-codon audit saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/stop_codon_mutation_rows.csv


,reference,stop_codon_rows,total_stop_codon_rows
0,amacGFP,298,428
1,cgreGFP,52,428
2,ppluGFP,78,428


excluded stop-codon mutations: 428 remaining: 141144
pocket residues: {'avGFP': 20, 'amacGFP': 15, 'cgreGFP': 17, 'ppluGFP': 14, 'sfGFP': 21, 'creGFP': 17}
sfGFP pocket positions: [42, 44, 61, 62, 63, 64, 65, 66, 67, 68, 69, 112, 121, 147, 148, 150, 204, 205, 206, 221, 223]
data: (141144, 4)


In [3]:
# ==========================================
# 3. 序列构建、PyMOL pocket 特征、3Di 提取
# ==========================================

def parse_mutation_token(token, position_offset=0):
    """Parse mutation token into the internal FASTA-based 0-index coordinate.

    position_offset is added to the mutation label position before converting to
    Python 0-index. For training-table aaMutations we use +1; for candidates,
    beforetopseqs, PyMOL pocket labels and ThermoMPNN-derived labels we use 0.
    """
    m = re.fullmatch(r"([A-Z])(\d+)([A-Z])", str(token).strip())
    if not m:
        return None
    label_pos = int(m.group(2))
    standard_pos = label_pos + int(position_offset)
    return m.group(1), standard_pos - 1, m.group(3)


def mutation_tuple_to_label(mut):
    """Convert internal 0-index mutation tuple back to standard FASTA 1-based label."""
    wt, pos0, aa = mut
    return f"{wt}{int(pos0) + 1}{aa}"


def split_mutation_string(mut_string, position_offset=0):
    """Split mutation string using a source-specific numbering offset."""
    if pd.isna(mut_string) or str(mut_string).upper() == "WT":
        return []
    toks = [t for t in str(mut_string).replace(";", ":").split(":") if t and t.upper() != "WT"]
    return [p for p in (parse_mutation_token(t, position_offset=position_offset) for t in toks) if p is not None]


def standardize_mutation_string(mut_string, position_offset=0):
    """Return a standard FASTA-coordinate mutation string after applying offset."""
    muts = split_mutation_string(mut_string, position_offset=position_offset)
    return ":".join(mutation_tuple_to_label(m) for m in muts) if muts else "WT"

def reference_for_family(family):
    fam = normalize_family_name(family)
    if fam in refs:
        return refs[fam]
    # 兼容历史数据里 creGFP / cgreGFP 两种写法。
    if fam == "cgreGFP" and "creGFP" in refs:
        return refs["creGFP"]
    if fam == "creGFP" and "cgreGFP" in refs:
        return refs["cgreGFP"]
    raise KeyError(f"Missing reference sequence for {family}; available={list(refs)}")

def apply_mutations(seq, mutations, strict=False):
    seq_list = list(seq)
    for wt, pos0, mut in mutations:
        if pos0 < 0 or pos0 >= len(seq_list):
            if strict:
                raise ValueError(f"Position out of range: {pos0+1}")
            continue
        if strict and seq_list[pos0] != wt:
            raise ValueError(f"WT mismatch at {pos0+1}: ref={seq_list[pos0]} mut_token={wt}")
        seq_list[pos0] = mut
    return "".join(seq_list)

def build_full_sequence(row):
    """Build training full_sequence from standardized mutation coordinates."""
    ref = reference_for_family(row["reference"])
    if "mut_list_standard" in row:
        muts = row["mut_list_standard"]
    elif "mutations_standard" in row:
        muts = split_mutation_string(row["mutations_standard"], position_offset=0)
    else:
        muts = split_mutation_string(row["mutations"], position_offset=TRAIN_MUTATION_POSITION_OFFSET)
    return apply_mutations(ref, muts, strict=False)

def global_bio_3(sequence):
    """全局生物物理 3 维：MW、pI、平均柔性。

    这 3 维保留最初版 68D 物理特征的定义，只作为全局折叠/表达背景，
    亮度相关的局部机制主要由下面的 chromophore-pocket 特征承担。
    """
    seq = str(sequence).strip().upper().replace("X", "A")
    try:
        from Bio.SeqUtils.ProtParam import ProteinAnalysis
        analysed = ProteinAnalysis(seq)
        mw = analysed.molecular_weight() / 1000.0
        pi = analysed.isoelectric_point()
        try:
            flex = analysed.flexibility()
            avg_flex = float(np.mean(flex)) if len(flex) else 0.0
        except Exception:
            avg_flex = 1.0
        return np.array([mw, pi, avg_flex], dtype=np.float32)
    except Exception:
        mw = sum(AA_VOL.get(a, 120.0) for a in seq) / 1000.0
        net_charge = sum(AA_CHARGE.get(a, 0.0) for a in seq)
        mean_hydro = float(np.mean([AA_HYDRO.get(a, 0.0) for a in seq])) if seq else 0.0
        return np.array([mw, 7.0 + 0.15 * np.tanh(net_charge / 10.0), mean_hydro], dtype=np.float32)

# 关键正电/催化相关位点。用于 local_charge_5 里的 critical_switch。
# 这些是 0-indexed；sfGFP/avGFP 的 Arg96 对应 95。
CRITICAL_CHARGE_SITE_0IDX = {
    "avGFP": 95,
    "amacGFP": 95,
    "cgreGFP": 63,
    "creGFP": 63,
    "ppluGFP": 87,
    "sfGFP": 95,
}

POCKET_PHYSICS_MAX_SITES = 20


def pocket_physics_68_feature_names(max_sites=POCKET_PHYSICS_MAX_SITES):
    """给 68 维物理特征生成可解释列名。"""
    names = []
    for i in range(max_sites):
        names.extend([
            f"Pocket_{i + 1:02d}_hydro",
            f"Pocket_{i + 1:02d}_volume",
            f"Pocket_{i + 1:02d}_charge",
        ])
    names.extend([
        "Local_Net_Charge",
        "Local_Positive_Count",
        "Local_Negative_Count",
        "Local_Charge_Density",
        "Critical_Positive_Site_Intact",
        "Global_MW_kDa",
        "Global_pI",
        "Global_Avg_Flexibility",
    ])
    return names

POCKET_FEATURE_NAMES = pocket_physics_68_feature_names()
print("pocket physics feature dim:", len(POCKET_FEATURE_NAMES))
try:
    pd.DataFrame({"feature_index": range(len(POCKET_FEATURE_NAMES)), "feature_name": POCKET_FEATURE_NAMES}).to_csv(
        POCKET_FEATURE_NAMES_OUT,
        index=False,
    )
    print("pocket feature names saved:", POCKET_FEATURE_NAMES_OUT)
except Exception as e:
    print("pocket feature names not saved:", repr(e))


def pymol_pocket_feature_vector(sequence, family, mutations=None, max_sites=POCKET_PHYSICS_MAX_SITES):
    """提取 68 维发色团邻域物理特征。

    设计来源对应最初版 physics_bundle_68.npy：
    - Pocket Physics 60D: 20 个 PyMOL pocket 槽位 x [疏水性, 体积, 电荷]
    - Local Charge 5D: pocket 局部净电荷、正/负电荷数、电荷密度、关键正电位点是否保留
    - Global Bio 3D: MW、pI、平均柔性

    不同 GFP family 的 PyMOL pocket 残基数不同，因此这里用 padding/truncation
    统一成 20 个槽位；真实 pocket 总体性质由 Local Charge 和后续突变计数列辅助解释。
    """
    fam = normalize_family_name(family)
    residues = PYMOL_POCKET_RESIDUES.get(fam, PYMOL_POCKET_RESIDUES["sfGFP"])
    seq = str(sequence).strip().upper()
    mutations = mutations or []

    # A. Pocket Physics 60D：前 20 个 pocket 槽位，不足补零，超过截断。
    pocket_60 = []
    for expected_aa, pos1, _aa3 in residues[:max_sites]:
        pos0 = pos1 - 1
        aa = seq[pos0] if 0 <= pos0 < len(seq) else expected_aa
        pocket_60.extend([
            AA_HYDRO.get(aa, 0.0),
            AA_VOL.get(aa, 120.0) / 200.0,
            AA_CHARGE.get(aa, 0.0),
        ])
    while len(pocket_60) < max_sites * 3:
        pocket_60.extend([0.0, 0.0, 0.0])
    pocket_60 = pocket_60[:max_sites * 3]

    # B. Local Charge 5D：使用该 family 的全部 PyMOL pocket 残基，并额外纳入关键正电位点。
    residue_pos0 = {pos1 - 1 for _expected_aa, pos1, _aa3 in residues}
    crit_idx = CRITICAL_CHARGE_SITE_0IDX.get(fam, CRITICAL_CHARGE_SITE_0IDX["sfGFP"])
    residue_pos0.add(crit_idx)

    local_aas = []
    for pos0 in sorted(residue_pos0):
        if 0 <= pos0 < len(seq):
            local_aas.append(seq[pos0])
    n_local = max(1, len(local_aas))
    pos_count = sum(a in {"R", "K"} for a in local_aas)
    neg_count = sum(a in {"D", "E"} for a in local_aas)
    net_charge = pos_count - neg_count
    charge_density = net_charge / n_local
    critical_switch = 1.0 if (0 <= crit_idx < len(seq) and seq[crit_idx] in {"R", "K"}) else 0.0
    local_charge_5 = [
        float(net_charge),
        float(pos_count),
        float(neg_count),
        float(charge_density),
        float(critical_switch),
    ]

    # C. Global Bio 3D。
    return np.array(pocket_60 + local_charge_5 + list(global_bio_3(seq)), dtype=np.float32)


# ==========================================
# sfGFP 坐标系对齐 pocket 特征
# ==========================================

ALIGNED_AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")
ALIGNED_AA_TO_IDX = {aa: i for i, aa in enumerate(ALIGNED_AA_ORDER)}


def build_pairwise_mapping_to_sfgfp(refs):
    """把每个 GFP family 的 WT 序列对齐到 sfGFP，建立 sfGFP 位点 -> family 位点映射。"""
    from Bio.Align import PairwiseAligner

    aligner = PairwiseAligner()
    aligner.mode = "global"
    aligner.match_score = 2
    aligner.mismatch_score = -1
    aligner.open_gap_score = -10
    aligner.extend_gap_score = -0.5

    sfgfp_seq = refs["sfGFP"]
    sfgfp_to_family = {}
    family_to_sfgfp = {}

    for raw_fam, fam_seq in refs.items():
        fam = normalize_family_name(raw_fam)
        aln = aligner.align(sfgfp_seq, fam_seq)[0]

        sf_to_fam = {}
        fam_to_sf = {}
        for (sf_start, sf_end), (fam_start, fam_end) in zip(aln.aligned[0], aln.aligned[1]):
            for sf_pos, fam_pos in zip(range(int(sf_start), int(sf_end)), range(int(fam_start), int(fam_end))):
                sf_to_fam[int(sf_pos)] = int(fam_pos)
                fam_to_sf[int(fam_pos)] = int(sf_pos)

        sfgfp_to_family[fam] = sf_to_fam
        family_to_sfgfp[fam] = fam_to_sf

    return sfgfp_to_family, family_to_sfgfp


SFGFP_TO_FAMILY_POS, FAMILY_TO_SFGFP_POS = build_pairwise_mapping_to_sfgfp(refs)

# 用 sfGFP 的 PyMOL pocket 位点作为统一坐标系，并额外加入 Arg96 这类关键电荷位点。
SFGFP_ALIGNED_POSITIONS_0IDX = sorted(SF_GFP_POCKET_POSITIONS_0IDX | {95})
print("sfGFP-aligned pocket positions:", [p + 1 for p in SFGFP_ALIGNED_POSITIONS_0IDX])
print("sfGFP-aligned position count:", len(SFGFP_ALIGNED_POSITIONS_0IDX))


def sfgfp_aligned_pocket_feature_names(positions_0idx=SFGFP_ALIGNED_POSITIONS_0IDX):
    """生成 sfGFP 对齐 pocket 特征名。每个位点 28 维。"""
    names = []
    for sf_pos0 in positions_0idx:
        prefix = f"Aligned_sfGFP_{sf_pos0 + 1}"
        names.extend([f"{prefix}_aa_{aa}" for aa in ALIGNED_AA_ORDER])
        names.extend([
            f"{prefix}_hydro",
            f"{prefix}_volume",
            f"{prefix}_charge",
            f"{prefix}_is_mutated_vs_family_WT",
            f"{prefix}_delta_hydro_vs_family_WT",
            f"{prefix}_delta_volume_vs_family_WT",
            f"{prefix}_delta_charge_vs_family_WT",
            f"{prefix}_is_missing_in_family",
        ])
    return names


ALIGNED_POCKET_FEATURE_NAMES = sfgfp_aligned_pocket_feature_names()
print("aligned pocket feature dim:", len(ALIGNED_POCKET_FEATURE_NAMES))

try:
    pd.DataFrame({
        "feature_index": range(len(ALIGNED_POCKET_FEATURE_NAMES)),
        "feature_name": ALIGNED_POCKET_FEATURE_NAMES,
    }).to_csv(ALIGNED_POCKET_FEATURE_NAMES_OUT, index=False)
    print("sequence-aligned feature names saved:", ALIGNED_POCKET_FEATURE_NAMES_OUT)
except Exception as e:
    print("aligned feature names not saved:", repr(e))


def sfgfp_aligned_pocket_feature_vector(sequence, family, positions_0idx=SFGFP_ALIGNED_POSITIONS_0IDX):
    """把任意 GFP family 的序列投影到 sfGFP 坐标系，再提取 key-site 特征。

    这组特征用于缓解跨 family 编号不一致：模型看到的是 sfGFP 坐标 62/148/204 等，
    而不是各 family 自己的原始 index。
    """
    fam = normalize_family_name(family)
    seq = str(sequence).strip().upper()
    ref_seq = reference_for_family(fam)
    sf_to_fam = SFGFP_TO_FAMILY_POS.get(fam, {})

    features = []
    for sf_pos0 in positions_0idx:
        fam_pos0 = sf_to_fam.get(int(sf_pos0), None)

        if fam_pos0 is None or fam_pos0 < 0 or fam_pos0 >= len(seq) or fam_pos0 >= len(ref_seq):
            aa = "X"
            wt = "X"
            is_missing = 1.0
        else:
            aa = seq[fam_pos0]
            wt = ref_seq[fam_pos0]
            is_missing = 0.0

        onehot = [0.0] * len(ALIGNED_AA_ORDER)
        if aa in ALIGNED_AA_TO_IDX:
            onehot[ALIGNED_AA_TO_IDX[aa]] = 1.0

        if is_missing:
            hydro = volume = charge = 0.0
            wt_hydro = wt_volume = wt_charge = 0.0
            is_mutated = 0.0
        else:
            hydro = AA_HYDRO.get(aa, 0.0)
            volume = AA_VOL.get(aa, 120.0) / 200.0
            charge = AA_CHARGE.get(aa, 0.0)
            wt_hydro = AA_HYDRO.get(wt, 0.0)
            wt_volume = AA_VOL.get(wt, 120.0) / 200.0
            wt_charge = AA_CHARGE.get(wt, 0.0)
            is_mutated = 0.0 if aa == wt else 1.0

        features.extend([
            *onehot,
            hydro,
            volume,
            charge,
            is_mutated,
            hydro - wt_hydro,
            volume - wt_volume,
            charge - wt_charge,
            is_missing,
        ])

    return np.array(features, dtype=np.float32)


# 保存序列对齐映射，方便人工检查关键 pocket 位点是否合理。
alignment_rows = []
for fam in sorted(SFGFP_TO_FAMILY_POS):
    ref_seq = reference_for_family(fam)
    mapping = SFGFP_TO_FAMILY_POS[fam]
    for sf_pos0 in SFGFP_ALIGNED_POSITIONS_0IDX:
        fam_pos0 = mapping.get(sf_pos0)
        alignment_rows.append({
            "family": fam,
            "sfGFP_pos": sf_pos0 + 1,
            "sfGFP_aa": SF_GFP[sf_pos0] if sf_pos0 < len(SF_GFP) else None,
            "family_pos": None if fam_pos0 is None else fam_pos0 + 1,
            "family_aa": None if fam_pos0 is None or fam_pos0 >= len(ref_seq) else ref_seq[fam_pos0],
        })
SFGFP_ALIGNMENT_MAPPING_DF = pd.DataFrame(alignment_rows)
SFGFP_ALIGNMENT_MAPPING_DF.to_csv(SFGFP_ALIGNMENT_MAPPING_OUT, index=False)
print("sfGFP sequence alignment mapping saved:", SFGFP_ALIGNMENT_MAPPING_OUT)
display(SFGFP_ALIGNMENT_MAPPING_DF.head(30))


# ==========================================
# 结构对齐 pocket 特征：sfGFP 坐标系 + PyMOL 风格核心叠合
# ==========================================

STRUCTURE_MAPPING_DISTANCE_CUTOFF = 4.0
STRUCTURE_CORE_REJECT_CUTOFF = 3.0
STRUCTURE_CORE_MAX_CYCLES = 5
STRUCTURE_ALIGN_ATOM_NAME = "CA"


def _aa3_to_aa1(resname):
    """三字母氨基酸名转一字母；兼容常见非标准残基失败情况。"""
    try:
        from Bio.Data.PDBData import protein_letters_3to1_extended
        return protein_letters_3to1_extended.get(str(resname).upper(), "X")
    except Exception:
        fallback = {
            "ALA": "A", "CYS": "C", "ASP": "D", "GLU": "E", "PHE": "F",
            "GLY": "G", "HIS": "H", "ILE": "I", "LYS": "K", "LEU": "L",
            "MET": "M", "ASN": "N", "PRO": "P", "GLN": "Q", "ARG": "R",
            "SER": "S", "THR": "T", "VAL": "V", "TRP": "W", "TYR": "Y",
        }
        return fallback.get(str(resname).upper(), "X")


def _load_structure_for_alignment(structure_path, structure_id):
    """读取 PDB/mmCIF 结构。结构整体方向、原点不同没关系，后面会自动旋转平移。"""
    from Bio.PDB import MMCIFParser, PDBParser
    structure_path = Path(structure_path)
    parser = MMCIFParser(QUIET=True) if structure_path.suffix.lower() in {".cif", ".mmcif"} else PDBParser(QUIET=True)
    return parser.get_structure(structure_id, str(structure_path))


def _extract_ca_records(structure):
    """选择 CA 原子最多的一条链，提取按结构顺序排列的残基记录。"""
    best_records = []
    for model in structure:
        for chain in model:
            records = []
            for residue in chain:
                if residue.id[0] != " " or STRUCTURE_ALIGN_ATOM_NAME not in residue:
                    continue
                aa = _aa3_to_aa1(residue.get_resname())
                if aa == "X":
                    continue
                records.append({
                    "chain_id": chain.id,
                    "residue": residue,
                    "aa": aa,
                    "resnum": int(residue.id[1]),
                    "icode": str(residue.id[2]).strip(),
                    "coord": residue[STRUCTURE_ALIGN_ATOM_NAME].get_coord().astype(float),
                })
            if len(records) > len(best_records):
                best_records = records
        break

    if not best_records:
        raise ValueError("结构中没有可用的标准氨基酸 CA 原子")

    for i, rec in enumerate(best_records):
        rec["pos0"] = i
        rec["pos1"] = i + 1
    return best_records


def _records_to_sequence(records):
    return "".join(rec["aa"] for rec in records)


def _pairwise_index_pairs(ref_seq, mov_seq):
    """先用序列对齐找初始 CA 对，后面再用结构距离剔除 outlier。"""
    from Bio.Align import PairwiseAligner
    aligner = PairwiseAligner()
    aligner.mode = "global"
    aligner.match_score = 2.0
    aligner.mismatch_score = -1.0
    aligner.open_gap_score = -10.0
    aligner.extend_gap_score = -0.5
    alignment = aligner.align(ref_seq, mov_seq)[0]

    pairs = []
    for ref_block, mov_block in zip(alignment.aligned[0], alignment.aligned[1]):
        ref_start, ref_end = map(int, ref_block)
        mov_start, mov_end = map(int, mov_block)
        for k in range(min(ref_end - ref_start, mov_end - mov_start)):
            pairs.append((ref_start + k, mov_start + k))
    return pairs


def _superpose_with_outlier_rejection(ref_records, mov_records, initial_pairs):
    """模仿 PyMOL super/align：反复剔除距离过远的 CA 对，得到核心 barrel RMSD。"""
    from Bio.PDB import Superimposer

    active_pairs = list(initial_pairs)
    if len(active_pairs) < 20:
        raise ValueError(f"可用于结构叠合的 CA 对太少: {len(active_pairs)}")

    sup = Superimposer()
    last_rms = np.nan
    for _cycle in range(STRUCTURE_CORE_MAX_CYCLES):
        fixed_atoms = [ref_records[i]["residue"][STRUCTURE_ALIGN_ATOM_NAME] for i, _j in active_pairs]
        moving_atoms = [mov_records[j]["residue"][STRUCTURE_ALIGN_ATOM_NAME] for _i, j in active_pairs]
        sup.set_atoms(fixed_atoms, moving_atoms)
        last_rms = float(sup.rms)

        rot, tran = sup.rotran
        keep_pairs = []
        for i, j in active_pairs:
            mov_coord = np.dot(mov_records[j]["coord"], rot) + tran
            dist = float(np.linalg.norm(ref_records[i]["coord"] - mov_coord))
            if dist <= STRUCTURE_CORE_REJECT_CUTOFF:
                keep_pairs.append((i, j))

        if len(keep_pairs) < 20 or len(keep_pairs) == len(active_pairs):
            break
        active_pairs = keep_pairs

    fixed_atoms = [ref_records[i]["residue"][STRUCTURE_ALIGN_ATOM_NAME] for i, _j in active_pairs]
    moving_atoms = [mov_records[j]["residue"][STRUCTURE_ALIGN_ATOM_NAME] for _i, j in active_pairs]
    sup.set_atoms(fixed_atoms, moving_atoms)
    return sup, active_pairs, float(sup.rms if not np.isnan(sup.rms) else last_rms)


def build_structure_mapping_to_sfgfp():
    """用 WT 结构建立 sfGFP pocket 位点到各 family 空间对应位点的映射表。"""
    rows = []
    ref_structure = _load_structure_for_alignment(WT_STRUCTURE_FILES["sfGFP"], "sfGFP")
    ref_records = _extract_ca_records(ref_structure)
    ref_seq = _records_to_sequence(ref_records)

    for fam, path in WT_STRUCTURE_FILES.items():
        fam = normalize_family_name(fam)
        if fam == "sfGFP":
            for sf_pos0 in SFGFP_ALIGNED_POSITIONS_0IDX:
                rec = ref_records[sf_pos0] if sf_pos0 < len(ref_records) else None
                rows.append({
                    "family": fam,
                    "sfgfp_pos0": sf_pos0,
                    "sfgfp_pos1": sf_pos0 + 1,
                    "sfgfp_aa": None if rec is None else rec["aa"],
                    "sfgfp_pdb_resnum": None if rec is None else rec["resnum"],
                    "mapped_pos0": sf_pos0 if rec is not None else np.nan,
                    "mapped_pos1": sf_pos0 + 1 if rec is not None else np.nan,
                    "mapped_pdb_resnum": None if rec is None else rec["resnum"],
                    "mapped_chain": None if rec is None else rec["chain_id"],
                    "mapped_aa": None if rec is None else rec["aa"],
                    "ca_distance": 0.0 if rec is not None else np.nan,
                    "core_rmsd_ca": 0.0,
                    "core_aligned_atoms": len(ref_records),
                    "initial_aligned_atoms": len(ref_records),
                    "mapping_confidence": "self" if rec is not None else "missing",
                    "is_duplicate_struct_mapping": 0.0,
                })
            continue

        try:
            mov_structure = _load_structure_for_alignment(path, fam)
            mov_records = _extract_ca_records(mov_structure)
            mov_seq = _records_to_sequence(mov_records)
            initial_pairs = _pairwise_index_pairs(ref_seq, mov_seq)
            sup, core_pairs, core_rmsd = _superpose_with_outlier_rejection(ref_records, mov_records, initial_pairs)

            rot, tran = sup.rotran
            mov_aligned_coords = np.vstack([np.dot(rec["coord"], rot) + tran for rec in mov_records])

            mapped_pos_values = []
            tmp_rows = []
            for sf_pos0 in SFGFP_ALIGNED_POSITIONS_0IDX:
                if sf_pos0 >= len(ref_records):
                    tmp_rows.append({
                        "family": fam,
                        "sfgfp_pos0": sf_pos0,
                        "sfgfp_pos1": sf_pos0 + 1,
                        "sfgfp_aa": None,
                        "sfgfp_pdb_resnum": None,
                        "mapped_pos0": np.nan,
                        "mapped_pos1": np.nan,
                        "mapped_pdb_resnum": None,
                        "mapped_chain": None,
                        "mapped_aa": None,
                        "ca_distance": np.nan,
                        "core_rmsd_ca": core_rmsd,
                        "core_aligned_atoms": len(core_pairs),
                        "initial_aligned_atoms": len(initial_pairs),
                        "mapping_confidence": "missing_sfgfp_site",
                    })
                    continue

                dists = np.linalg.norm(mov_aligned_coords - ref_records[sf_pos0]["coord"].reshape(1, 3), axis=1)
                nearest_idx = int(np.argmin(dists))
                nearest_dist = float(dists[nearest_idx])
                nearest_rec = mov_records[nearest_idx]
                mapped_pos_values.append(nearest_idx)

                if nearest_dist <= STRUCTURE_MAPPING_DISTANCE_CUTOFF and core_rmsd <= 2.0:
                    confidence = "high"
                elif nearest_dist <= STRUCTURE_MAPPING_DISTANCE_CUTOFF + 2.0 and core_rmsd <= 4.0:
                    confidence = "medium"
                else:
                    confidence = "low"

                tmp_rows.append({
                    "family": fam,
                    "sfgfp_pos0": sf_pos0,
                    "sfgfp_pos1": sf_pos0 + 1,
                    "sfgfp_aa": ref_records[sf_pos0]["aa"],
                    "sfgfp_pdb_resnum": ref_records[sf_pos0]["resnum"],
                    "mapped_pos0": nearest_idx,
                    "mapped_pos1": nearest_idx + 1,
                    "mapped_pdb_resnum": nearest_rec["resnum"],
                    "mapped_chain": nearest_rec["chain_id"],
                    "mapped_aa": nearest_rec["aa"],
                    "ca_distance": nearest_dist,
                    "core_rmsd_ca": core_rmsd,
                    "core_aligned_atoms": len(core_pairs),
                    "initial_aligned_atoms": len(initial_pairs),
                    "mapping_confidence": confidence,
                })

            counts = pd.Series(mapped_pos_values).value_counts().to_dict()
            for row in tmp_rows:
                mp = row.get("mapped_pos0")
                row["is_duplicate_struct_mapping"] = float(counts.get(int(mp), 0) > 1) if pd.notna(mp) else 0.0
            rows.extend(tmp_rows)

        except Exception as e:
            print(f"结构对齐失败，{fam} 将使用缺失结构特征:", repr(e))
            for sf_pos0 in SFGFP_ALIGNED_POSITIONS_0IDX:
                rows.append({
                    "family": fam,
                    "sfgfp_pos0": sf_pos0,
                    "sfgfp_pos1": sf_pos0 + 1,
                    "sfgfp_aa": SF_GFP[sf_pos0] if sf_pos0 < len(SF_GFP) else None,
                    "sfgfp_pdb_resnum": None,
                    "mapped_pos0": np.nan,
                    "mapped_pos1": np.nan,
                    "mapped_pdb_resnum": None,
                    "mapped_chain": None,
                    "mapped_aa": None,
                    "ca_distance": np.nan,
                    "core_rmsd_ca": np.nan,
                    "core_aligned_atoms": 0,
                    "initial_aligned_atoms": 0,
                    "mapping_confidence": "failed",
                    "is_duplicate_struct_mapping": 0.0,
                })

    return pd.DataFrame(rows)


STRUCTURE_ALIGNMENT_MAPPING_DF = build_structure_mapping_to_sfgfp()
STRUCTURE_ALIGNMENT_MAPPING_DF.to_csv(STRUCTURAL_ALIGNMENT_MAPPING_OUT, index=False)
print("structure alignment mapping saved:", STRUCTURAL_ALIGNMENT_MAPPING_OUT)
display(STRUCTURE_ALIGNMENT_MAPPING_DF.head(40))
print(pd.crosstab(STRUCTURE_ALIGNMENT_MAPPING_DF["family"], STRUCTURE_ALIGNMENT_MAPPING_DF["mapping_confidence"]))

STRUCT_SFGFP_TO_FAMILY_POS = {}
STRUCT_MAPPING_INFO = {}
for row in STRUCTURE_ALIGNMENT_MAPPING_DF.to_dict("records"):
    fam = normalize_family_name(row["family"])
    sf_pos0 = int(row["sfgfp_pos0"])
    mapped_pos0 = None if pd.isna(row.get("mapped_pos0")) else int(row["mapped_pos0"])
    STRUCT_SFGFP_TO_FAMILY_POS.setdefault(fam, {})[sf_pos0] = mapped_pos0
    STRUCT_MAPPING_INFO[(fam, sf_pos0)] = row


def structure_aligned_pocket_feature_names(positions_0idx=SFGFP_ALIGNED_POSITIONS_0IDX):
    """生成结构对齐 pocket 特征名。每个位点 34 维。"""
    names = []
    for sf_pos0 in positions_0idx:
        prefix = f"Struct_sfGFP_{sf_pos0 + 1}"
        names.extend([f"{prefix}_aa_{aa}" for aa in ALIGNED_AA_ORDER])
        names.extend([
            f"{prefix}_hydro",
            f"{prefix}_volume",
            f"{prefix}_charge",
            f"{prefix}_is_mutated_vs_family_WT",
            f"{prefix}_delta_hydro_vs_family_WT",
            f"{prefix}_delta_volume_vs_family_WT",
            f"{prefix}_delta_charge_vs_family_WT",
            f"{prefix}_is_missing_in_family",
            f"{prefix}_ca_distance_scaled",
            f"{prefix}_core_rmsd_scaled",
            f"{prefix}_mapping_confidence_score",
            f"{prefix}_seq_struct_mapping_agree",
            f"{prefix}_duplicate_struct_mapping",
            f"{prefix}_pos_shift_scaled",
        ])
    return names


def _mapping_confidence_score(label):
    return {
        "self": 1.0,
        "high": 1.0,
        "medium": 0.5,
        "low": 0.1,
        "failed": 0.0,
        "missing": 0.0,
        "missing_sfgfp_site": 0.0,
    }.get(str(label), 0.0)


def structure_aligned_pocket_feature_vector(sequence, family, positions_0idx=SFGFP_ALIGNED_POSITIONS_0IDX):
    """基于 WT 结构叠合映射，把任意序列投影到 sfGFP pocket 空间位点。"""
    fam = normalize_family_name(family)
    seq = str(sequence).strip().upper()
    ref_seq = reference_for_family(fam)
    struct_map = STRUCT_SFGFP_TO_FAMILY_POS.get(fam, {})
    seq_map = SFGFP_TO_FAMILY_POS.get(fam, {})

    features = []
    for sf_pos0 in positions_0idx:
        sf_pos0 = int(sf_pos0)
        fam_pos0 = struct_map.get(sf_pos0, None)
        info = STRUCT_MAPPING_INFO.get((fam, sf_pos0), {})
        seq_pos0 = seq_map.get(sf_pos0, None)

        if fam_pos0 is None or fam_pos0 < 0 or fam_pos0 >= len(seq) or fam_pos0 >= len(ref_seq):
            aa = "X"
            wt = "X"
            is_missing = 1.0
        else:
            aa = seq[fam_pos0]
            wt = ref_seq[fam_pos0]
            is_missing = 0.0

        onehot = [0.0] * len(ALIGNED_AA_ORDER)
        if aa in ALIGNED_AA_TO_IDX:
            onehot[ALIGNED_AA_TO_IDX[aa]] = 1.0

        if is_missing:
            hydro = volume = charge = 0.0
            wt_hydro = wt_volume = wt_charge = 0.0
            is_mutated = 0.0
        else:
            hydro = AA_HYDRO.get(aa, 0.0)
            volume = AA_VOL.get(aa, 120.0) / 200.0
            charge = AA_CHARGE.get(aa, 0.0)
            wt_hydro = AA_HYDRO.get(wt, 0.0)
            wt_volume = AA_VOL.get(wt, 120.0) / 200.0
            wt_charge = AA_CHARGE.get(wt, 0.0)
            is_mutated = 0.0 if aa == wt else 1.0

        ca_dist = info.get("ca_distance", np.nan)
        core_rmsd = info.get("core_rmsd_ca", np.nan)
        ca_distance_scaled = 0.0 if pd.isna(ca_dist) else min(float(ca_dist), 10.0) / 10.0
        core_rmsd_scaled = 0.0 if pd.isna(core_rmsd) else min(float(core_rmsd), 10.0) / 10.0
        confidence_score = _mapping_confidence_score(info.get("mapping_confidence", "failed"))
        seq_struct_agree = 1.0 if (seq_pos0 is not None and fam_pos0 is not None and int(seq_pos0) == int(fam_pos0)) else 0.0
        duplicate_mapping = float(info.get("is_duplicate_struct_mapping", 0.0) or 0.0)
        pos_shift_scaled = 0.0 if fam_pos0 is None else float(int(fam_pos0) - int(sf_pos0)) / 20.0

        features.extend([
            *onehot,
            hydro,
            volume,
            charge,
            is_mutated,
            hydro - wt_hydro,
            volume - wt_volume,
            charge - wt_charge,
            is_missing,
            ca_distance_scaled,
            core_rmsd_scaled,
            confidence_score,
            seq_struct_agree,
            duplicate_mapping,
            pos_shift_scaled,
        ])

    return np.array(features, dtype=np.float32)


STRUCT_ALIGNED_POCKET_FEATURE_NAMES = structure_aligned_pocket_feature_names()
COMBINED_POCKET_FEATURE_NAMES = POCKET_FEATURE_NAMES + ALIGNED_POCKET_FEATURE_NAMES + STRUCT_ALIGNED_POCKET_FEATURE_NAMES
print("structure-aligned pocket feature dim:", len(STRUCT_ALIGNED_POCKET_FEATURE_NAMES))
print("combined pocket feature dim:", len(COMBINED_POCKET_FEATURE_NAMES))

try:
    pd.DataFrame({
        "feature_index": range(len(STRUCT_ALIGNED_POCKET_FEATURE_NAMES)),
        "feature_name": STRUCT_ALIGNED_POCKET_FEATURE_NAMES,
    }).to_csv(STRUCT_ALIGNED_POCKET_FEATURE_NAMES_OUT, index=False)
    pd.DataFrame({
        "feature_index": range(len(COMBINED_POCKET_FEATURE_NAMES)),
        "feature_name": COMBINED_POCKET_FEATURE_NAMES,
    }).to_csv(COMBINED_POCKET_FEATURE_NAMES_OUT, index=False)
    print("structure/combined feature names saved")
except Exception as e:
    print("structure feature names not saved:", repr(e))


def combined_pocket_feature_vector(sequence, family, mutations=None):
    """68D PyMOL pocket physics + 序列对齐 pocket + 结构对齐 pocket features。"""
    return np.concatenate([
        pymol_pocket_feature_vector(sequence, family, mutations),
        sfgfp_aligned_pocket_feature_vector(sequence, family),
        structure_aligned_pocket_feature_vector(sequence, family),
    ]).astype(np.float32)


def combined_pocket_feature_matrix(sequences, families, mutation_lists=None):
    """批量生成 combined pocket features，保证训练/候选/WT/往年序列使用同一套维度。"""
    sequences = list(sequences)
    families = [normalize_family_name(f) for f in list(families)]
    if mutation_lists is None:
        mutation_lists = [[] for _ in sequences]
    return np.vstack([
        combined_pocket_feature_vector(seq, fam, muts)
        for seq, fam, muts in zip(sequences, families, mutation_lists)
    ]).astype(np.float32)


sys.path.insert(0, str(SAPROT_CODE_PATH))
from utils.foldseek_util import get_struc_seq

def extract_3di_from_structure(structure_path, chain="A"):
    parsed = get_struc_seq(str(FOLDSEEK_BIN), str(structure_path), [chain], plddt_mask=False)
    value = parsed[chain]
    # SaProt foldseek_util usually returns (aa_seq, foldseek_seq, combined_seq).
    aa_seq = value[0]
    three_di = value[1]
    return aa_seq, three_di

WT_3DI = {}
WT_STRUCTURE_SEQ = {}
for fam, path in WT_STRUCTURE_FILES.items():
    aa_seq, three_di = extract_3di_from_structure(path)
    WT_STRUCTURE_SEQ[fam] = aa_seq
    WT_3DI[fam] = three_di
    print(fam, "structure seq len", len(aa_seq), "3Di len", len(three_di))

def stitch_saprot(seq, three_di, strict=True):
    seq = str(seq).strip().upper()
    if strict and len(seq) != len(three_di):
        raise ValueError(f"SaProt stitch length mismatch: seq={len(seq)} 3Di={len(three_di)}")
    n = min(len(seq), len(three_di))
    return "".join(seq[i] + three_di[i] for i in range(n))


pocket physics feature dim: 68
pocket feature names saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/pocket_physics_68_feature_names.csv
sfGFP-aligned pocket positions: [42, 44, 61, 62, 63, 64, 65, 66, 67, 68, 69, 96, 112, 121, 147, 148, 150, 204, 205, 206, 221, 223]
sfGFP-aligned position count: 22
aligned pocket feature dim: 616
sequence-aligned feature names saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/sfgfp_aligned_pocket_feature_names.csv
sfGFP sequence alignment mapping saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/sfgfp_aligned_pocket_position_mapping.csv


,family,sfGFP_pos,sfGFP_aa,family_pos,family_aa
0,amacGFP,42,L,42,L
1,amacGFP,44,L,44,I
2,amacGFP,61,V,61,V
3,amacGFP,62,T,62,T
4,amacGFP,63,T,63,T
5,amacGFP,64,L,64,L
6,amacGFP,65,T,65,S
7,amacGFP,66,Y,66,Y
8,amacGFP,67,G,67,G
9,amacGFP,68,V,68,I


structure alignment mapping saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/structure_aligned_pocket_mapping_to_sfgfp.csv


,family,sfgfp_pos0,sfgfp_pos1,sfgfp_aa,sfgfp_pdb_resnum,mapped_pos0,mapped_pos1,mapped_pdb_resnum,mapped_chain,mapped_aa,ca_distance,core_rmsd_ca,core_aligned_atoms,initial_aligned_atoms,mapping_confidence,is_duplicate_struct_mapping
0,avGFP,41,42,L,42,41,42,42,A,L,0.044543,0.208381,234,238,high,0.0
1,avGFP,43,44,L,44,43,44,44,A,L,0.178923,0.208381,234,238,high,0.0
2,avGFP,60,61,V,61,60,61,61,A,V,0.040885,0.208381,234,238,high,0.0
3,avGFP,61,62,T,62,61,62,62,A,T,0.082643,0.208381,234,238,high,0.0
4,avGFP,62,63,T,63,62,63,63,A,T,0.085892,0.208381,234,238,high,0.0
5,avGFP,63,64,L,64,63,64,64,A,L,0.137661,0.208381,234,238,high,0.0
6,avGFP,64,65,T,65,64,65,65,A,S,0.172958,0.208381,234,238,high,0.0
7,avGFP,65,66,Y,66,65,66,66,A,Y,0.154380,0.208381,234,238,high,0.0
8,avGFP,66,67,G,67,66,67,67,A,G,0.125902,0.208381,234,238,high,0.0
9,avGFP,67,68,V,68,67,68,68,A,V,0.102408,0.208381,234,238,high,0.0


mapping_confidence  high  self
family                        
amacGFP               22     0
avGFP                 22     0
cgreGFP               22     0
ppluGFP               22     0
sfGFP                  0    22
structure-aligned pocket feature dim: 748
combined pocket feature dim: 1432
structure/combined feature names saved
avGFP structure seq len 238 3Di len 238
amacGFP structure seq len 238 3Di len 238
cgreGFP structure seq len 235 3Di len 235
ppluGFP structure seq len 222 3Di len 222
sfGFP structure seq len 238 3Di len 238


In [4]:
# ==========================================
# 4. 构建训练集：full_sequence、saprot_input、pocket features
# ==========================================

train_df = brightness_df.copy()
train_df["mutations_raw"] = train_df["mutations"].fillna("WT").astype(str)
train_df["mutation_numbering_offset"] = TRAIN_MUTATION_POSITION_OFFSET
train_df["mutations_standard"] = train_df["mutations_raw"].map(
    lambda s: standardize_mutation_string(s, position_offset=TRAIN_MUTATION_POSITION_OFFSET)
)
train_df["mut_list_standard"] = train_df["mutations_standard"].map(
    lambda s: split_mutation_string(s, position_offset=0)
)
train_df["full_sequence"] = train_df.apply(build_full_sequence, axis=1)
train_df["mut_list"] = train_df["mut_list_standard"]

def mutation_offset_match_summary(df, offsets=(0, TRAIN_MUTATION_POSITION_OFFSET)):
    """Compare mutation-token WT letters against reference sequences under candidate offsets."""
    rows = []
    for fam, sub in df.groupby("reference"):
        ref = reference_for_family(fam)
        for offset in offsets:
            total = 0
            matched = 0
            out_of_range = 0
            for raw in sub["mutations_raw"].fillna("WT").astype(str):
                for wt, pos0, _mut in split_mutation_string(raw, position_offset=offset):
                    total += 1
                    if pos0 < 0 or pos0 >= len(ref):
                        out_of_range += 1
                    elif ref[pos0] == wt:
                        matched += 1
            rows.append({
                "family": normalize_family_name(fam),
                "tested_offset": int(offset),
                "mutation_tokens": int(total),
                "wt_letter_matches_reference": int(matched),
                "out_of_range": int(out_of_range),
                "match_rate": float(matched / max(1, total)),
            })
    return pd.DataFrame(rows).sort_values(["family", "tested_offset"])

numbering_report = mutation_offset_match_summary(train_df)
numbering_report["source"] = "GFP_data.xlsx:brightness/aaMutations"
numbering_report["raw_coordinate_system"] = "training table labels are shifted -1 relative to AAseqs FASTA"
numbering_report["internal_coordinate_system"] = "AAseqs FASTA 1-based, M=1; Python pos0=FASTA_position-1"
numbering_report["applied_position_offset"] = TRAIN_MUTATION_POSITION_OFFSET
numbering_report["standardization_version"] = MUTATION_NUMBERING_STANDARDIZATION_VERSION
numbering_report.to_csv(TRAIN_MUTATION_NUMBERING_REPORT_OUT, index=False)
train_df[["reference", "mutations_raw", "mutations_standard", "mutation_numbering_offset"]].to_csv(
    TRAIN_MUTATION_STANDARDIZED_OUT,
    index=False,
)
print("mutation numbering standardization report saved:", TRAIN_MUTATION_NUMBERING_REPORT_OUT)
display(numbering_report)

def saprot_input_for_train(row):
    fam = normalize_family_name(row["reference"])
    return stitch_saprot(row["full_sequence"], WT_3DI[fam], strict=True)

train_df["saprot_input"] = train_df.apply(saprot_input_for_train, axis=1)

print(train_df[["mutations_raw", "mutations_standard", "reference", "value", "full_sequence", "saprot_input"]].head())
print("train rows:", len(train_df))

train_pymol_pocket_features = np.vstack([
    pymol_pocket_feature_vector(seq, fam, muts)
    for seq, fam, muts in zip(train_df["full_sequence"], train_df["reference"], train_df["mut_list"])
]).astype(np.float32)

train_aligned_pocket_features = np.vstack([
    sfgfp_aligned_pocket_feature_vector(seq, fam)
    for seq, fam in zip(train_df["full_sequence"], train_df["reference"])
]).astype(np.float32)

train_struct_aligned_pocket_features = np.vstack([
    structure_aligned_pocket_feature_vector(seq, fam)
    for seq, fam in zip(train_df["full_sequence"], train_df["reference"])
]).astype(np.float32)

train_pocket_features = combined_pocket_feature_matrix(
    train_df["full_sequence"].tolist(),
    train_df["reference"].tolist(),
    train_df["mut_list"].tolist(),
)

np.save(RUN_OUTPUT_DIR / "recomputed_train_pymol_pocket_features.npy", train_pymol_pocket_features)
np.save(TRAIN_ALIGNED_POCKET_FEATURE_OUT, train_aligned_pocket_features)
np.save(TRAIN_STRUCT_ALIGNED_POCKET_FEATURE_OUT, train_struct_aligned_pocket_features)
np.save(TRAIN_COMBINED_POCKET_FEATURE_OUT, train_pocket_features)
print("train_pymol_pocket_features:", train_pymol_pocket_features.shape)
print("train_sequence_aligned_pocket_features:", train_aligned_pocket_features.shape)
print("train_structure_aligned_pocket_features:", train_struct_aligned_pocket_features.shape)
print("train combined pocket features:", train_pocket_features.shape)


mutation numbering standardization report saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/train_mutation_numbering_standardization_report.csv


,family,tested_offset,mutation_tokens,wt_letter_matches_reference,out_of_range,match_rate,source,raw_coordinate_system,internal_coordinate_system,applied_position_offset,standardization_version
0,amacGFP,0,102380,3697,0,0.036111,GFP_data.xlsx:brightness/aaMutations,training table labels are shifted -1 relative ...,"AAseqs FASTA 1-based, M=1; Python pos0=FASTA_p...",1,training_aaMutations_offset_plus_1_to_AAseqs_F...
1,amacGFP,1,102380,102380,0,1.000000,GFP_data.xlsx:brightness/aaMutations,training table labels are shifted -1 relative ...,"AAseqs FASTA 1-based, M=1; Python pos0=FASTA_p...",1,training_aaMutations_offset_plus_1_to_AAseqs_F...
2,avGFP,0,200547,9189,0,0.045820,GFP_data.xlsx:brightness/aaMutations,training table labels are shifted -1 relative ...,"AAseqs FASTA 1-based, M=1; Python pos0=FASTA_p...",1,training_aaMutations_offset_plus_1_to_AAseqs_F...
3,avGFP,1,200547,200547,0,1.000000,GFP_data.xlsx:brightness/aaMutations,training table labels are shifted -1 relative ...,"AAseqs FASTA 1-based, M=1; Python pos0=FASTA_p...",1,training_aaMutations_offset_plus_1_to_AAseqs_F...
4,cgreGFP,0,74486,4281,0,0.057474,GFP_data.xlsx:brightness/aaMutations,training table labels are shifted -1 relative ...,"AAseqs FASTA 1-based, M=1; Python pos0=FASTA_p...",1,training_aaMutations_offset_plus_1_to_AAseqs_F...
5,cgreGFP,1,74486,74486,0,1.000000,GFP_data.xlsx:brightness/aaMutations,training table labels are shifted -1 relative ...,"AAseqs FASTA 1-based, M=1; Python pos0=FASTA_p...",1,training_aaMutations_offset_plus_1_to_AAseqs_F...
6,ppluGFP,0,84315,4362,0,0.051735,GFP_data.xlsx:brightness/aaMutations,training table labels are shifted -1 relative ...,"AAseqs FASTA 1-based, M=1; Python pos0=FASTA_p...",1,training_aaMutations_offset_plus_1_to_AAseqs_F...
7,ppluGFP,1,84315,84315,0,1.000000,GFP_data.xlsx:brightness/aaMutations,training table labels are shifted -1 relative ...,"AAseqs FASTA 1-based, M=1; Python pos0=FASTA_p...",1,training_aaMutations_offset_plus_1_to_AAseqs_F...


                   mutations_raw             mutations_standard reference  \
0                             WT                             WT     avGFP   
1                          A109D                          A110D     avGFP   
2  A109D:N145D:I187V:M232T:L235P  A110D:N146D:I188V:M233T:L236P     avGFP   
3        A109D:Y142N:H147L:E221G        A110D:Y143N:H148L:E222G     avGFP   
4                          A109G                          A110G     avGFP   

      value                                      full_sequence  \
0  3.719212  MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...   
1  1.301030  MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...   
2  1.301031  MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...   
3  1.301189  MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...   
4  3.708478  MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...   

                                        saprot_input  
0  MDSAKPGLEVEVLLFQTQGAVKVFPKIELWVEEWLEDWGEDAVANV...  
1  MDSAKPGLEVEVLLFQTQGAVKVFPKI

In [5]:
# ==========================================
# 5. 重新计算训练集 ESM / SaProt embedding
# ==========================================

import torch
import hashlib
import json
from transformers import AutoTokenizer, EsmModel, AutoModel
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
print("device:", device, "dtype:", dtype)

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1)
    summed = torch.sum(last_hidden_state * mask, dim=1)
    denom = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / denom


def sequence_list_hash(items):
    """对输入序列列表做稳定哈希，用于判断 embedding 缓存是否还能复用。"""
    h = hashlib.sha256()
    for item in items:
        text = str(item).strip()
        h.update(str(len(text)).encode("utf-8"))
        h.update(b":")
        h.update(text.encode("utf-8"))
        h.update(bytes([10]))
    return h.hexdigest()


def embedding_manifest_path(save_path):
    save_path = Path(save_path)
    return Path(str(save_path) + ".manifest.json")


def try_load_embedding_cache(inputs, save_path, model_label, model_path, force=False):
    """读取 embedding 缓存。manifest 匹配时最安全；旧 npy 可按行数兼容读取。"""
    save_path = Path(save_path)
    inputs = list(inputs)
    if force or not save_path.exists():
        return None

    manifest_path = embedding_manifest_path(save_path)
    expected_hash = sequence_list_hash(inputs)

    if manifest_path.exists():
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            if (
                manifest.get("model_label") == model_label
                and manifest.get("model_path") == str(model_path)
                and manifest.get("input_count") == len(inputs)
                and manifest.get("input_hash") == expected_hash
            ):
                arr = np.load(save_path)
                if arr.shape[0] == len(inputs):
                    print(f"使用 embedding 缓存: {save_path} shape={arr.shape}")
                    return arr.astype(np.float32)
            print(f"缓存 manifest 不匹配，将重算: {save_path}")
        except Exception as e:
            print(f"读取 manifest 失败，将重算 {save_path}: {repr(e)}")
        return None

    # 兼容已经存在的旧 npy。它没有序列哈希，只有行数校验；第一次重算后会自动写 manifest。
    if TRUST_LEGACY_EMBEDDING_CACHE:
        try:
            arr = np.load(save_path)
            if arr.shape[0] == len(inputs):
                print(f"使用旧版 embedding 缓存，无 manifest，仅按行数校验: {save_path} shape={arr.shape}")
                return arr.astype(np.float32)
            print(f"旧版缓存行数不匹配，将重算: {save_path} cache_rows={arr.shape[0]} expected={len(inputs)}")
        except Exception as e:
            print(f"读取旧版缓存失败，将重算 {save_path}: {repr(e)}")

    return None


def save_embedding_manifest(inputs, save_path, model_label, model_path, arr):
    manifest = {
        "model_label": model_label,
        "model_path": str(model_path),
        "input_count": len(inputs),
        "input_hash": sequence_list_hash(inputs),
        "shape": list(arr.shape),
        "dtype": str(arr.dtype),
    }
    embedding_manifest_path(save_path).write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")


def compute_esm_embeddings(sequences, save_path, batch_size=16, force=False):
    sequences = [str(s).strip().upper() for s in sequences]
    cached = try_load_embedding_cache(sequences, save_path, "ESM2", ESM_MODEL_PATH, force=force)
    if cached is not None:
        return cached

    tokenizer = AutoTokenizer.from_pretrained(str(ESM_MODEL_PATH))
    model = EsmModel.from_pretrained(str(ESM_MODEL_PATH), torch_dtype=dtype).to(device)
    model.eval()
    all_emb = []
    with torch.no_grad():
        for i in tqdm(range(0, len(sequences), batch_size), desc=f"ESM -> {Path(save_path).name}"):
            batch = list(sequences[i:i+batch_size])
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
            outputs = model(**inputs)
            pooled = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
            all_emb.append(pooled.cpu().to(torch.float32).numpy())
    arr = np.vstack(all_emb).astype(np.float32)
    np.save(save_path, arr)
    save_embedding_manifest(sequences, save_path, "ESM2", ESM_MODEL_PATH, arr)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return arr


def compute_saprot_embeddings(saprot_inputs, save_path, batch_size=16, force=False):
    saprot_inputs = [str(s).strip() for s in saprot_inputs]
    cached = try_load_embedding_cache(saprot_inputs, save_path, "SaProt", SAPROT_MODEL_PATH, force=force)
    if cached is not None:
        return cached

    tokenizer = AutoTokenizer.from_pretrained(str(SAPROT_MODEL_PATH), trust_remote_code=True)
    model = AutoModel.from_pretrained(str(SAPROT_MODEL_PATH), trust_remote_code=True, torch_dtype=dtype).to(device)
    model.eval()
    all_emb = []
    with torch.no_grad():
        for i in tqdm(range(0, len(saprot_inputs), batch_size), desc=f"SaProt -> {Path(save_path).name}"):
            batch = list(saprot_inputs[i:i+batch_size])
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
            outputs = model(**inputs)
            pooled = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
            all_emb.append(pooled.cpu().to(torch.float32).numpy())
    arr = np.vstack(all_emb).astype(np.float32)
    np.save(save_path, arr)
    save_embedding_manifest(saprot_inputs, save_path, "SaProt", SAPROT_MODEL_PATH, arr)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return arr

train_esm = compute_esm_embeddings(
    train_df["full_sequence"].tolist(),
    TRAIN_ESM_OUT,
    ESM_BATCH_SIZE,
    force=RECOMPUTE_TRAIN_EMBEDDINGS,
)

train_saprot = compute_saprot_embeddings(
    train_df["saprot_input"].tolist(),
    TRAIN_SAPROT_OUT,
    SAPROT_BATCH_SIZE,
    force=RECOMPUTE_TRAIN_EMBEDDINGS,
)

print("train_esm:", train_esm.shape)
print("train_saprot:", train_saprot.shape)


/home/liuchang/anaconda3/envs/GFP/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda dtype: torch.bfloat16
缓存 manifest 不匹配，将重算: /home/liuchang/PythonProject2/server_full_recompute_outputs/recomputed_train_esm2_embeddings.npy


Some weights of EsmModel were not initialized from the model checkpoint at /home/liuchang/esm2_t33_650M and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
ESM -> recomputed_train_esm2_embeddings.npy: 100%|██████████| 8822/8822 [21:29<00:00,  6.84it/s]


缓存 manifest 不匹配，将重算: /home/liuchang/PythonProject2/server_full_recompute_outputs/recomputed_train_saprot_embeddings.npy


Some weights of EsmModel were not initialized from the model checkpoint at /home/liuchang/saprot_650M and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight', 'esm.embeddings.position_embeddings.weight', 'esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
SaProt -> recomputed_train_saprot_embeddings.npy: 100%|██████████| 8822/8822 [05:12<00:00, 28.20it/s]


train_esm: (141144, 1280)
train_saprot: (141144, 1280)


In [6]:
# ==========================================
# 6. 从零训练 XGBoost 与物理增强门控 NN
# ==========================================

from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

# 原始亮度标签
y_raw = train_df["value"].values.astype(np.float32)

# family 内归一化排名标签：更适合跨 GFP family 迁移。
# 同时补充 z-score 与 top-k 标签，便于后续诊断模型是否真的能识别 family 内高亮序列。
train_df["family"] = train_df["reference"].map(normalize_family_name)
train_df["brightness_rank"] = train_df.groupby("family")["value"].rank(pct=True)
train_df["brightness_zscore"] = train_df.groupby("family")["value"].transform(
    lambda s: (s - s.mean()) / max(float(s.std(ddof=0)), 1e-6)
)
train_df["is_family_top10"] = (train_df["brightness_rank"] >= 0.90).astype(int)
train_df["is_family_top20"] = (train_df["brightness_rank"] >= 0.80).astype(int)

y_rank = train_df["brightness_rank"].values.astype(np.float32)
y_zscore = train_df["brightness_zscore"].values.astype(np.float32)

# family-balanced sample weight：避免样本量大的 family 支配模型。
family_counts = train_df["family"].value_counts().to_dict()
family_sample_weight = train_df["family"].map(lambda f: 1.0 / family_counts[f]).values.astype(np.float32)
family_sample_weight = family_sample_weight / family_sample_weight.mean()

# WT-anchor delta features：让 XGBoost 学“相对本 family WT 改变了什么”，而不是记住 family 背景。
MODEL_FEATURE_USE_DELTA = True
MODEL_FAMILIES = sorted(set(pd.unique(train_df["family"])) | {"sfGFP"})
WT_FAMILY_SEQUENCES = {fam: reference_for_family(fam) for fam in MODEL_FAMILIES}
WT_FAMILY_SAPROT_INPUTS = {
    fam: stitch_saprot(WT_FAMILY_SEQUENCES[fam], WT_3DI[fam], strict=False)
    for fam in MODEL_FAMILIES
}
WT_FAMILY_ESM_ARRAY = compute_esm_embeddings(
    [WT_FAMILY_SEQUENCES[fam] for fam in MODEL_FAMILIES],
    RUN_OUTPUT_DIR / "wt_family_anchor_esm2_embeddings.npy",
    ESM_BATCH_SIZE,
    force=RECOMPUTE_TRAIN_EMBEDDINGS,
)
WT_FAMILY_SAPROT_ARRAY = compute_saprot_embeddings(
    [WT_FAMILY_SAPROT_INPUTS[fam] for fam in MODEL_FAMILIES],
    RUN_OUTPUT_DIR / "wt_family_anchor_saprot_embeddings.npy",
    SAPROT_BATCH_SIZE,
    force=RECOMPUTE_TRAIN_EMBEDDINGS,
)
WT_FAMILY_POCKET_ARRAY = combined_pocket_feature_matrix(
    [WT_FAMILY_SEQUENCES[fam] for fam in MODEL_FAMILIES],
    MODEL_FAMILIES,
    [[] for _ in MODEL_FAMILIES],
)
WT_FAMILY_ESM = {fam: WT_FAMILY_ESM_ARRAY[i] for i, fam in enumerate(MODEL_FAMILIES)}
WT_FAMILY_SAPROT = {fam: WT_FAMILY_SAPROT_ARRAY[i] for i, fam in enumerate(MODEL_FAMILIES)}
WT_FAMILY_POCKET = {fam: WT_FAMILY_POCKET_ARRAY[i] for i, fam in enumerate(MODEL_FAMILIES)}


def _family_anchor_matrix(families, anchor_dict):
    families = [normalize_family_name(f) for f in families]
    return np.vstack([anchor_dict[fam] for fam in families]).astype(np.float32)


def build_model_feature_matrix(esm, sap, pocket_features, families):
    """构建 XGBoost/stacking 使用的统一特征。

    当前默认使用 WT-anchor delta：
    [ESM(mut)-ESM(WT_family), SaProt(mut)-SaProt(WT_family), Pocket(mut)-Pocket(WT_family), Pocket(mut)]
    这样模型重点学习突变效应，而不是不同 GFP family 的绝对背景差异。
    """
    esm = np.asarray(esm, dtype=np.float32)
    sap = np.asarray(sap, dtype=np.float32)
    pocket_features = np.asarray(pocket_features, dtype=np.float32)
    families = [normalize_family_name(f) for f in list(families)]

    if MODEL_FEATURE_USE_DELTA:
        wt_esm = _family_anchor_matrix(families, WT_FAMILY_ESM)
        wt_sap = _family_anchor_matrix(families, WT_FAMILY_SAPROT)
        wt_pocket = _family_anchor_matrix(families, WT_FAMILY_POCKET)
        return np.concatenate([
            esm - wt_esm,
            sap - wt_sap,
            pocket_features - wt_pocket,
            pocket_features,
        ], axis=1).astype(np.float32)

    return np.concatenate([esm, sap, pocket_features], axis=1).astype(np.float32)


def build_delta_components(esm, sap, pocket_features, families):
    """Return own-family WT-anchor delta components for all heavy models."""
    esm = np.asarray(esm, dtype=np.float32)
    sap = np.asarray(sap, dtype=np.float32)
    pocket_features = np.asarray(pocket_features, dtype=np.float32)
    families = [normalize_family_name(f) for f in list(families)]
    missing = sorted(set(families) - set(WT_FAMILY_ESM))
    if missing:
        raise KeyError(f"Missing WT-anchor features for families: {missing}")
    wt_esm = _family_anchor_matrix(families, WT_FAMILY_ESM)
    wt_sap = _family_anchor_matrix(families, WT_FAMILY_SAPROT)
    wt_pocket = _family_anchor_matrix(families, WT_FAMILY_POCKET)
    return (
        (esm - wt_esm).astype(np.float32),
        (sap - wt_sap).astype(np.float32),
        (pocket_features - wt_pocket).astype(np.float32),
        pocket_features.astype(np.float32),
    )


def build_gating_inputs(esm, sap, pocket_features, families):
    """Build GatingNN inputs. Delta mode uses ESM/SaProt delta and [pocket_delta, pocket_abs]."""
    if GATING_USE_DELTA:
        esm_delta, sap_delta, pocket_delta, pocket_abs = build_delta_components(esm, sap, pocket_features, families)
        phys = np.concatenate([pocket_delta, pocket_abs], axis=1).astype(np.float32)
        return esm_delta, sap_delta, phys
    return np.asarray(esm, dtype=np.float32), np.asarray(sap, dtype=np.float32), np.asarray(pocket_features, dtype=np.float32)


X_full = build_model_feature_matrix(
    train_esm,
    train_saprot,
    train_pocket_features,
    train_df["family"].tolist(),
)

print("raw brightness:", pd.Series(y_raw).describe())
print("family rank brightness:", pd.Series(y_rank).describe())

# 后面 GatingNN 也使用 family 内 rank 作为监督信号，和 XGBoost rank 模型保持一致。
# raw brightness 只保留给 XGBoost raw 模型，用来估计相对 WT 的亮度比值。
y = y_rank
np.save(TRAIN_FULL_FEATURE_OUT, X_full)
print("X_full:", X_full.shape)

groups = train_df["family"].values
unique_families = sorted(pd.unique(groups))
print("validation families:", unique_families)


def make_family_aware_split(groups):
    """生成主验证集。

    within_family_stratified：每个 family 内按亮度分层抽验证集，用于训练 stacking/final score。
    groupkfold / leave_one_family_out：更严格，只建议做泛化诊断。
    """
    mode = str(VALIDATION_SPLIT_MODE).lower()
    groups_arr = np.asarray(groups)

    if mode == "within_family_stratified":
        rng = np.random.default_rng(RANDOM_SEED)
        val_indices = []
        for fam in unique_families:
            fam_idx = np.where(groups_arr == fam)[0]
            fam_values = y_raw[fam_idx]
            # 在每个 family 内按亮度排序分箱，保证验证集覆盖高/中/低亮度。
            n_bins = min(10, max(2, len(fam_idx) // 2000))
            ranks = pd.Series(fam_values).rank(method="first")
            try:
                bins = pd.qcut(ranks, q=n_bins, labels=False, duplicates="drop")
            except Exception:
                bins = pd.Series(np.zeros(len(fam_idx), dtype=int))
            tmp = pd.DataFrame({"idx": fam_idx, "bin": np.asarray(bins, dtype=int)})
            for _bin, sub in tmp.groupby("bin"):
                n_val = max(1, int(round(len(sub) * VALIDATION_FRACTION)))
                n_val = min(n_val, len(sub))
                val_indices.extend(rng.choice(sub["idx"].values, size=n_val, replace=False).tolist())
        idx_val = np.array(sorted(set(val_indices)), dtype=int)
        idx_train = np.setdiff1d(np.arange(len(groups_arr)), idx_val)
        return idx_train, idx_val, f"within_family_stratified:{VALIDATION_FRACTION}"

    if mode == "leave_one_family_out":
        holdout = normalize_family_name(VALIDATION_HOLDOUT_FAMILY) if VALIDATION_HOLDOUT_FAMILY else unique_families[0]
        if holdout not in set(unique_families):
            raise ValueError(f"VALIDATION_HOLDOUT_FAMILY={holdout} 不在训练 family 中: {unique_families}")
        idx_val = np.where(groups_arr == holdout)[0]
        idx_train = np.where(groups_arr != holdout)[0]
        return idx_train, idx_val, f"leave_one_family_out:{holdout}"

    if mode == "groupkfold":
        n_splits = min(5, len(unique_families))
        splitter = GroupKFold(n_splits=n_splits)
        idx_train, idx_val = next(splitter.split(X_full, y, groups_arr))
        heldout = sorted(pd.unique(groups_arr[idx_val]))
        return idx_train, idx_val, f"groupkfold_first_fold:{heldout}"

    raise ValueError(f"未知 VALIDATION_SPLIT_MODE: {VALIDATION_SPLIT_MODE}")


idx_train, idx_val, validation_name = make_family_aware_split(groups)
print("validation split:", validation_name)
print("train rows:", len(idx_train), "val rows:", len(idx_val))
print("train families:", sorted(pd.unique(groups[idx_train])))
print("val families:", sorted(pd.unique(groups[idx_val])))

xgb_raw_model = None
xgb_rank_model = None

try:
    from xgboost import XGBRegressor

    def make_xgb_model(n_estimators=2500):
        return XGBRegressor(
            n_estimators=n_estimators,
            learning_rate=0.03,
            max_depth=7,
            subsample=0.85,
            colsample_bytree=0.75,
            tree_method="hist",
            device="cuda" if torch.cuda.is_available() else "cpu",
            eval_metric="rmse",
            random_state=RANDOM_SEED,
        )

    def fit_xgb_with_validation(model, X_tr, y_tr, X_va, y_va, sample_weight=None, verbose=100):
        """带 early stopping 的 XGBoost 训练；若当前 xgboost 版本不支持则自动退回普通 fit。"""
        fit_kwargs = {"eval_set": [(X_va, y_va)], "verbose": verbose}
        if sample_weight is not None:
            fit_kwargs["sample_weight"] = sample_weight
        try:
            model.set_params(early_stopping_rounds=XGB_EARLY_STOPPING_ROUNDS)
            model.fit(X_tr, y_tr, **fit_kwargs)
        except TypeError:
            model.fit(X_tr, y_tr, **fit_kwargs)
        except ValueError as e:
            print("XGBoost early stopping fallback:", repr(e))
            try:
                model.set_params(early_stopping_rounds=None)
            except Exception:
                pass
            model.fit(X_tr, y_tr, **fit_kwargs)
        return model

    # 1. raw brightness 模型：用于输出绝对预测和相对 WT 比值
    xgb_raw_model = make_xgb_model()
    fit_xgb_with_validation(
        xgb_raw_model,
        X_full[idx_train],
        y_raw[idx_train],
        X_full[idx_val],
        y_raw[idx_val],
        sample_weight=family_sample_weight[idx_train],
        verbose=100,
    )
    pred_raw = xgb_raw_model.predict(X_full[idx_val])
    print(
        "XGBoost RAW R2:",
        r2_score(y_raw[idx_val], pred_raw),
        "RMSE:",
        np.sqrt(mean_squared_error(y_raw[idx_val], pred_raw)),
    )
    xgb_raw_model.save_model(str(XGB_RAW_MODEL_OUT))

    # 2. family rank 模型：用于候选筛选主分数
    xgb_rank_model = make_xgb_model()
    fit_xgb_with_validation(
        xgb_rank_model,
        X_full[idx_train],
        y_rank[idx_train],
        X_full[idx_val],
        y_rank[idx_val],
        sample_weight=family_sample_weight[idx_train],
        verbose=100,
    )
    pred_rank = xgb_rank_model.predict(X_full[idx_val])
    print(
        "XGBoost FAMILY-RANK R2:",
        r2_score(y_rank[idx_val], pred_rank),
        "RMSE:",
        np.sqrt(mean_squared_error(y_rank[idx_val], pred_rank)),
    )

    def ranking_diagnostics(y_true_rank, y_pred_rank, top_frac=0.10):
        """排序诊断：Spearman 与 top-k recall 比 R2 更接近最终筛选任务。"""
        y_true_rank = np.asarray(y_true_rank, dtype=float)
        y_pred_rank = np.asarray(y_pred_rank, dtype=float)
        try:
            spearman = float(pd.Series(y_true_rank).corr(pd.Series(y_pred_rank), method="spearman"))
        except Exception:
            spearman = np.nan
        n_top = max(1, int(np.ceil(len(y_true_rank) * top_frac)))
        true_top = set(np.argsort(-y_true_rank)[:n_top])
        pred_top = set(np.argsort(-y_pred_rank)[:n_top])
        recall = len(true_top & pred_top) / max(1, len(true_top))
        return spearman, float(recall)

    val_spearman, val_top10_recall = ranking_diagnostics(y_rank[idx_val], pred_rank, top_frac=0.10)
    _, val_top20_recall = ranking_diagnostics(y_rank[idx_val], pred_rank, top_frac=0.20)
    print("XGBoost FAMILY-RANK Spearman:", val_spearman, "Top10 recall:", val_top10_recall, "Top20 recall:", val_top20_recall)
    xgb_rank_model.save_model(str(XGB_RANK_MODEL_OUT))

    # Leave-one-family-out 诊断：逐个 family 留出，检查跨家族泛化。
    # 这个表只用于判断模型可信度，不参与候选预测。
    if RUN_LEAVE_ONE_FAMILY_OUT_DIAGNOSTIC and len(unique_families) >= 2:
        lofo_rows = []
        logo = LeaveOneGroupOut()
        for lofo_train_idx, lofo_val_idx in logo.split(X_full, y_rank, groups):
            held_family = sorted(pd.unique(groups[lofo_val_idx]))[0]
            print(f"LOFO diagnostic: hold out {held_family}, train={len(lofo_train_idx)}, val={len(lofo_val_idx)}")

            raw_diag = make_xgb_model(n_estimators=LOFO_XGB_ESTIMATORS)
            raw_diag.fit(X_full[lofo_train_idx], y_raw[lofo_train_idx], sample_weight=family_sample_weight[lofo_train_idx], verbose=False)
            raw_diag_pred = raw_diag.predict(X_full[lofo_val_idx])

            rank_diag = make_xgb_model(n_estimators=LOFO_XGB_ESTIMATORS)
            rank_diag.fit(X_full[lofo_train_idx], y_rank[lofo_train_idx], sample_weight=family_sample_weight[lofo_train_idx], verbose=False)
            rank_diag_pred = rank_diag.predict(X_full[lofo_val_idx])

            lofo_spearman, lofo_top10 = ranking_diagnostics(y_rank[lofo_val_idx], rank_diag_pred, top_frac=0.10)
            _, lofo_top20 = ranking_diagnostics(y_rank[lofo_val_idx], rank_diag_pred, top_frac=0.20)
            lofo_rows.append({
                "heldout_family": held_family,
                "train_rows": int(len(lofo_train_idx)),
                "val_rows": int(len(lofo_val_idx)),
                "xgb_raw_r2": float(r2_score(y_raw[lofo_val_idx], raw_diag_pred)),
                "xgb_raw_rmse": float(np.sqrt(mean_squared_error(y_raw[lofo_val_idx], raw_diag_pred))),
                "xgb_rank_r2": float(r2_score(y_rank[lofo_val_idx], rank_diag_pred)),
                "xgb_rank_rmse": float(np.sqrt(mean_squared_error(y_rank[lofo_val_idx], rank_diag_pred))),
                "xgb_rank_spearman": lofo_spearman,
                "xgb_rank_top10_recall": lofo_top10,
                "xgb_rank_top20_recall": lofo_top20,
            })

        lofo_diagnostic_df = pd.DataFrame(lofo_rows).sort_values("heldout_family")
        lofo_diagnostic_df.to_csv(LOFO_DIAGNOSTIC_OUT, index=False)
        display(lofo_diagnostic_df)
        print("LOFO diagnostic saved:", LOFO_DIAGNOSTIC_OUT)

    # 验证完成后，正式候选预测的 XGBoost 用全量训练集重训。
    # pred_raw / pred_rank 仍保留 family-aware 验证集预测，用于 stacking 和可信度评估。
    if REFIT_XGB_ON_ALL_DATA_AFTER_VALIDATION:
        print("Refitting final XGBoost models on all training families...")
        xgb_raw_model = make_xgb_model()
        xgb_raw_model.fit(X_full, y_raw, sample_weight=family_sample_weight, verbose=False)
        xgb_raw_model.save_model(str(XGB_RAW_MODEL_OUT))

        xgb_rank_model = make_xgb_model()
        xgb_rank_model.fit(X_full, y_rank, sample_weight=family_sample_weight, verbose=False)
        xgb_rank_model.save_model(str(XGB_RANK_MODEL_OUT))
        print("Final XGBoost models refit on all data and saved.")

    # 兼容旧代码：xgb_model 默认指向 raw 模型
    xgb_model = xgb_raw_model

except Exception as e:
    print("XGBoost skipped:", repr(e))
    xgb_model = None

import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.benchmark = False

class PhysicsEnhancedGatingNet(nn.Module):
    """Residual-gated ESM/SaProt model.

    旧版是 guided_esm = esm * sigmoid(gate)，容易把 ESM embedding 直接关掉。
    这里改成残差门控：guided_esm = esm * (1 + scale * gate)，保留 ESM 主信号。
    """
    def __init__(self, esm_dim=1280, sap_dim=1280, phys_dim=68, dropout=0.30, residual_scale=0.20):
        super().__init__()
        self.residual_scale = residual_scale
        self.phys_proj = nn.Sequential(
            nn.Linear(phys_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
        )
        self.gate = nn.Sequential(
            nn.Linear(sap_dim + 64, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, esm_dim),
            nn.Sigmoid(),
        )
        self.esm_proj = nn.Sequential(nn.Linear(esm_dim, 512), nn.ReLU(), nn.Dropout(dropout * 0.5))
        self.sap_proj = nn.Sequential(nn.Linear(sap_dim, 512), nn.ReLU(), nn.Dropout(dropout * 0.5))
        self.regressor = nn.Sequential(
            nn.Linear(512 + 512 + 64, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )

    def forward(self, esm, sap, phys):
        p = self.phys_proj(phys)
        w = self.gate(torch.cat([sap, p], dim=1))
        guided_esm = esm * (1.0 + self.residual_scale * w)
        e = self.esm_proj(guided_esm)
        s = self.sap_proj(sap)
        return self.regressor(torch.cat([e, s, p], dim=1))

GATING_ESM_INPUT, GATING_SAP_INPUT, GATING_PHYS_INPUT = build_gating_inputs(
    train_esm,
    train_saprot,
    train_pocket_features,
    train_df["family"].tolist(),
)
delta_feature_summary = pd.DataFrame([{
    "xgb_uses_delta": bool(MODEL_FEATURE_USE_DELTA),
    "gating_uses_delta": bool(GATING_USE_DELTA),
    "lightweight_uses_delta": bool(LIGHTWEIGHT_USE_DELTA),
    "train_esm_dim": int(train_esm.shape[1]),
    "train_saprot_dim": int(train_saprot.shape[1]),
    "pocket_abs_dim": int(train_pocket_features.shape[1]),
    "xgb_feature_dim": int(X_full.shape[1]),
    "gating_esm_dim": int(GATING_ESM_INPUT.shape[1]),
    "gating_sap_dim": int(GATING_SAP_INPUT.shape[1]),
    "gating_phys_dim": int(GATING_PHYS_INPUT.shape[1]),
}])
delta_feature_summary.to_csv(DELTA_FEATURE_SUMMARY_OUT, index=False)
print("all-model delta feature summary saved:", DELTA_FEATURE_SUMMARY_OUT)
display(delta_feature_summary)

sc_esm = StandardScaler()
sc_sap = StandardScaler()
sc_phys = StandardScaler()

esm_train = sc_esm.fit_transform(GATING_ESM_INPUT[idx_train])
sap_train = sc_sap.fit_transform(GATING_SAP_INPUT[idx_train])
phys_train = sc_phys.fit_transform(GATING_PHYS_INPUT[idx_train])
esm_val = sc_esm.transform(GATING_ESM_INPUT[idx_val])
sap_val = sc_sap.transform(GATING_SAP_INPUT[idx_val])
phys_val = sc_phys.transform(GATING_PHYS_INPUT[idx_val])

train_ds = TensorDataset(
    torch.tensor(esm_train, dtype=torch.float32),
    torch.tensor(sap_train, dtype=torch.float32),
    torch.tensor(phys_train, dtype=torch.float32),
    torch.tensor(y_rank[idx_train].reshape(-1, 1), dtype=torch.float32),
    torch.tensor(family_sample_weight[idx_train].reshape(-1, 1), dtype=torch.float32),
)

loader_generator = torch.Generator()
loader_generator.manual_seed(RANDOM_SEED)
loader = DataLoader(train_ds, batch_size=512, shuffle=True, generator=loader_generator)

gating_model = PhysicsEnhancedGatingNet(
    esm_dim=GATING_ESM_INPUT.shape[1],
    sap_dim=GATING_SAP_INPUT.shape[1],
    phys_dim=GATING_PHYS_INPUT.shape[1],
    dropout=GATING_DROPOUT,
    residual_scale=GATING_RESIDUAL_SCALE,
).to(device)
opt = torch.optim.AdamW(gating_model.parameters(), lr=GATING_LR, weight_decay=1e-4)
loss_fn = nn.MSELoss()

def lr_lambda(epoch_idx):
    # Linear warmup, then cosine decay.
    if epoch_idx < GATING_WARMUP_EPOCHS:
        return float(epoch_idx + 1) / float(max(1, GATING_WARMUP_EPOCHS))
    progress = (epoch_idx - GATING_WARMUP_EPOCHS) / float(max(1, GATING_MAX_EPOCHS - GATING_WARMUP_EPOCHS))
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lr_lambda)

best_val_r2 = -np.inf
best_state = None
best_epoch = 0
epochs_without_improve = 0

for epoch in range(1, GATING_MAX_EPOCHS + 1):
    gating_model.train()
    losses = []
    for eb, sb, pb, yb, wb in loader:
        eb, sb, pb, yb, wb = eb.to(device), sb.to(device), pb.to(device), yb.to(device), wb.to(device)
        opt.zero_grad()
        pred_batch = gating_model(eb, sb, pb)
        loss = ((pred_batch - yb) ** 2 * wb).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gating_model.parameters(), max_norm=GATING_GRAD_CLIP_NORM)
        opt.step()
        losses.append(float(loss.detach().cpu()))

    scheduler.step()

    gating_model.eval()
    with torch.no_grad():
        val_pred_epoch = gating_model(
            torch.tensor(esm_val, dtype=torch.float32).to(device),
            torch.tensor(sap_val, dtype=torch.float32).to(device),
            torch.tensor(phys_val, dtype=torch.float32).to(device),
        ).cpu().numpy().reshape(-1)

    val_r2 = r2_score(y_rank[idx_val], val_pred_epoch)
    val_rmse = np.sqrt(mean_squared_error(y_rank[idx_val], val_pred_epoch))
    print(
        f"Epoch {epoch:02d}",
        f"train MSE {np.mean(losses):.4f}",
        f"val R2 {val_r2:.4f}",
        f"val RMSE {val_rmse:.4f}",
        f"lr {scheduler.get_last_lr()[0]:.2e}",
    )

    if val_r2 > best_val_r2 + 1e-4:
        best_val_r2 = val_r2
        best_state = {k: v.detach().cpu().clone() for k, v in gating_model.state_dict().items()}
        best_epoch = epoch
        epochs_without_improve = 0
    else:
        epochs_without_improve += 1
        if epochs_without_improve >= GATING_EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch}; best val R2 {best_val_r2:.4f}")
            break

if best_state is not None:
    gating_model.load_state_dict(best_state)
    gating_model.to(device)

gating_model.eval()
with torch.no_grad():
    pred_nn = gating_model(
        torch.tensor(esm_val, dtype=torch.float32).to(device),
        torch.tensor(sap_val, dtype=torch.float32).to(device),
        torch.tensor(phys_val, dtype=torch.float32).to(device),
    ).cpu().numpy().reshape(-1)

print(
    "Gating NN FAMILY-RANK R2:",
    r2_score(y_rank[idx_val], pred_nn),
    "RMSE:",
    np.sqrt(mean_squared_error(y_rank[idx_val], pred_nn)),
)

torch.save({
    "model_state_dict": gating_model.state_dict(),
    "esm_dim": GATING_ESM_INPUT.shape[1],
    "sap_dim": GATING_SAP_INPUT.shape[1],
    "phys_dim": GATING_PHYS_INPUT.shape[1],
    "dropout": GATING_DROPOUT,
    "residual_scale": GATING_RESIDUAL_SCALE,
    "validation_split": validation_name,
    "best_epoch": best_epoch,
}, GATING_RANK_MODEL_OUT)

# 验证完成后，正式候选预测的 Gating NN 也用全量训练集重训。
# pred_nn 仍保留为 family-aware 验证预测，用于 rank ensemble / stacking；这里替换的是后续候选预测用的 gating_model 和 scaler。
if REFIT_GATING_ON_ALL_DATA_AFTER_VALIDATION:
    final_epochs = int(FINAL_GATING_REFIT_EPOCHS if FINAL_GATING_REFIT_EPOCHS is not None else max(1, best_epoch if best_epoch else GATING_MAX_EPOCHS))
    print(f"Refitting final Gating NN on all training families for {final_epochs} epochs...")

    sc_esm_final = StandardScaler()
    sc_sap_final = StandardScaler()
    sc_phys_final = StandardScaler()
    esm_all = sc_esm_final.fit_transform(GATING_ESM_INPUT)
    sap_all = sc_sap_final.fit_transform(GATING_SAP_INPUT)
    phys_all = sc_phys_final.fit_transform(GATING_PHYS_INPUT)

    all_ds = TensorDataset(
        torch.tensor(esm_all, dtype=torch.float32),
        torch.tensor(sap_all, dtype=torch.float32),
        torch.tensor(phys_all, dtype=torch.float32),
        torch.tensor(y_rank.reshape(-1, 1), dtype=torch.float32),
        torch.tensor(family_sample_weight.reshape(-1, 1), dtype=torch.float32),
    )
    all_loader_generator = torch.Generator()
    all_loader_generator.manual_seed(RANDOM_SEED)
    all_loader = DataLoader(all_ds, batch_size=512, shuffle=True, generator=all_loader_generator)

    final_gating_model = PhysicsEnhancedGatingNet(
        esm_dim=GATING_ESM_INPUT.shape[1],
    sap_dim=GATING_SAP_INPUT.shape[1],
    phys_dim=GATING_PHYS_INPUT.shape[1],
        dropout=GATING_DROPOUT,
        residual_scale=GATING_RESIDUAL_SCALE,
    ).to(device)
    final_opt = torch.optim.AdamW(final_gating_model.parameters(), lr=GATING_LR, weight_decay=1e-4)
    final_scheduler = torch.optim.lr_scheduler.LambdaLR(final_opt, lr_lambda=lr_lambda)

    for epoch in range(1, final_epochs + 1):
        final_gating_model.train()
        losses = []
        for eb, sb, pb, yb, wb in all_loader:
            eb, sb, pb, yb, wb = eb.to(device), sb.to(device), pb.to(device), yb.to(device), wb.to(device)
            final_opt.zero_grad()
            pred_batch = final_gating_model(eb, sb, pb)
            loss = ((pred_batch - yb) ** 2 * wb).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(final_gating_model.parameters(), max_norm=GATING_GRAD_CLIP_NORM)
            final_opt.step()
            losses.append(float(loss.detach().cpu()))
        final_scheduler.step()
        print(f"Final Gating epoch {epoch:02d}/{final_epochs} train MSE {np.mean(losses):.4f}")

    gating_model = final_gating_model
    gating_model.eval()
    sc_esm = sc_esm_final
    sc_sap = sc_sap_final
    sc_phys = sc_phys_final

    torch.save({
        "model_state_dict": gating_model.state_dict(),
        "esm_dim": GATING_ESM_INPUT.shape[1],
    "sap_dim": GATING_SAP_INPUT.shape[1],
    "phys_dim": GATING_PHYS_INPUT.shape[1],
        "dropout": GATING_DROPOUT,
        "residual_scale": GATING_RESIDUAL_SCALE,
        "refit_on_all_data": True,
        "final_epochs": final_epochs,
    }, GATING_RANK_MODEL_OUT)
    print("Final Gating NN refit on all data and saved:", GATING_RANK_MODEL_OUT)


使用 embedding 缓存: /home/liuchang/PythonProject2/server_full_recompute_outputs/wt_family_anchor_esm2_embeddings.npy shape=(5, 1280)
使用 embedding 缓存: /home/liuchang/PythonProject2/server_full_recompute_outputs/wt_family_anchor_saprot_embeddings.npy shape=(5, 1280)
raw brightness: count    141144.000000
mean          3.335396
std           0.945644
min           1.283419
25%           2.785870
50%           3.654869
75%           4.011656
max           4.603003
dtype: float64
family rank brightness: count    141144.000000
mean          0.500014
std           0.287863
min           0.000019
25%           0.235563
50%           0.500012
75%           0.750015
max           1.000000
dtype: float64
X_full: (141144, 5424)
validation families: ['amacGFP', 'avGFP', 'cgreGFP', 'ppluGFP']
validation split: within_family_stratified:0.15
train rows: 119964 val rows: 21180
train families: ['amacGFP', 'avGFP', 'cgreGFP', 'ppluGFP']
val families: ['amacGFP', 'avGFP', 'cgreGFP', 'ppluGFP']
[0]	validation

/home/liuchang/anaconda3/envs/GFP/lib/python3.11/site-packages/xgboost/core.py:751: UserWarning: [23:47:42] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


XGBoost RAW R2: 0.7430895566940308 RMSE: 0.47913288087183015
[0]	validation_0-rmse:0.28636
[100]	validation_0-rmse:0.23860
[200]	validation_0-rmse:0.22702
[300]	validation_0-rmse:0.22135
[400]	validation_0-rmse:0.21688
[500]	validation_0-rmse:0.21321
[600]	validation_0-rmse:0.21032
[700]	validation_0-rmse:0.20792
[800]	validation_0-rmse:0.20598
[900]	validation_0-rmse:0.20436
[1000]	validation_0-rmse:0.20280
[1100]	validation_0-rmse:0.20151
[1200]	validation_0-rmse:0.20038
[1300]	validation_0-rmse:0.19941
[1400]	validation_0-rmse:0.19844
[1500]	validation_0-rmse:0.19764
[1600]	validation_0-rmse:0.19696
[1700]	validation_0-rmse:0.19631
[1800]	validation_0-rmse:0.19567
[1900]	validation_0-rmse:0.19513
[2000]	validation_0-rmse:0.19462
[2100]	validation_0-rmse:0.19423
[2200]	validation_0-rmse:0.19380
[2300]	validation_0-rmse:0.19340
[2400]	validation_0-rmse:0.19303
[2499]	validation_0-rmse:0.19265
XGBoost FAMILY-RANK R2: 0.5517892837524414 RMSE: 0.19264703604823732
XGBoost FAMILY-RANK Spea

,heldout_family,train_rows,val_rows,xgb_raw_r2,xgb_raw_rmse,xgb_rank_r2,xgb_rank_rmse,xgb_rank_spearman,xgb_rank_top10_recall,xgb_rank_top20_recall
0,amacGFP,107633,33511,-1.921977,0.774983,0.074723,0.277678,0.293286,0.133353,0.290765
1,avGFP,89429,51715,0.028433,1.043468,0.030091,0.284299,0.476613,0.226991,0.385188
2,cgreGFP,116628,24516,-1.024430,1.095119,0.120996,0.266230,0.452599,0.208809,0.368883
3,ppluGFP,109742,31402,-6.873715,1.450456,-0.745411,0.381380,0.338360,0.118434,0.263811


LOFO diagnostic saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/leave_one_family_out_diagnostic.csv
Refitting final XGBoost models on all training families...
Final XGBoost models refit on all data and saved.
all-model delta feature summary saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/all_model_delta_feature_summary.csv


,xgb_uses_delta,gating_uses_delta,lightweight_uses_delta,train_esm_dim,train_saprot_dim,pocket_abs_dim,xgb_feature_dim,gating_esm_dim,gating_sap_dim,gating_phys_dim
0,True,True,True,1280,1280,1432,5424,1280,1280,2864


Epoch 01 train MSE 0.1547 val R2 0.4197 val RMSE 0.2192 lr 2.00e-04
Epoch 02 train MSE 0.2475 val R2 0.4865 val RMSE 0.2062 lr 3.00e-04
Epoch 03 train MSE 0.1934 val R2 0.5220 val RMSE 0.1989 lr 3.00e-04
Epoch 04 train MSE 0.1030 val R2 0.5357 val RMSE 0.1961 lr 2.99e-04
Epoch 05 train MSE 0.0393 val R2 0.5492 val RMSE 0.1932 lr 2.96e-04
Epoch 06 train MSE 0.0371 val R2 0.5819 val RMSE 0.1861 lr 2.92e-04
Epoch 07 train MSE 0.0353 val R2 0.5932 val RMSE 0.1835 lr 2.86e-04
Epoch 08 train MSE 0.0341 val R2 0.6041 val RMSE 0.1811 lr 2.78e-04
Epoch 09 train MSE 0.0326 val R2 0.6101 val RMSE 0.1797 lr 2.68e-04
Epoch 10 train MSE 0.0319 val R2 0.6273 val RMSE 0.1757 lr 2.58e-04
Epoch 11 train MSE 0.0307 val R2 0.6271 val RMSE 0.1757 lr 2.46e-04
Epoch 12 train MSE 0.0298 val R2 0.6282 val RMSE 0.1754 lr 2.32e-04
Epoch 13 train MSE 0.0287 val R2 0.6446 val RMSE 0.1716 lr 2.18e-04
Epoch 14 train MSE 0.0279 val R2 0.6458 val RMSE 0.1713 lr 2.04e-04
Epoch 15 train MSE 0.0271 val R2 0.6530 val RMSE

In [7]:
# ==========================================
# 训练 rank ensemble + 轻量异构模型，并定义统一预测函数
# ==========================================

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

rank_ensemble_model = None
stacking_rank_model = None

rank_val_cols = {}
if "pred_rank" in globals() and xgb_rank_model is not None:
    rank_val_cols["xgb_rank"] = np.clip(pred_rank, 0.0, 1.0)
if "pred_nn" in globals() and gating_model is not None:
    rank_val_cols["gnn_rank"] = np.clip(pred_nn, 0.0, 1.0)

if len(rank_val_cols) >= 2:
    val_pred_table = pd.DataFrame(rank_val_cols)
    val_pred_table["y_rank"] = y_rank[idx_val]

    # positive=True 强制非负权重，避免生物设计里难解释的“反向扣分模型”。
    rank_ensemble_model = LinearRegression(positive=True)
    rank_ensemble_model.fit(
        val_pred_table[["xgb_rank", "gnn_rank"]].values,
        val_pred_table["y_rank"].values,
    )

    print("rank ensemble intercept:", rank_ensemble_model.intercept_)
    print("rank ensemble weights:", dict(zip(["xgb_rank", "gnn_rank"], rank_ensemble_model.coef_)))
    joblib.dump(rank_ensemble_model, RANK_ENSEMBLE_MODEL_OUT)
else:
    print("rank ensemble skipped: 至少需要 XGBoost-rank 和 GatingNN-rank 两个验证集预测。")

# ==========================================
# 轻量模型 1：k-mer TF-IDF + Ridge
# ==========================================

def sequence_to_spaced_kmers(seq, k=3):
    seq = str(seq).strip().upper()
    return " ".join(seq[i:i + k] for i in range(max(0, len(seq) - k + 1)))


def sequence_delta_tokens(seq, family):
    """Encode mutations relative to the sequence's own-family WT as text tokens."""
    seq = str(seq).strip().upper()
    ref = reference_for_family(family)
    n = min(len(seq), len(ref))
    tokens = []
    for i in range(n):
        if seq[i] != ref[i]:
            tokens.extend([
                f"DELTA_{ref[i]}{i+1}{seq[i]}",
                f"POS_{i+1}",
                f"FROM_{ref[i]}",
                f"TO_{seq[i]}",
            ])
    if len(seq) != len(ref):
        tokens.append(f"LEN_DELTA_{len(seq)-len(ref)}")
    return " ".join(tokens) if tokens else "WT_DELTA_NONE"


def sequence_to_delta_kmer_text(seq, family, k=3):
    """K-mer text augmented with own-family WT delta tokens."""
    base = sequence_to_spaced_kmers(seq, k=k)
    if LIGHTWEIGHT_USE_DELTA:
        return base + " " + sequence_delta_tokens(seq, family)
    return base

kmer_text = [
    sequence_to_delta_kmer_text(seq, fam, k=3)
    for seq, fam in zip(train_df["full_sequence"], train_df["family"])
]

# 验证集预测必须只用 idx_train 训练，避免 k-mer 模型提前看见被留出的 family。
kmer_val_model = make_pipeline(
    TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=50000),
    Ridge(alpha=1.0),
)
kmer_val_model.fit([kmer_text[i] for i in idx_train], y_rank[idx_train])
kmer_val_pred = np.clip(kmer_val_model.predict([kmer_text[i] for i in idx_val]), 0.0, 1.0)
print(
    "K-mer Ridge FAMILY-RANK R2:",
    r2_score(y_rank[idx_val], kmer_val_pred),
    "RMSE:",
    np.sqrt(mean_squared_error(y_rank[idx_val], kmer_val_pred)),
)

# 候选预测用全量训练集重新拟合一遍。
kmer_rank_model = make_pipeline(
    TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=50000),
    Ridge(alpha=1.0),
)
kmer_rank_model.fit(kmer_text, y_rank)
joblib.dump(kmer_rank_model, RUN_OUTPUT_DIR / "kmer_tfidf_ridge_family_rank.joblib")

# ==========================================
# 轻量模型 2：position one-hot + Ridge
# ==========================================

AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_ORDER)}
ONEHOT_MAX_LEN = int(max(train_df["full_sequence"].map(len).max(), len(SF_GFP)))

def sequence_to_position_onehot(seq, max_len=ONEHOT_MAX_LEN):
    seq = str(seq).strip().upper()
    arr = np.zeros((max_len, len(AA_ORDER)), dtype=np.float32)
    for i, aa in enumerate(seq[:max_len]):
        j = AA_TO_IDX.get(aa)
        if j is not None:
            arr[i, j] = 1.0
    return arr.reshape(-1)


def sequence_to_position_delta_features(seq, family, max_len=ONEHOT_MAX_LEN):
    """Position features augmented with own-family WT one-hot delta."""
    abs_feat = sequence_to_position_onehot(seq, max_len=max_len)
    if not LIGHTWEIGHT_USE_DELTA:
        return abs_feat
    wt_feat = sequence_to_position_onehot(reference_for_family(family), max_len=max_len)
    delta_feat = abs_feat - wt_feat
    return np.concatenate([delta_feat, abs_feat], axis=0).astype(np.float32)

X_pos = np.vstack([
    sequence_to_position_delta_features(seq, fam)
    for seq, fam in zip(train_df["full_sequence"], train_df["family"])
])

# 同样只用 idx_train 训练验证模型，防止 position model 泄露被留出的 family。
position_val_model = make_pipeline(
    StandardScaler(with_mean=False),
    Ridge(alpha=10.0),
)
position_val_model.fit(X_pos[idx_train], y_rank[idx_train])
pos_val_pred = np.clip(position_val_model.predict(X_pos[idx_val]), 0.0, 1.0)
print(
    "Position one-hot Ridge FAMILY-RANK R2:",
    r2_score(y_rank[idx_val], pos_val_pred),
    "RMSE:",
    np.sqrt(mean_squared_error(y_rank[idx_val], pos_val_pred)),
)

# 候选预测用全量训练集重新拟合一遍。
position_rank_model = make_pipeline(
    StandardScaler(with_mean=False),
    Ridge(alpha=10.0),
)
position_rank_model.fit(X_pos, y_rank)
joblib.dump(position_rank_model, RUN_OUTPUT_DIR / "position_onehot_ridge_family_rank.joblib")

def add_lightweight_predictions(df, families=None):
    """添加 k-mer Ridge 与 position Ridge；二者都加入 own-family WT delta 特征。"""
    df = df.copy()
    seqs = df["Sequence"].astype(str).str.upper().tolist()
    if families is None:
        if "family" in df.columns:
            families = df["family"].map(normalize_family_name).tolist()
        else:
            families = ["sfGFP"] * len(df)
    families = [normalize_family_name(f) for f in families]

    if "kmer_rank_model" in globals() and kmer_rank_model is not None:
        texts = [sequence_to_delta_kmer_text(s, fam, k=3) for s, fam in zip(seqs, families)]
        df["Kmer_RankScore"] = np.clip(kmer_rank_model.predict(texts), 0.0, 1.0)

    if "position_rank_model" in globals() and position_rank_model is not None:
        X_local = np.vstack([sequence_to_position_delta_features(s, fam) for s, fam in zip(seqs, families)])
        df["Position_RankScore"] = np.clip(position_rank_model.predict(X_local), 0.0, 1.0)

    light_cols = [c for c in ["Kmer_RankScore", "Position_RankScore"] if c in df.columns]
    if light_cols:
        df["Lightweight_Consensus"] = df[light_cols].mean(axis=1)
    else:
        df["Lightweight_Consensus"] = np.nan

    return df


# ==========================================
# 四模型 stacking：XGB-rank + GNN-rank + k-mer + position one-hot
# ==========================================

stack_train = pd.DataFrame(index=np.arange(len(idx_val)))
if "pred_rank" in globals() and xgb_rank_model is not None:
    stack_train["XGBoost_RankScore"] = np.clip(pred_rank, 0.0, 1.0)
if "pred_nn" in globals() and gating_model is not None:
    stack_train["GatingNN_RankScore"] = np.clip(pred_nn, 0.0, 1.0)
if "kmer_val_pred" in globals():
    stack_train["Kmer_RankScore"] = np.clip(kmer_val_pred, 0.0, 1.0)
if "pos_val_pred" in globals():
    stack_train["Position_RankScore"] = np.clip(pos_val_pred, 0.0, 1.0)

STACKING_FEATURE_COLS = [
    c for c in [
        "XGBoost_RankScore",
        "GatingNN_RankScore",
        "Kmer_RankScore",
        "Position_RankScore",
    ]
    if c in stack_train.columns
]

if len(STACKING_FEATURE_COLS) >= 2:
    stacking_rank_model = LinearRegression(positive=True)
    stacking_rank_model.fit(stack_train[STACKING_FEATURE_COLS].values, y_rank[idx_val])
    stack_pred = np.clip(stacking_rank_model.predict(stack_train[STACKING_FEATURE_COLS].values), 0.0, 1.0)
    print("stacking rank model features:", STACKING_FEATURE_COLS)
    print("stacking rank intercept:", stacking_rank_model.intercept_)
    print("stacking rank weights:", dict(zip(STACKING_FEATURE_COLS, stacking_rank_model.coef_)))
    print(
        "Stacking FAMILY-RANK R2:",
        r2_score(y_rank[idx_val], stack_pred),
        "RMSE:",
        np.sqrt(mean_squared_error(y_rank[idx_val], stack_pred)),
    )
    joblib.dump(
        {"model": stacking_rank_model, "feature_cols": STACKING_FEATURE_COLS},
        RUN_OUTPUT_DIR / "four_model_stacking_family_rank.joblib",
    )
else:
    print("stacking skipped: available rank model count < 2")

# ==========================================
# 数据驱动最终亮度排序模型
# ==========================================

FINAL_BRIGHTNESS_FEATURE_COLS = [
    "Raw_Brightness_Percentile",
    "Heavy_Rank_Ensemble",
    "Lightweight_Consensus",
    "Model_Agreement",
]


def percentile_against_reference(values, reference_values):
    """把预测值转成相对于训练集标签分布的分位数，避免 raw value 尺度漂移。"""
    ref = np.asarray(reference_values, dtype=np.float32)
    ref = ref[np.isfinite(ref)]
    arr = np.asarray(values, dtype=np.float32)
    if len(ref) == 0:
        return np.full(arr.shape, 0.5, dtype=np.float32)
    ref = np.sort(ref)
    return (np.searchsorted(ref, arr, side="right") / len(ref)).astype(np.float32)


def model_agreement_from_columns(frame, cols):
    """把多个模型的标准差转成 0-1 agreement；越接近 1 表示模型越一致。"""
    available = [c for c in cols if c in frame.columns]
    if len(available) < 2:
        return np.ones(len(frame), dtype=np.float32)
    std = frame[available].astype(float).std(axis=1).fillna(0.0).values
    return np.clip(1.0 - std / 0.5, 0.0, 1.0).astype(np.float32)


def build_final_brightness_feature_frame(df):
    """构造 final brightness meta model 使用的可解释特征。"""
    out = pd.DataFrame(index=df.index)
    if "XGBoost_Brightness" in df.columns:
        out["Raw_Brightness_Percentile"] = percentile_against_reference(df["XGBoost_Brightness"].values, y_raw)
    else:
        out["Raw_Brightness_Percentile"] = 0.5

    out["Heavy_Rank_Ensemble"] = df.get("Heavy_Rank_Ensemble", pd.Series(0.5, index=df.index)).fillna(0.5).astype(float)
    out["Lightweight_Consensus"] = df.get("Lightweight_Consensus", pd.Series(0.5, index=df.index)).fillna(out["Heavy_Rank_Ensemble"]).astype(float)
    out["Model_Agreement"] = model_agreement_from_columns(
        df,
        ["XGBoost_RankScore", "GatingNN_RankScore", "Kmer_RankScore", "Position_RankScore"],
    )
    return out[FINAL_BRIGHTNESS_FEATURE_COLS].astype(np.float32)


final_brightness_meta_model = None

# 用验证集上的模型输出学习最终亮度排序权重；这一步替代手工 0.50/0.25/0.15/0.10 加权。
final_val = pd.DataFrame(index=np.arange(len(idx_val)))
if "pred_raw" in globals():
    final_val["XGBoost_Brightness"] = pred_raw
if "pred_rank" in globals():
    final_val["XGBoost_RankScore"] = np.clip(pred_rank, 0.0, 1.0)
if "pred_nn" in globals():
    final_val["GatingNN_RankScore"] = np.clip(pred_nn, 0.0, 1.0)
if "kmer_val_pred" in globals():
    final_val["Kmer_RankScore"] = np.clip(kmer_val_pred, 0.0, 1.0)
if "pos_val_pred" in globals():
    final_val["Position_RankScore"] = np.clip(pos_val_pred, 0.0, 1.0)

if "rank_ensemble_model" in globals() and rank_ensemble_model is not None and {"XGBoost_RankScore", "GatingNN_RankScore"}.issubset(final_val.columns):
    final_val["Heavy_Rank_Ensemble"] = np.clip(
        rank_ensemble_model.predict(final_val[["XGBoost_RankScore", "GatingNN_RankScore"]].values),
        0.0,
        1.0,
    )
elif {"XGBoost_RankScore", "GatingNN_RankScore"}.issubset(final_val.columns):
    final_val["Heavy_Rank_Ensemble"] = final_val[["XGBoost_RankScore", "GatingNN_RankScore"]].mean(axis=1)
elif "XGBoost_RankScore" in final_val.columns:
    final_val["Heavy_Rank_Ensemble"] = final_val["XGBoost_RankScore"]
elif "GatingNN_RankScore" in final_val.columns:
    final_val["Heavy_Rank_Ensemble"] = final_val["GatingNN_RankScore"]

light_cols = [c for c in ["Kmer_RankScore", "Position_RankScore"] if c in final_val.columns]
if light_cols:
    final_val["Lightweight_Consensus"] = final_val[light_cols].mean(axis=1)
else:
    final_val["Lightweight_Consensus"] = final_val.get("Heavy_Rank_Ensemble", pd.Series(0.5, index=final_val.index))

if len(final_val) and "XGBoost_Brightness" in final_val.columns and "Heavy_Rank_Ensemble" in final_val.columns:
    X_final_val = build_final_brightness_feature_frame(final_val)
    final_brightness_meta_model = LinearRegression(positive=True)
    final_brightness_meta_model.fit(X_final_val.values, y_rank[idx_val])
    final_val_pred = np.clip(final_brightness_meta_model.predict(X_final_val.values), 0.0, 1.0)
    print("final brightness meta features:", FINAL_BRIGHTNESS_FEATURE_COLS)
    print("final brightness meta intercept:", final_brightness_meta_model.intercept_)
    print("final brightness meta weights:", dict(zip(FINAL_BRIGHTNESS_FEATURE_COLS, final_brightness_meta_model.coef_)))
    print(
        "Final brightness meta FAMILY-RANK R2:",
        r2_score(y_rank[idx_val], final_val_pred),
        "RMSE:",
        np.sqrt(mean_squared_error(y_rank[idx_val], final_val_pred)),
    )
    joblib.dump(
        {"model": final_brightness_meta_model, "feature_cols": FINAL_BRIGHTNESS_FEATURE_COLS},
        FINAL_BRIGHTNESS_META_MODEL_OUT,
    )
else:
    print("final brightness meta model skipped: validation prediction columns are incomplete.")


def add_data_driven_final_brightness(df):
    """给任意预测表添加验证集学习得到的最终亮度分。"""
    df = df.copy()
    feature_frame = build_final_brightness_feature_frame(df)
    for col in FINAL_BRIGHTNESS_FEATURE_COLS:
        df[col] = feature_frame[col].values

    if "final_brightness_meta_model" in globals() and final_brightness_meta_model is not None:
        learned = final_brightness_meta_model.predict(feature_frame.values)
        df["Data_Driven_Brightness_Score"] = np.clip(learned, 0.0, 1.0)
    else:
        df["Data_Driven_Brightness_Score"] = np.clip(
            0.5 * feature_frame["Raw_Brightness_Percentile"]
            + 0.3 * feature_frame["Heavy_Rank_Ensemble"]
            + 0.2 * feature_frame["Lightweight_Consensus"],
            0.0,
            1.0,
        )
    return df

def add_brightness_predictions(df, X, esm, sap, pocket_features, families=None):
    """给序列表添加统一亮度预测列。

    Heavy models:
    - XGBoost_Brightness: raw brightness，用于 WT-relative 参考。
    - XGBoost_RankScore / GatingNN_RankScore: ESM/SaProt/pocket 的重模型 rank 分。

    Light models:
    - Kmer_RankScore: k-mer TF-IDF + Ridge。
    - Position_RankScore: position one-hot + Ridge。

    Brightness_Rescore 是重模型 rank ensemble 与轻模型 consensus 的融合分。
    """
    df = df.copy()
    rank_preds = []

    if "xgb_raw_model" in globals() and xgb_raw_model is not None:
        df["XGBoost_Brightness"] = xgb_raw_model.predict(X)
    elif "xgb_model" in globals() and xgb_model is not None:
        df["XGBoost_Brightness"] = xgb_model.predict(X)

    if "xgb_rank_model" in globals() and xgb_rank_model is not None:
        xgb_rank = np.clip(xgb_rank_model.predict(X), 0.0, 1.0)
        df["XGBoost_RankScore"] = xgb_rank
        rank_preds.append(xgb_rank)

    if "gating_model" in globals() and gating_model is not None:
        gating_model.eval()
        with torch.no_grad():
            if families is None:
                if "family" in df.columns:
                    families = df["family"].map(normalize_family_name).tolist()
                else:
                    families = ["sfGFP"] * len(df)
            g_esm, g_sap, g_phys = build_gating_inputs(esm, sap, pocket_features, families)
            esm_scaled = sc_esm.transform(g_esm).astype(np.float32)
            sap_scaled = sc_sap.transform(g_sap).astype(np.float32)
            phys_scaled = sc_phys.transform(g_phys).astype(np.float32)

            gnn_rank = gating_model(
                torch.tensor(esm_scaled, dtype=torch.float32).to(device),
                torch.tensor(sap_scaled, dtype=torch.float32).to(device),
                torch.tensor(phys_scaled, dtype=torch.float32).to(device),
            ).detach().cpu().numpy().reshape(-1)

        gnn_rank = np.clip(gnn_rank, 0.0, 1.0)
        df["GatingNN_RankScore"] = gnn_rank
        rank_preds.append(gnn_rank)

    if not rank_preds:
        raise RuntimeError("没有可用 rank 亮度模型：xgb_rank_model 和 gating_model 都为空。")

    rank_arr = np.vstack(rank_preds)

    if (
        "rank_ensemble_model" in globals()
        and rank_ensemble_model is not None
        and {"XGBoost_RankScore", "GatingNN_RankScore"}.issubset(df.columns)
    ):
        heavy_score = rank_ensemble_model.predict(df[["XGBoost_RankScore", "GatingNN_RankScore"]].values)
        df["Heavy_Rank_Ensemble"] = np.clip(heavy_score, 0.0, 1.0)
    else:
        df["Heavy_Rank_Ensemble"] = np.clip(rank_arr.mean(axis=0), 0.0, 1.0)

    df = add_lightweight_predictions(df, families=families)

    if (
        "stacking_rank_model" in globals()
        and stacking_rank_model is not None
        and "STACKING_FEATURE_COLS" in globals()
        and all(c in df.columns for c in STACKING_FEATURE_COLS)
    ):
        df["Brightness_Rescore"] = np.clip(
            stacking_rank_model.predict(df[STACKING_FEATURE_COLS].values),
            0.0,
            1.0,
        )
    elif df["Lightweight_Consensus"].notna().any():
        df["Brightness_Rescore"] = np.clip(
            HEAVY_MODEL_WEIGHT * df["Heavy_Rank_Ensemble"].fillna(0.0)
            + LIGHT_MODEL_WEIGHT * df["Lightweight_Consensus"].fillna(df["Heavy_Rank_Ensemble"]),
            0.0,
            1.0,
        )
    else:
        df["Brightness_Rescore"] = df["Heavy_Rank_Ensemble"]

    uncertainty_sources = [rank_arr]
    light_cols = [c for c in ["Kmer_RankScore", "Position_RankScore"] if c in df.columns]
    if light_cols:
        light_arr = df[light_cols].values.T
        uncertainty_sources.append(light_arr)
    all_rank_arr = np.vstack(uncertainty_sources)

    df["Brightness_Uncertainty"] = all_rank_arr.std(axis=0) if all_rank_arr.shape[0] > 1 else 0.0
    df["Relative_Uncertainty"] = (
        df["Brightness_Uncertainty"]
        / df["Brightness_Rescore"].abs().clip(lower=1e-6)
    )

    # Data_Driven_Brightness_Score 是验证集学习得到的最终亮度排序分；
    # Brightness_Rescore 仍保留为 rank/consensus 诊断列。
    df = add_data_driven_final_brightness(df)

    # 统一所有预测表的最终分数列，避免只有 cand 表有 Final_Learned_Score、
    # 而 WT / beforetopseqs / 单点扫描表缺列导致后续排序 KeyError。
    if "Final_Learned_Score" not in df.columns:
        if "Data_Driven_Brightness_Score" in df.columns:
            df["Final_Learned_Score"] = df["Data_Driven_Brightness_Score"]
        else:
            df["Final_Learned_Score"] = df["Brightness_Rescore"]
    if "Final_Rescore" not in df.columns:
        df["Final_Rescore"] = df["Final_Learned_Score"]

    return df


rank ensemble intercept: -0.006497681
rank ensemble weights: {'xgb_rank': 0.05287216, 'gnn_rank': 0.9774788}


K-mer Ridge FAMILY-RANK R2: 0.7097663562975511 RMSE: 0.15502262793130195
Position one-hot Ridge FAMILY-RANK R2: 0.5963758230209351 RMSE: 0.18281415017516214
stacking rank model features: ['XGBoost_RankScore', 'GatingNN_RankScore', 'Kmer_RankScore', 'Position_RankScore']
stacking rank intercept: -0.06410049081017022
stacking rank weights: {'XGBoost_RankScore': 0.0, 'GatingNN_RankScore': 0.5238954017575367, 'Kmer_RankScore': 0.6071433276560094, 'Position_RankScore': 0.0}
Stacking FAMILY-RANK R2: 0.7637270411099628 RMSE: 0.13987117644091715
final brightness meta features: ['Raw_Brightness_Percentile', 'Heavy_Rank_Ensemble', 'Lightweight_Consensus', 'Model_Agreement']
final brightness meta intercept: -0.22911757
final brightness meta weights: {'Raw_Brightness_Percentile': 0.0, 'Heavy_Rank_Ensemble': 0.568789, 'Lightweight_Consensus': 0.56280077, 'Model_Agreement': 0.18952154}
Final brightness meta FAMILY-RANK R2: 0.7556614279747009 RMSE: 0.14223851949651758


In [8]:

# ==========================================
# 7b. 全模型 leave-one-family-out 泛化审计与跨 family 集成权重
# ==========================================

def safe_spearman(y_true, y_pred):
    """Spearman correlation with NaN protection."""
    try:
        value = pd.Series(np.asarray(y_true, dtype=float)).corr(
            pd.Series(np.asarray(y_pred, dtype=float)),
            method="spearman",
        )
        return float(0.0 if pd.isna(value) else value)
    except Exception:
        return 0.0


def topk_recall(y_true, y_pred, top_frac=0.10):
    """Recall of true top-fraction sequences recovered by predicted top-fraction."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) == 0:
        return np.nan
    k = max(1, int(np.ceil(len(y_true) * float(top_frac))))
    return float(len(set(np.argsort(-y_true)[:k]) & set(np.argsort(-y_pred)[:k])) / k)


def ndcg_at_k(y_true, y_pred, k=100):
    """NDCG@k for ranking diagnostics."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) == 0:
        return np.nan
    k = min(int(k), len(y_true))
    order = np.argsort(-y_pred)[:k]
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    dcg = float(np.sum(np.maximum(y_true[order], 0.0) * discounts))
    ideal = np.sort(np.maximum(y_true, 0.0))[::-1][:k]
    idcg = float(np.sum(ideal * discounts))
    return dcg / idcg if idcg > 1e-12 else 0.0


def lofo_metric_row(heldout_family, model_name, y_true, y_pred, n_train, error=""):
    """Build one standardized LOFO metric row."""
    if error:
        return {
            "heldout_family": heldout_family,
            "model_name": model_name,
            "n_train": int(n_train),
            "n_val": int(len(y_true)),
            "r2": np.nan,
            "rmse": np.nan,
            "spearman": np.nan,
            "top10_recall": np.nan,
            "top20_recall": np.nan,
            "ndcg100": np.nan,
            "error": error,
        }
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "heldout_family": heldout_family,
        "model_name": model_name,
        "n_train": int(n_train),
        "n_val": int(len(y_true)),
        "r2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))) if len(y_true) else np.nan,
        "spearman": safe_spearman(y_true, y_pred),
        "top10_recall": topk_recall(y_true, y_pred, 0.10),
        "top20_recall": topk_recall(y_true, y_pred, 0.20),
        "ndcg100": ndcg_at_k(y_true, y_pred, 100),
        "error": "",
    }


def fit_predict_gating_lofo(train_idx, val_idx):
    """Train a lightweight GatingNN in one LOFO fold and return validation predictions."""
    rng_local = np.random.default_rng(RANDOM_SEED)
    fit_idx = np.asarray(train_idx, dtype=int)
    if LOFO_GNN_MAX_TRAIN_ROWS is not None and len(fit_idx) > int(LOFO_GNN_MAX_TRAIN_ROWS):
        fit_idx = rng_local.choice(fit_idx, size=int(LOFO_GNN_MAX_TRAIN_ROWS), replace=False)

    lofo_sc_esm = StandardScaler()
    lofo_sc_sap = StandardScaler()
    lofo_sc_phys = StandardScaler()
    lofo_g_esm, lofo_g_sap, lofo_g_phys = build_gating_inputs(train_esm, train_saprot, train_pocket_features, train_df["family"].tolist())
    e_train = lofo_sc_esm.fit_transform(lofo_g_esm[fit_idx])
    s_train = lofo_sc_sap.fit_transform(lofo_g_sap[fit_idx])
    p_train = lofo_sc_phys.fit_transform(lofo_g_phys[fit_idx])
    e_val = lofo_sc_esm.transform(lofo_g_esm[val_idx])
    s_val = lofo_sc_sap.transform(lofo_g_sap[val_idx])
    p_val = lofo_sc_phys.transform(lofo_g_phys[val_idx])

    ds = TensorDataset(
        torch.tensor(e_train, dtype=torch.float32),
        torch.tensor(s_train, dtype=torch.float32),
        torch.tensor(p_train, dtype=torch.float32),
        torch.tensor(y_rank[fit_idx].reshape(-1, 1), dtype=torch.float32),
        torch.tensor(family_sample_weight[fit_idx].reshape(-1, 1), dtype=torch.float32),
    )
    loader_generator = torch.Generator()
    loader_generator.manual_seed(RANDOM_SEED)
    loader = DataLoader(ds, batch_size=512, shuffle=True, generator=loader_generator)
    model = PhysicsEnhancedGatingNet(
        esm_dim=lofo_g_esm.shape[1],
        sap_dim=lofo_g_sap.shape[1],
        phys_dim=lofo_g_phys.shape[1],
        dropout=GATING_DROPOUT,
        residual_scale=GATING_RESIDUAL_SCALE,
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=GATING_LR, weight_decay=1e-4)
    model.train()
    for epoch in range(1, int(LOFO_GNN_EPOCHS) + 1):
        losses = []
        for eb, sb, pb, yb, wb in loader:
            eb, sb, pb, yb, wb = eb.to(device), sb.to(device), pb.to(device), yb.to(device), wb.to(device)
            opt.zero_grad()
            pred = model(eb, sb, pb)
            loss = ((pred - yb) ** 2 * wb).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GATING_GRAD_CLIP_NORM)
            opt.step()
            losses.append(float(loss.detach().cpu()))
        print(f"  GatingNN LOFO epoch {epoch}/{LOFO_GNN_EPOCHS} MSE {np.mean(losses):.4f}")
    model.eval()
    with torch.no_grad():
        pred_val = model(
            torch.tensor(e_val, dtype=torch.float32).to(device),
            torch.tensor(s_val, dtype=torch.float32).to(device),
            torch.tensor(p_val, dtype=torch.float32).to(device),
        ).detach().cpu().numpy().reshape(-1)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.clip(pred_val, 0.0, 1.0)


def run_lofo_submodel_audit():
    """Evaluate all brightness submodels by leave-one-family-out diagnostics."""
    if not RUN_LOFO_SUBMODEL_AUDIT:
        print("LOFO submodel audit skipped by RUN_LOFO_SUBMODEL_AUDIT=False")
        return pd.DataFrame(), pd.DataFrame()

    rows = []
    families = sorted(pd.unique(train_df["family"]))
    for heldout in families:
        train_idx_fold = np.where(train_df["family"].values != heldout)[0]
        val_idx_fold = np.where(train_df["family"].values == heldout)[0]
        y_val_fold = y_rank[val_idx_fold]
        print(f"LOFO all-model audit: hold out {heldout}, train={len(train_idx_fold)}, val={len(val_idx_fold)}")
        fold_pred = {}

        try:
            raw_model = make_xgb_model(n_estimators=LOFO_XGB_ESTIMATORS)
            raw_model.fit(
                X_full[train_idx_fold],
                y_raw[train_idx_fold],
                sample_weight=family_sample_weight[train_idx_fold],
                verbose=False,
            )
            raw_pred = raw_model.predict(X_full[val_idx_fold])
            fold_pred["XGBoost_Raw"] = percentile_against_reference(raw_pred, raw_model.predict(X_full[train_idx_fold]))
        except Exception as e:
            rows.append(lofo_metric_row(heldout, "XGBoost_Raw", y_val_fold, [], len(train_idx_fold), repr(e)))

        try:
            rank_model = make_xgb_model(n_estimators=LOFO_XGB_ESTIMATORS)
            rank_model.fit(
                X_full[train_idx_fold],
                y_rank[train_idx_fold],
                sample_weight=family_sample_weight[train_idx_fold],
                verbose=False,
            )
            fold_pred["XGBoost_Rank"] = np.clip(rank_model.predict(X_full[val_idx_fold]), 0.0, 1.0)
        except Exception as e:
            rows.append(lofo_metric_row(heldout, "XGBoost_Rank", y_val_fold, [], len(train_idx_fold), repr(e)))

        try:
            if RUN_LOFO_FULL:
                fold_pred["GatingNN_Rank"] = fit_predict_gating_lofo(train_idx_fold, val_idx_fold)
            else:
                raise RuntimeError("skipped because RUN_LOFO_FULL=False")
        except Exception as e:
            rows.append(lofo_metric_row(heldout, "GatingNN_Rank", y_val_fold, [], len(train_idx_fold), repr(e)))

        try:
            km = make_pipeline(
                TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=50000),
                Ridge(alpha=1.0),
            )
            km.fit([kmer_text[i] for i in train_idx_fold], y_rank[train_idx_fold])
            fold_pred["Kmer_Ridge"] = np.clip(km.predict([kmer_text[i] for i in val_idx_fold]), 0.0, 1.0)
        except Exception as e:
            rows.append(lofo_metric_row(heldout, "Kmer_Ridge", y_val_fold, [], len(train_idx_fold), repr(e)))

        try:
            pm = make_pipeline(StandardScaler(with_mean=False), Ridge(alpha=10.0))
            pm.fit(X_pos[train_idx_fold], y_rank[train_idx_fold])
            fold_pred["Position_Ridge"] = np.clip(pm.predict(X_pos[val_idx_fold]), 0.0, 1.0)
        except Exception as e:
            rows.append(lofo_metric_row(heldout, "Position_Ridge", y_val_fold, [], len(train_idx_fold), repr(e)))

        if {"XGBoost_Rank", "GatingNN_Rank"}.issubset(fold_pred):
            fold_pred["Rank_Ensemble_XGB_GNN"] = np.vstack([
                fold_pred["XGBoost_Rank"],
                fold_pred["GatingNN_Rank"],
            ]).mean(axis=0)

        four_cols = [c for c in ["XGBoost_Rank", "GatingNN_Rank", "Kmer_Ridge", "Position_Ridge"] if c in fold_pred]
        if four_cols:
            fold_pred["FourModel_Stacking"] = np.vstack([fold_pred[c] for c in four_cols]).mean(axis=0)

        final_cols = [c for c in ["XGBoost_Raw", "Rank_Ensemble_XGB_GNN", "FourModel_Stacking", "Kmer_Ridge"] if c in fold_pred]
        if final_cols:
            fold_pred["Final_Brightness_Meta"] = np.vstack([fold_pred[c] for c in final_cols]).mean(axis=0)

        for model_name, pred in fold_pred.items():
            rows.append(lofo_metric_row(heldout, model_name, y_val_fold, pred, len(train_idx_fold)))

    metrics = pd.DataFrame(rows)
    metrics.to_csv(LOFO_SUBMODEL_METRICS_OUT, index=False)
    ok = metrics[metrics["error"].fillna("") == ""].copy()
    if ok.empty:
        summary = pd.DataFrame()
    else:
        summary = ok.groupby("model_name", as_index=False).agg(
            mean_r2=("r2", "mean"),
            mean_rmse=("rmse", "mean"),
            mean_spearman=("spearman", "mean"),
            mean_top10_recall=("top10_recall", "mean"),
            mean_top20_recall=("top20_recall", "mean"),
            mean_ndcg100=("ndcg100", "mean"),
        )
        summary["cross_family_score"] = (
            0.50 * summary["mean_spearman"].fillna(0.0)
            + 0.30 * (summary["mean_top10_recall"].fillna(0.0) - 0.10)
            + 0.20 * (summary["mean_top20_recall"].fillna(0.0) - 0.20)
        ).clip(lower=0.0)
        summary = summary.sort_values(["cross_family_score", "mean_spearman", "mean_top10_recall"], ascending=False)
    summary.to_csv(LOFO_SUBMODEL_SUMMARY_OUT, index=False)
    print("LOFO submodel metrics saved:", LOFO_SUBMODEL_METRICS_OUT)
    print("LOFO submodel summary saved:", LOFO_SUBMODEL_SUMMARY_OUT)
    display(summary)
    return metrics, summary


def compute_cross_family_weights(lofo_summary):
    """Compute ensemble weights from maximum cross-family generalization score."""
    model_to_score_col = {
        "XGBoost_Raw": "XGBoost_Raw_Score",
        "XGBoost_Rank": "XGBoost_RankScore",
        "GatingNN_Rank": "GatingNN_RankScore",
        "Kmer_Ridge": "Kmer_RankScore",
        "Position_Ridge": "Position_RankScore",
        "Rank_Ensemble_XGB_GNN": "Heavy_Rank_Ensemble",
        "FourModel_Stacking": "Brightness_Rescore",
        "Final_Brightness_Meta": "Data_Driven_Brightness_Score",
    }
    if lofo_summary is None or lofo_summary.empty:
        out = pd.DataFrame({"model_name": ["Brightness_Rescore"], "score_column": ["Brightness_Rescore"], "cross_family_score": [1.0]})
    else:
        out = lofo_summary[["model_name", "cross_family_score", "mean_spearman", "mean_top10_recall", "mean_top20_recall"]].copy()
        out["score_column"] = out["model_name"].map(model_to_score_col)
        out = out.dropna(subset=["score_column"])
        out["cross_family_score"] = out["cross_family_score"].clip(lower=0.0)
        if out.empty or out["cross_family_score"].sum() <= 0:
            best = lofo_summary.sort_values(["mean_spearman", "mean_top10_recall"], ascending=False).head(1).copy()
            best["score_column"] = best["model_name"].map(model_to_score_col)
            best["cross_family_score"] = 1.0
            out = best.dropna(subset=["score_column"])
    total = float(out["cross_family_score"].sum())
    out["weight"] = out["cross_family_score"] / total if total > 0 else 1.0 / max(1, len(out))
    out["is_best_by_cross_family_score"] = out["cross_family_score"] == out["cross_family_score"].max()
    out.to_csv(CROSSFAMILY_MODEL_WEIGHTS_OUT, index=False)
    print("cross-family ensemble weights saved:", CROSSFAMILY_MODEL_WEIGHTS_OUT)
    display(out)
    return out


def apply_crossfamily_ensemble_scores(df):
    """Add CrossFamily_EnsembleScore using LOFO-derived model weights."""
    df = df.copy()
    if "XGBoost_Raw_Score" not in df.columns and "XGBoost_Raw_Relative_to_WT" in df.columns:
        df["XGBoost_Raw_Score"] = np.clip(df["XGBoost_Raw_Relative_to_WT"].astype(float), 0.0, 1.5) / 1.5
    if "crossfamily_model_weights" not in globals() or crossfamily_model_weights.empty:
        df["CrossFamily_EnsembleScore"] = df.get("Brightness_Rescore", pd.Series(0.5, index=df.index))
        df["CrossFamily_Weight_Coverage"] = 0.0
        return df
    score = np.zeros(len(df), dtype=np.float32)
    used = 0.0
    missing = []
    for _, row in crossfamily_model_weights.iterrows():
        col = row["score_column"]
        weight = float(row["weight"])
        if col in df.columns:
            score += weight * df[col].fillna(0.0).astype(float).values
            used += weight
        else:
            missing.append(col)
    if missing:
        print("warning: cross-family ensemble skipped missing columns:", sorted(set(missing)))
    if used > 0:
        df["CrossFamily_EnsembleScore"] = np.clip(score / used, 0.0, 1.0)
    else:
        df["CrossFamily_EnsembleScore"] = df.get("Brightness_Rescore", pd.Series(0.5, index=df.index))
    df["CrossFamily_Weight_Coverage"] = used
    return df


lofo_submodel_metrics, lofo_submodel_summary = run_lofo_submodel_audit()
crossfamily_model_weights = compute_cross_family_weights(lofo_submodel_summary)


LOFO all-model audit: hold out amacGFP, train=107633, val=33511


  GatingNN LOFO epoch 1/5 MSE 0.5174
  GatingNN LOFO epoch 2/5 MSE 0.4010
  GatingNN LOFO epoch 3/5 MSE 0.2931
  GatingNN LOFO epoch 4/5 MSE 0.0920
  GatingNN LOFO epoch 5/5 MSE 0.3923
LOFO all-model audit: hold out avGFP, train=89429, val=51715
  GatingNN LOFO epoch 1/5 MSE 0.3761
  GatingNN LOFO epoch 2/5 MSE 0.6503
  GatingNN LOFO epoch 3/5 MSE 0.8740
  GatingNN LOFO epoch 4/5 MSE 0.7186
  GatingNN LOFO epoch 5/5 MSE 0.0767
LOFO all-model audit: hold out cgreGFP, train=116628, val=24516
  GatingNN LOFO epoch 1/5 MSE 0.5238
  GatingNN LOFO epoch 2/5 MSE 0.9957
  GatingNN LOFO epoch 3/5 MSE 0.4186
  GatingNN LOFO epoch 4/5 MSE 0.2688
  GatingNN LOFO epoch 5/5 MSE 0.0820


/home/liuchang/anaconda3/envs/GFP/lib/python3.11/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


LOFO all-model audit: hold out ppluGFP, train=109742, val=31402
  GatingNN LOFO epoch 1/5 MSE 0.0727
  GatingNN LOFO epoch 2/5 MSE 0.0506
  GatingNN LOFO epoch 3/5 MSE 0.0454
  GatingNN LOFO epoch 4/5 MSE 0.0424
  GatingNN LOFO epoch 5/5 MSE 0.0397


/home/liuchang/anaconda3/envs/GFP/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.7689246734065023e-09.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


LOFO submodel metrics saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/lofo_submodel_metrics.csv
LOFO submodel summary saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/lofo_submodel_summary.csv


/home/liuchang/anaconda3/envs/GFP/lib/python3.11/site-packages/pandas/core/nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


,model_name,mean_r2,mean_rmse,mean_spearman,mean_top10_recall,mean_top20_recall,mean_ndcg100,cross_family_score
1,FourModel_Stacking,-0.577352,0.352196,0.444764,0.203689,0.358029,0.634085,0.285095
0,Final_Brightness_Meta,-0.656545,0.365625,0.438952,0.199383,0.351540,0.679851,0.279599
6,XGBoost_Rank,-0.128204,0.302159,0.390211,0.171897,0.327162,0.707821,0.242107
7,XGBoost_Raw,-0.858979,0.388414,0.367440,0.173621,0.318299,0.698602,0.229466
5,Rank_Ensemble_XGB_GNN,-0.164184,0.306020,0.358213,0.170110,0.317580,0.605134,0.223655
3,Kmer_Ridge,-2.074209,0.494499,0.264306,0.149641,0.287118,0.580919,0.164469
4,Position_Ridge,-1.852955,0.467984,0.243705,0.145363,0.285755,0.520913,0.152613
2,GatingNN_Rank,-0.568737,0.357354,0.240108,0.141751,0.274772,0.503853,0.147534


cross-family ensemble weights saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/crossfamily_model_weights.csv


,model_name,cross_family_score,mean_spearman,mean_top10_recall,mean_top20_recall,score_column,weight,is_best_by_cross_family_score
1,FourModel_Stacking,0.285095,0.444764,0.203689,0.358029,Brightness_Rescore,0.165317,True
0,Final_Brightness_Meta,0.279599,0.438952,0.199383,0.351540,Data_Driven_Brightness_Score,0.162130,False
6,XGBoost_Rank,0.242107,0.390211,0.171897,0.327162,XGBoost_RankScore,0.140390,False
7,XGBoost_Raw,0.229466,0.367440,0.173621,0.318299,XGBoost_Raw_Score,0.133060,False
5,Rank_Ensemble_XGB_GNN,0.223655,0.358213,0.170110,0.317580,Heavy_Rank_Ensemble,0.129690,False
3,Kmer_Ridge,0.164469,0.264306,0.149641,0.287118,Kmer_RankScore,0.095370,False
4,Position_Ridge,0.152613,0.243705,0.145363,0.285755,Position_RankScore,0.088495,False
2,GatingNN_Rank,0.147534,0.240108,0.141751,0.274772,GatingNN_RankScore,0.085550,False


In [9]:

# ==========================================
# 7c. Family subset comparison: all vs exclude pplu vs exclude pplu+cgre
# ==========================================

def family_subset_indices(mode):
    """Return row indices for a requested family subset comparison mode."""
    mode = str(mode)
    fams = train_df["family"].map(normalize_family_name).values
    if mode == "all":
        keep = np.ones(len(train_df), dtype=bool)
    elif mode == "exclude_pplu":
        keep = fams != "ppluGFP"
    elif mode == "exclude_pplu_cgre":
        keep = ~np.isin(fams, ["ppluGFP", "cgreGFP"])
    else:
        raise ValueError(f"unknown family subset mode: {mode}")
    idx = np.where(keep)[0]
    print(f"family subset {mode}: rows={len(idx)} families={sorted(pd.unique(fams[idx]))}")
    return idx


def stratified_subset_split(indices, val_fraction=VALIDATION_FRACTION):
    """Within-family stratified split for one family subset."""
    rng = np.random.default_rng(RANDOM_SEED)
    indices = np.asarray(indices, dtype=int)
    fams = train_df["family"].values
    val = []
    for fam in sorted(pd.unique(fams[indices])):
        fam_idx = indices[fams[indices] == fam]
        values = y_raw[fam_idx]
        n_bins = min(10, max(2, len(fam_idx) // 2000))
        ranks = pd.Series(values).rank(method="first")
        try:
            bins = pd.qcut(ranks, q=n_bins, labels=False, duplicates="drop")
        except Exception:
            bins = pd.Series(np.zeros(len(fam_idx), dtype=int))
        tmp = pd.DataFrame({"idx": fam_idx, "bin": np.asarray(bins, dtype=int)})
        for _bin, sub in tmp.groupby("bin"):
            n_val = max(1, int(round(len(sub) * val_fraction)))
            n_val = min(n_val, len(sub))
            val.extend(rng.choice(sub["idx"].values, size=n_val, replace=False).tolist())
    val = np.array(sorted(set(val)), dtype=int)
    train = np.setdiff1d(indices, val)
    return train, val


def subset_family_weights(indices):
    """Family-balanced sample weights within a subset."""
    fams = train_df.iloc[indices]["family"].values
    counts = pd.Series(fams).value_counts().to_dict()
    weights = np.array([1.0 / counts[f] for f in fams], dtype=np.float32)
    return weights / max(float(weights.mean()), 1e-12)


def evaluate_family_subset_mode(mode):
    """Train quick XGBoost/K-mer/Position models for one family-subset mode."""
    indices = family_subset_indices(mode)
    tr, va = stratified_subset_split(indices)
    tr_weights = subset_family_weights(tr)
    rows = []

    def add_row(model_name, pred):
        rows.append({
            "mode": mode,
            "model_name": model_name,
            "train_rows": int(len(tr)),
            "val_rows": int(len(va)),
            "families": ";".join(sorted(pd.unique(train_df.iloc[indices]["family"]))),
            "r2": float(r2_score(y_rank[va], pred)),
            "rmse": float(np.sqrt(mean_squared_error(y_rank[va], pred))),
            "spearman": safe_spearman(y_rank[va], pred) if "safe_spearman" in globals() else float(pd.Series(y_rank[va]).corr(pd.Series(pred), method="spearman")),
            "top10_recall": topk_recall(y_rank[va], pred, 0.10) if "topk_recall" in globals() else np.nan,
            "top20_recall": topk_recall(y_rank[va], pred, 0.20) if "topk_recall" in globals() else np.nan,
        })

    xgb_rank_cmp = make_xgb_model(n_estimators=FAMILY_SUBSET_XGB_ESTIMATORS)
    xgb_rank_cmp.fit(X_full[tr], y_rank[tr], sample_weight=tr_weights, verbose=False)
    add_row("XGBoost_Rank", np.clip(xgb_rank_cmp.predict(X_full[va]), 0, 1))

    xgb_raw_cmp = make_xgb_model(n_estimators=FAMILY_SUBSET_XGB_ESTIMATORS)
    xgb_raw_cmp.fit(X_full[tr], y_raw[tr], sample_weight=tr_weights, verbose=False)
    raw_pred = xgb_raw_cmp.predict(X_full[va])
    raw_rank_like = percentile_against_reference(raw_pred, xgb_raw_cmp.predict(X_full[tr]))
    add_row("XGBoost_Raw_percentile", np.clip(raw_rank_like, 0, 1))

    kmer_cmp = make_pipeline(
        TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=50000),
        Ridge(alpha=1.0),
    )
    kmer_cmp.fit([kmer_text[i] for i in tr], y_rank[tr])
    add_row("Kmer_Ridge", np.clip(kmer_cmp.predict([kmer_text[i] for i in va]), 0, 1))

    pos_cmp = make_pipeline(StandardScaler(with_mean=False), Ridge(alpha=10.0))
    pos_cmp.fit(X_pos[tr], y_rank[tr])
    add_row("Position_Ridge", np.clip(pos_cmp.predict(X_pos[va]), 0, 1))

    return pd.DataFrame(rows)


def lofo_for_family_subset(mode):
    """LOFO XGBoost-rank comparison inside a subset, useful for family-transfer diagnosis."""
    indices = family_subset_indices(mode)
    fams = train_df["family"].values
    rows = []
    for held in sorted(pd.unique(fams[indices])):
        tr = indices[fams[indices] != held]
        va = indices[fams[indices] == held]
        if len(tr) == 0 or len(va) == 0:
            continue
        weights = subset_family_weights(tr)
        model = make_xgb_model(n_estimators=FAMILY_SUBSET_XGB_ESTIMATORS)
        model.fit(X_full[tr], y_rank[tr], sample_weight=weights, verbose=False)
        pred = np.clip(model.predict(X_full[va]), 0, 1)
        rows.append({
            "mode": mode,
            "heldout_family": held,
            "train_rows": int(len(tr)),
            "val_rows": int(len(va)),
            "xgb_rank_r2": float(r2_score(y_rank[va], pred)),
            "xgb_rank_rmse": float(np.sqrt(mean_squared_error(y_rank[va], pred))),
            "xgb_rank_spearman": safe_spearman(y_rank[va], pred) if "safe_spearman" in globals() else float(pd.Series(y_rank[va]).corr(pd.Series(pred), method="spearman")),
            "xgb_rank_top10_recall": topk_recall(y_rank[va], pred, 0.10) if "topk_recall" in globals() else np.nan,
            "xgb_rank_top20_recall": topk_recall(y_rank[va], pred, 0.20) if "topk_recall" in globals() else np.nan,
        })
    return pd.DataFrame(rows)


if RUN_FAMILY_SUBSET_COMPARISON:
    subset_comparison_df = pd.concat(
        [evaluate_family_subset_mode(mode) for mode in FAMILY_SUBSET_COMPARISON_MODES],
        ignore_index=True,
    )
    subset_comparison_df.to_csv(FAMILY_SUBSET_COMPARISON_OUT, index=False)
    display(subset_comparison_df.sort_values(["model_name", "spearman"], ascending=[True, False]))
    print("family subset model comparison saved:", FAMILY_SUBSET_COMPARISON_OUT)

    subset_lofo_df = pd.concat(
        [lofo_for_family_subset(mode) for mode in FAMILY_SUBSET_COMPARISON_MODES],
        ignore_index=True,
    )
    subset_lofo_df.to_csv(FAMILY_SUBSET_LOFO_OUT, index=False)
    display(subset_lofo_df)
    print("family subset LOFO comparison saved:", FAMILY_SUBSET_LOFO_OUT)
else:
    print("family subset comparison skipped.")


family subset all: rows=141144 families=['amacGFP', 'avGFP', 'cgreGFP', 'ppluGFP']
family subset exclude_pplu: rows=109742 families=['amacGFP', 'avGFP', 'cgreGFP']


/home/liuchang/anaconda3/envs/GFP/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 4.924707752707036e-09.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


family subset exclude_pplu_cgre: rows=85226 families=['amacGFP', 'avGFP']


,mode,model_name,train_rows,val_rows,families,r2,rmse,spearman,top10_recall,top20_recall
2,all,Kmer_Ridge,119964,21180,amacGFP;avGFP;cgreGFP;ppluGFP,0.709766,0.155023,0.877047,0.512748,0.668555
6,exclude_pplu,Kmer_Ridge,93272,16470,amacGFP;avGFP;cgreGFP,0.693259,0.159373,0.867167,0.486339,0.659381
10,exclude_pplu_cgre,Kmer_Ridge,72436,12790,amacGFP;avGFP,0.699050,0.158426,0.864359,0.449570,0.622361
11,exclude_pplu_cgre,Position_Ridge,72436,12790,amacGFP;avGFP,0.602373,0.182103,0.827766,0.401876,0.557467
3,all,Position_Ridge,119964,21180,amacGFP;avGFP;cgreGFP;ppluGFP,0.596376,0.182814,0.813639,0.420208,0.572002
7,exclude_pplu,Position_Ridge,93272,16470,amacGFP;avGFP;cgreGFP,0.579838,0.186525,0.811830,0.394657,0.560716
8,exclude_pplu_cgre,XGBoost_Rank,72436,12790,amacGFP;avGFP,0.512759,0.201582,0.726555,0.339328,0.503518
4,exclude_pplu,XGBoost_Rank,93272,16470,amacGFP;avGFP;cgreGFP,0.492531,0.204990,0.715618,0.349120,0.505464
0,all,XGBoost_Rank,119964,21180,amacGFP;avGFP;cgreGFP;ppluGFP,0.488000,0.205900,0.711604,0.348914,0.499764
9,exclude_pplu_cgre,XGBoost_Raw_percentile,72436,12790,amacGFP;avGFP,0.008379,0.287576,0.492869,0.159500,0.306880


family subset model comparison saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/family_subset_model_comparison.csv
family subset all: rows=141144 families=['amacGFP', 'avGFP', 'cgreGFP', 'ppluGFP']
family subset exclude_pplu: rows=109742 families=['amacGFP', 'avGFP', 'cgreGFP']
family subset exclude_pplu_cgre: rows=85226 families=['amacGFP', 'avGFP']


,mode,heldout_family,train_rows,val_rows,xgb_rank_r2,xgb_rank_rmse,xgb_rank_spearman,xgb_rank_top10_recall,xgb_rank_top20_recall
0,all,amacGFP,107633,33511,0.080349,0.276832,0.301241,0.137530,0.292854
1,all,avGFP,89429,51715,-0.023613,0.292064,0.483951,0.230472,0.386252
2,all,cgreGFP,116628,24516,0.128609,0.265075,0.452086,0.232055,0.372961
3,all,ppluGFP,109742,31402,-0.869877,0.394744,0.313230,0.113021,0.265563
4,exclude_pplu,amacGFP,76231,33511,0.033973,0.283727,0.274328,0.132458,0.287334
5,exclude_pplu,avGFP,58027,51715,-0.233863,0.320659,0.451887,0.224285,0.376003
6,exclude_pplu,cgreGFP,85226,24516,-0.442519,0.341054,0.409368,0.218189,0.351142
7,exclude_pplu_cgre,amacGFP,51715,33511,-0.050929,0.295932,0.214041,0.129177,0.271819
8,exclude_pplu_cgre,avGFP,33511,51715,-0.327620,0.332618,0.408807,0.213844,0.362951


family subset LOFO comparison saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/family_subset_lofo_comparison.csv


In [10]:
# ==========================================
# 7. 直接调用 ThermoMPNN，重新生成 sfGFP 单点突变稳定性结果
# ==========================================

def shell_join(cmd):
    return " ".join(str(x) for x in cmd)

def find_thermompnn_csv(out_dir):
    """在 ThermoMPNN 输出目录中寻找最像单点扫描结果的 CSV。"""
    candidates = sorted(Path(out_dir).rglob("*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        return None
    for p in candidates:
        name = p.name.lower()
        if any(k in name for k in ["ddg", "mutation", "scan", "thermo", "predict"]):
            return p
    return candidates[0]

def normalize_thermompnn_table(path):
    """把不同 ThermoMPNN 输出格式统一成 pos0/wt/mut/ddg 四列。

    注意：ddg 越负通常表示越稳定。如果你安装的 ThermoMPNN 输出符号相反，
    需要在这里把 ddg 乘以 -1。
    """
    tm = pd.read_csv(path)
    lower_map = {c.lower(): c for c in tm.columns}

    rename = {}
    for target, aliases in {
        "pos0": ["pos0", "position", "pos", "residue_index", "residue_idx", "idx", "site"],
        "wt": ["wt", "wildtype", "wild_type", "native", "from"],
        "mut": ["mut", "mutation", "mutant", "aa", "to"],
        "ddg": ["ddg", "ddg_pred", "pred_ddg", "prediction", "score"],
    }.items():
        for a in aliases:
            if a in lower_map:
                rename[lower_map[a]] = target
                break

    tm = tm.rename(columns=rename).copy()

    # 有些输出会用 A123V 这样的 mutation 字符串，而不是拆开的 wt/position/mut。
    if not {"pos0", "wt", "mut"}.issubset(tm.columns):
        mut_col = None
        for c in tm.columns:
            if c.lower() in {"mutation", "mutations", "variant", "substitution"}:
                mut_col = c
                break
        if mut_col is not None:
            parsed = tm[mut_col].astype(str).str.extract(r"([A-Z])(\d+)([A-Z])")
            if "wt" not in tm.columns:
                tm["wt"] = parsed[0]
            if "pos0" not in tm.columns:
                tm["pos0"] = pd.to_numeric(parsed[1], errors="coerce") - 1
            if "mut" not in tm.columns:
                tm["mut"] = parsed[2]

    missing = {"pos0", "wt", "mut", "ddg"} - set(tm.columns)
    if missing:
        raise ValueError(f"ThermoMPNN 输出 {path} 缺少必要列 {missing}，实际列为 {list(tm.columns)}")

    tm = tm[["pos0", "wt", "mut", "ddg"]].copy()
    tm["pos0"] = pd.to_numeric(tm["pos0"], errors="coerce").astype("Int64")
    tm["wt"] = tm["wt"].astype(str).str.upper().str[0]
    tm["mut"] = tm["mut"].astype(str).str.upper().str[0]
    tm["ddg"] = pd.to_numeric(tm["ddg"], errors="coerce")
    tm = tm.dropna(subset=["pos0", "ddg"])
    tm["pos0"] = tm["pos0"].astype(int)
    tm = tm[tm["mut"].isin(STANDARD_AA) & tm["wt"].isin(STANDARD_AA)]
    return tm

def build_thermompnn_commands():
    """构造一组可能的 ThermoMPNN 命令。

    ThermoMPNN 不同版本入口脚本命名不完全一致，所以这里先支持自定义模板，
    再按常见脚本名自动尝试。若全部失败，notebook 会打印可用脚本，方便下一步改模板。
    """
    THERMOMPNN_OUT_DIR.mkdir(parents=True, exist_ok=True)
    if THERMOMPNN_COMMAND_TEMPLATE:
        return [THERMOMPNN_COMMAND_TEMPLATE.format(
            pdb=THERMOMPNN_STRUCTURE,
            out_dir=THERMOMPNN_OUT_DIR,
            out_csv=THERMOMPNN_RAW_OUT,
            code_dir=THERMOMPNN_CODE_PATH,
        )]

    scripts = []
    for pattern in ["*infer*.py", "*predict*.py", "*scan*.py", "*ddg*.py", "run*.py", "main*.py"]:
        scripts.extend(THERMOMPNN_CODE_PATH.rglob(pattern))
    scripts = sorted(set(scripts), key=lambda p: (0 if "thermo" in p.name.lower() else 1, len(str(p))))

    commands = []
    for script in scripts[:20]:
        commands.extend([
            f"python {script} --pdb {THERMOMPNN_STRUCTURE} --out_dir {THERMOMPNN_OUT_DIR}",
            f"python {script} --pdb_path {THERMOMPNN_STRUCTURE} --out_dir {THERMOMPNN_OUT_DIR}",
            f"python {script} --structure {THERMOMPNN_STRUCTURE} --out_dir {THERMOMPNN_OUT_DIR}",
            f"python {script} --input {THERMOMPNN_STRUCTURE} --output {THERMOMPNN_RAW_OUT}",
        ])
    return commands

def run_thermompnn_single_scan():
    """调用 ThermoMPNN 生成 sfGFP 单点突变 ddG 表。"""
    if THERMOMPNN_RAW_OUT.exists() and not FORCE_RERUN_THERMOMPNN:
        print("使用本次输出目录中已有 ThermoMPNN 结果:", THERMOMPNN_RAW_OUT)
        return normalize_thermompnn_table(THERMOMPNN_RAW_OUT)

    if not THERMOMPNN_CODE_PATH.exists():
        raise FileNotFoundError(f"未找到 ThermoMPNN 代码目录: {THERMOMPNN_CODE_PATH}")
    if not THERMOMPNN_STRUCTURE.exists():
        raise FileNotFoundError(f"未找到 ThermoMPNN 输入结构: {THERMOMPNN_STRUCTURE}")

    commands = build_thermompnn_commands()
    failures = []
    for cmd in commands:
        print("尝试 ThermoMPNN 命令:", cmd)
        result = subprocess.run(
            cmd,
            shell=True,
            cwd=str(THERMOMPNN_CODE_PATH),
            text=True,
            capture_output=True,
        )
        found = find_thermompnn_csv(THERMOMPNN_OUT_DIR)
        if result.returncode == 0 and found is not None:
            tm = normalize_thermompnn_table(found)
            tm.to_csv(THERMOMPNN_RAW_OUT, index=False)
            print("ThermoMPNN 结果已标准化保存:", THERMOMPNN_RAW_OUT)
            return tm
        failures.append((cmd, result.returncode, result.stderr[-1000:]))

    py_files = sorted(str(p.relative_to(THERMOMPNN_CODE_PATH)) for p in THERMOMPNN_CODE_PATH.rglob("*.py"))[:80]
    print("自动尝试失败。ThermoMPNN-main 中可见 Python 脚本如下:")
    for p in py_files:
        print("  ", p)
    print("最近几条失败信息:")
    for cmd, code, err in failures[-5:]:
        print("\\nCMD:", cmd)
        print("returncode:", code)
        print(err)
    raise RuntimeError("未能自动调用 ThermoMPNN。请把正确命令填到 THERMOMPNN_COMMAND_TEMPLATE。")

thermo_df = run_thermompnn_single_scan()
thermo_df["mutation_label"] = thermo_df["wt"] + (thermo_df["pos0"] + 1).astype(str) + thermo_df["mut"]
thermo_stabilizing_df = thermo_df[
    (thermo_df["ddg"] < -0.5)
    & (~thermo_df["pos0"].isin(PROTECTED_POSITIONS_0IDX))
    & (thermo_df["wt"] != thermo_df["mut"])
].sort_values("ddg").copy()
thermo_stabilizing_df.to_csv(THERMO_STABILIZING_OUT, index=False)
thermo_ddg_map = {(int(r.pos0), str(r.mut)): float(r.ddg) for r in thermo_df.itertuples()}
print("ThermoMPNN 单点突变数:", len(thermo_df))
print("ThermoMPNN 稳定化突变数:", len(thermo_stabilizing_df))
display(thermo_stabilizing_df.head(20))


尝试 ThermoMPNN 命令: conda run -n deepstab_env python /data/liuchang/ThermoMPNN-main/analysis/custom_inference.py --pdb /home/liuchang/PythonProject2/fold_sfgfp_model_0.pdb --out_dir /home/liuchang/PythonProject2/server_full_recompute_outputs/thermompnn_sfgfp_strict_aligned
ThermoMPNN 结果已标准化保存: /home/liuchang/PythonProject2/server_full_recompute_outputs/thermompnn_sfgfp_strict_aligned/thermompnn_single_mutation_scan.csv
ThermoMPNN 单点突变数: 4760
ThermoMPNN 稳定化突变数: 39


,pos0,wt,mut,ddg,mutation_label
3530,176,Q,M,-1.073905,Q177M
3317,165,K,V,-1.020803,K166V
3379,168,H,Y,-0.940831,H169Y
3684,184,N,F,-0.893226,N185F
4087,204,S,I,-0.878991,S205I
3307,165,K,I,-0.869730,K166I
4097,204,S,V,-0.847533,S205V
2959,147,H,Y,-0.845503,H148Y
3699,184,N,Y,-0.825864,N185Y
887,44,K,I,-0.783431,K45I


In [11]:
# ==========================================
# 8. pseudo-sfGFP pocket 单点扫描 + pocket/scaffold 约束 GA
# ==========================================

rng = np.random.default_rng(GA_RANDOM_SEED)

def in_sfgfp_pocket(mut):
    _wt, pos0, _aa = normalize_mut(mut)
    return (pos0 in SF_GFP_POCKET_POSITIONS_0IDX) and (pos0 not in PROTECTED_POSITIONS_0IDX)

def normalize_mut(m):
    """确保突变始终是 (wt:str, pos0:int, mut:str)，避免 numpy 类型污染。"""
    return (str(m[0]), int(m[1]), str(m[2]))

def normalize_combo(combo):
    """规范化一个突变组合，并按位置排序。"""
    out = []
    for m in combo:
        try:
            out.append(normalize_mut(m))
        except Exception:
            continue
    return tuple(sorted(out, key=lambda x: x[1]))

def mutation_label(muts):
    muts = normalize_combo(muts)
    return ";".join(f"{wt}{pos0 + 1}{mut}" for wt, pos0, mut in muts)

def split_candidate_mutations(label):
    return split_mutation_string(str(label).replace(";", ":"))

def apply_candidate_mutations(base_seq, muts):
    muts = normalize_combo(muts)
    seq = list(base_seq)
    used = set()

    for wt, pos0, mut in muts:
        if pos0 in used:
            return None
        if pos0 < 0 or pos0 >= len(seq):
            return None
        if seq[pos0] != wt:
            return None
        if mut not in STANDARD_AA or mut == wt:
            return None

        seq[pos0] = mut
        used.add(pos0)

    return "".join(seq)

def combo_counts(muts):
    muts = normalize_combo(muts)
    pocket_n = sum(in_sfgfp_pocket(m) for m in muts)
    scaffold_n = len(muts) - pocket_n
    return len(muts), pocket_n, scaffold_n

def valid_mutation_combo(muts, allow_single_pocket_scan=False):
    muts = normalize_combo(muts)

    if not muts:
        return False
    if len({int(m[1]) for m in muts}) != len(muts):
        return False
    if any(pos0 in PROTECTED_POSITIONS_0IDX for _wt, pos0, _mut in muts):
        return False

    total_n, pocket_n, scaffold_n = combo_counts(muts)

    # pseudo-sfGFP 单点扫描需要允许 1 个 pocket 突变单独存在。
    if allow_single_pocket_scan:
        return total_n == 1 and pocket_n == 1

    if total_n < MIN_TOTAL_MUTATIONS_PER_CANDIDATE:
        return False
    if total_n > MAX_TOTAL_MUTATIONS_PER_CANDIDATE:
        return False
    if pocket_n < MIN_POCKET_MUTATIONS_PER_CANDIDATE:
        return False
    if pocket_n > MAX_POCKET_MUTATIONS_PER_CANDIDATE:
        return False
    if scaffold_n < MIN_SCAFFOLD_MUTATIONS_PER_CANDIDATE:
        return False
    if scaffold_n > MAX_SCAFFOLD_MUTATIONS_PER_CANDIDATE:
        return False

    return True

def thermo_ddg_sum_for_muts(muts):
    muts = normalize_combo(muts)
    vals = [
        thermo_ddg_map[(pos0, mut)]
        for _wt, pos0, mut in muts
        if "thermo_ddg_map" in globals() and (pos0, mut) in thermo_ddg_map
    ]
    return float(np.sum(vals)) if vals else np.nan

def make_candidate_df(mut_combos, source, allow_single_pocket_scan=False):
    rows = []
    seen = set()

    for muts in mut_combos:
        muts = normalize_combo(muts)

        if not valid_mutation_combo(muts, allow_single_pocket_scan=allow_single_pocket_scan):
            continue

        seq = apply_candidate_mutations(SF_GFP, muts)
        if not seq or seq in seen:
            continue

        seen.add(seq)
        total_n, pocket_n, scaffold_n = combo_counts(muts)

        rows.append({
            "Sequence": seq,
            "Mutations": mutation_label(muts),
            "Mutation_Count": total_n,
            "Candidate_Source": source,
            "Pocket_Mutation_Count": pocket_n,
            "Scaffold_Mutation_Count": scaffold_n,
            "Pre_Thermo_ddG_sum": thermo_ddg_sum_for_muts(muts),
        })

    return pd.DataFrame(rows)

def score_candidate_df(df, tag):
    df = df.copy().reset_index(drop=True)

    if df.empty:
        raise RuntimeError(f"{tag} candidate dataframe is empty.")

    df["saprot_input"] = df["Sequence"].map(
        lambda s: stitch_saprot(s, WT_3DI["sfGFP"], strict=False)
    )

    mut_lists = [
        split_candidate_mutations(muts)
        for muts in df["Mutations"].fillna("").astype(str)
    ]

    pocket_features = combined_pocket_feature_matrix(
        df["Sequence"],
        ["sfGFP"] * len(df),
        mut_lists,
    )

    esm_path = RUN_OUTPUT_DIR / f"{tag}_esm2_embeddings.npy"
    sap_path = RUN_OUTPUT_DIR / f"{tag}_saprot_embeddings.npy"

    esm = compute_esm_embeddings(
        df["Sequence"].tolist(),
        esm_path,
        ESM_BATCH_SIZE,
        force=RECOMPUTE_CANDIDATE_EMBEDDINGS,
    )

    sap = compute_saprot_embeddings(
        df["saprot_input"].tolist(),
        sap_path,
        SAPROT_BATCH_SIZE,
        force=RECOMPUTE_CANDIDATE_EMBEDDINGS,
    )

    X = build_model_feature_matrix(esm, sap, pocket_features, ["sfGFP"] * len(df))
    df = add_brightness_predictions(df, X, esm, sap, pocket_features, families=["sfGFP"] * len(df))

    return df, esm, sap, pocket_features

# ==========================================
# 1. 生成所有 sfGFP pocket 单点突变，并作为 pseudo-sfGFP 局部扫描
# ==========================================

single_mutations = []

for pos0 in sorted(SF_GFP_POCKET_POSITIONS_0IDX):
    if pos0 in PROTECTED_POSITIONS_0IDX or pos0 >= len(SF_GFP):
        continue

    wt = SF_GFP[pos0]
    if wt not in STANDARD_AA:
        continue

    for mut in sorted(STANDARD_AA):
        if mut == wt:
            continue
        if GA_AVOID_CYS and mut == "C":
            continue
        single_mutations.append((wt, pos0, mut))

single_df = make_candidate_df(
    [(m,) for m in single_mutations],
    "pseudo_sfgfp_pocket_single_scan",
    allow_single_pocket_scan=True,
)

single_df, single_esm, single_sap, single_pocket = score_candidate_df(
    single_df,
    "pseudo_sfgfp_pocket_single_scan",
)

single_df = single_df.sort_values(
    ["Brightness_Rescore", "XGBoost_Brightness"],
    ascending=False,
).reset_index(drop=True)

single_df.to_csv(
    RUN_OUTPUT_DIR / "pseudo_sfgfp_pocket_single_mutant_scan_pre_wt_ratio.csv",
    index=False,
)

print("pseudo-sfGFP pocket single mutants:", single_df.shape)
display(single_df[[
    "Mutations",
    "XGBoost_Brightness",
    "XGBoost_RankScore",
    "GatingNN_RankScore",
    "Brightness_Rescore",
    "Brightness_Uncertainty",
    "Pre_Thermo_ddG_sum",
]].head(30))

# ==========================================
# 2. 构建两个突变库：pocket 亮度库 + scaffold 稳定化库
# ==========================================

pocket_single_for_pool = single_df.copy()

# pocket 单点库：优先取 rank 高、模型分歧不太大的单点。
pocket_single_for_pool = pocket_single_for_pool[
    pocket_single_for_pool["Relative_Uncertainty"].fillna(0.0) <= MAX_RELATIVE_UNCERTAINTY_FOR_TOP
].copy()
if pocket_single_for_pool.empty:
    pocket_single_for_pool = single_df.copy()

pocket_labels = pocket_single_for_pool.head(GA_TOP_SINGLE_MUTATIONS)["Mutations"].tolist()
pocket_brightness_pool = []
for label in pocket_labels:
    muts = split_candidate_mutations(label)
    if muts:
        pocket_brightness_pool.append(normalize_mut(muts[0]))

# scaffold 稳定化库：来自 ThermoMPNN，排除 pocket 和保护位点。
scaffold_thermo_pool = []
if "thermo_stabilizing_df" in globals() and not thermo_stabilizing_df.empty:
    thermo_pool_df = thermo_stabilizing_df.copy()
else:
    thermo_pool_df = thermo_df[thermo_df["ddg"] < STABILITY_SCAFFOLD_DDG_MAX].copy()

thermo_pool_df = thermo_pool_df[
    (~thermo_pool_df["pos0"].isin(PROTECTED_POSITIONS_0IDX))
].copy()

for r in thermo_pool_df.sort_values("ddg").itertuples():
    m = normalize_mut((r.wt, int(r.pos0), r.mut))
    if in_sfgfp_pocket(m):
        continue
    if GA_AVOID_CYS and m[2] == "C":
        continue
    scaffold_thermo_pool.append(m)
    if len(scaffold_thermo_pool) >= GA_TOP_SCAFFOLD_THERMO_MUTATIONS:
        break

print("pocket brightness pool:", len(pocket_brightness_pool))
print("scaffold thermo pool:", len(scaffold_thermo_pool))
print("pocket examples:", [mutation_label([m]) for m in pocket_brightness_pool[:15]])
print("scaffold examples:", [mutation_label([m]) for m in scaffold_thermo_pool[:15]])

if not pocket_brightness_pool:
    raise RuntimeError("pocket_brightness_pool 为空，无法生成候选。")
if not scaffold_thermo_pool:
    raise RuntimeError("scaffold_thermo_pool 为空，请检查 ThermoMPNN 结果。")

def random_combo_from_two_pools():
    pocket_n = int(rng.integers(MIN_POCKET_MUTATIONS_PER_CANDIDATE, MAX_POCKET_MUTATIONS_PER_CANDIDATE + 1))
    scaffold_n = int(rng.integers(MIN_SCAFFOLD_MUTATIONS_PER_CANDIDATE, MAX_SCAFFOLD_MUTATIONS_PER_CANDIDATE + 1))
    total_n = pocket_n + scaffold_n

    if total_n > MAX_TOTAL_MUTATIONS_PER_CANDIDATE:
        scaffold_n = MAX_TOTAL_MUTATIONS_PER_CANDIDATE - pocket_n
    if total_n < MIN_TOTAL_MUTATIONS_PER_CANDIDATE:
        scaffold_n = max(scaffold_n, MIN_TOTAL_MUTATIONS_PER_CANDIDATE - pocket_n)

    if len(pocket_brightness_pool) < pocket_n or len(scaffold_thermo_pool) < scaffold_n:
        return None

    p_idx = rng.choice(len(pocket_brightness_pool), size=pocket_n, replace=False)
    s_idx = rng.choice(len(scaffold_thermo_pool), size=scaffold_n, replace=False)
    combo = [pocket_brightness_pool[int(i)] for i in p_idx] + [scaffold_thermo_pool[int(i)] for i in s_idx]
    combo = normalize_combo(combo)
    return combo if valid_mutation_combo(combo) else None


def conservative_random_mutation():
    """Small exploration mutation outside the closed pools.

    限制在非 pocket、非保护位点，且优先保守替换，避免完全随机破坏结构。
    """
    mutable_positions = [
        i for i, aa in enumerate(SF_GFP)
        if aa in STANDARD_AA
        and i not in PROTECTED_POSITIONS_0IDX
        and i not in SF_GFP_POCKET_POSITIONS_0IDX
    ]
    for _ in range(50):
        pos0 = int(rng.choice(mutable_positions))
        wt = SF_GFP[pos0]
        groups = [g for g in CONSERVATIVE_AA_GROUPS if wt in g]
        if groups:
            choices = [aa for aa in groups[0] if aa != wt and aa in STANDARD_AA]
        else:
            choices = [aa for aa in STANDARD_AA if aa != wt]
        if GA_AVOID_CYS:
            choices = [aa for aa in choices if aa != "C"]
        if choices:
            return normalize_mut((wt, pos0, str(rng.choice(choices))))
    return None

# ==========================================
# 3. 初始化 GA 种群：少量 pocket 调亮 + scaffold 稳定化
# ==========================================

population = set()

# 系统组合：top pocket 单点分别搭配 top scaffold 稳定化突变。
for p in pocket_brightness_pool[:min(30, len(pocket_brightness_pool))]:
    for s in scaffold_thermo_pool[:min(40, len(scaffold_thermo_pool))]:
        combo = normalize_combo([p, s])
        # 如果只有 2 个突变，补一个 scaffold，使其满足 3-6 个总突变约束。
        if len(combo) < MIN_TOTAL_MUTATIONS_PER_CANDIDATE:
            for extra in scaffold_thermo_pool:
                if extra[1] not in {m[1] for m in combo}:
                    combo = normalize_combo(list(combo) + [extra])
                    break
        if valid_mutation_combo(combo):
            population.add(combo)
        if len(population) >= GA_POP_SIZE:
            break
    if len(population) >= GA_POP_SIZE:
        break

# 加入随机组合，增加多样性。
attempts = 0
while len(population) < GA_POP_SIZE and attempts < GA_POP_SIZE * 80:
    attempts += 1
    combo = random_combo_from_two_pools()
    if combo is not None:
        population.add(combo)

print("initial GA population:", len(population))

# ==========================================
# 4. GA 交叉与突变函数
# ==========================================

def repair_combo(combo):
    combo = list(normalize_combo(combo))
    by_pos = {}
    for m in combo:
        by_pos.setdefault(int(m[1]), m)
    combo = list(by_pos.values())

    # 控制总突变数。
    while len(combo) > MAX_TOTAL_MUTATIONS_PER_CANDIDATE:
        idx = int(rng.integers(0, len(combo)))
        combo.pop(idx)

    # 控制 pocket 上限。
    while sum(in_sfgfp_pocket(m) for m in combo) > MAX_POCKET_MUTATIONS_PER_CANDIDATE:
        pocket_indices = [i for i, m in enumerate(combo) if in_sfgfp_pocket(m)]
        combo.pop(int(rng.choice(pocket_indices)))

    # 至少一个 pocket。
    while sum(in_sfgfp_pocket(m) for m in combo) < MIN_POCKET_MUTATIONS_PER_CANDIDATE:
        m = pocket_brightness_pool[int(rng.integers(0, len(pocket_brightness_pool)))]
        if m[1] not in {x[1] for x in combo}:
            combo.append(m)

    # 至少一个 scaffold，且总突变数达到下限。
    while (
        len(combo) < MIN_TOTAL_MUTATIONS_PER_CANDIDATE
        or (len(combo) - sum(in_sfgfp_pocket(m) for m in combo)) < MIN_SCAFFOLD_MUTATIONS_PER_CANDIDATE
    ):
        m = scaffold_thermo_pool[int(rng.integers(0, len(scaffold_thermo_pool)))]
        if m[1] not in {x[1] for x in combo}:
            combo.append(m)

    combo = normalize_combo(combo)
    return combo if valid_mutation_combo(combo) else None

def crossover(parent_a, parent_b):
    merged = [normalize_mut(m) for m in list(parent_a) + list(parent_b)]
    rng.shuffle(merged)
    by_pos = {}
    for m in merged:
        by_pos.setdefault(int(m[1]), m)
    return repair_combo(by_pos.values())

def mutate_combo(combo):
    combo = list(normalize_combo(combo))

    if rng.random() < GA_MUTATION_RATE:
        if rng.random() < GA_EXPLORATION_RATE:
            new_mut = conservative_random_mutation()
            if new_mut is None:
                pool = scaffold_thermo_pool
                new_mut = normalize_mut(pool[int(rng.integers(0, len(pool)))])
        else:
            pool = pocket_brightness_pool if rng.random() < 0.35 else scaffold_thermo_pool
            new_mut = normalize_mut(pool[int(rng.integers(0, len(pool)))])

        if combo and rng.random() < 0.55:
            # 优先替换同类型突变，保持 pocket/scaffold 比例。
            same_type_idx = [
                i for i, old in enumerate(combo)
                if in_sfgfp_pocket(old) == in_sfgfp_pocket(new_mut)
            ]
            if same_type_idx:
                combo[int(rng.choice(same_type_idx))] = new_mut
            else:
                combo[int(rng.integers(0, len(combo)))] = new_mut
        else:
            combo.append(new_mut)

    return repair_combo(combo)

# ==========================================
# 5. GA 主循环：rank + raw brightness + ThermoMPNN + 突变结构约束
# ==========================================

all_scored = []

for gen in range(1, GA_GENERATIONS + 1):
    pop_df = make_candidate_df(population, f"ga_gen_{gen}_pocket_scaffold")
    pop_df, _, _, _ = score_candidate_df(pop_df, f"ga_gen_{gen}_pocket_scaffold")

    # GA 预筛也使用验证集学习得到的亮度分数；ThermoMPNN 在这里作为 tie-break，最终仍会 hard filter。
    if "Data_Driven_Brightness_Score" in pop_df.columns:
        pop_df["GA_Fitness"] = pop_df["Data_Driven_Brightness_Score"].fillna(pop_df["Brightness_Rescore"])
    else:
        pop_df["GA_Fitness"] = pop_df["Brightness_Rescore"].fillna(0.0)

    # 下面两项只是明确的排序 tie-break，不再是主要手工加权模型。
    pop_df["GA_Fitness"] = pop_df["GA_Fitness"] + 0.02 * pop_df["XGBoost_Brightness"].rank(pct=True)
    pop_df["GA_Fitness"] = pop_df["GA_Fitness"] + 0.01 * (1.0 + np.tanh((-pop_df["Pre_Thermo_ddG_sum"].fillna(0.0)) / 4.0))
    pop_df["GA_Fitness"] = pop_df["GA_Fitness"] - 0.01 * pop_df["Brightness_Uncertainty"].fillna(0.0)

    pop_df = pop_df.sort_values("GA_Fitness", ascending=False).reset_index(drop=True)
    all_scored.append(pop_df)

    print(
        f"GA gen {gen}:",
        "best rank", float(pop_df["Brightness_Rescore"].iloc[0]),
        "best raw", float(pop_df["XGBoost_Brightness"].iloc[0]),
        "best ddG", float(pop_df["Pre_Thermo_ddG_sum"].iloc[0]),
        "best fitness", float(pop_df["GA_Fitness"].iloc[0]),
        "population", len(pop_df),
    )

    elites = []
    for label in pop_df.head(GA_ELITE_SIZE)["Mutations"]:
        muts = normalize_combo(split_candidate_mutations(label.replace(";", ":")))
        if valid_mutation_combo(muts):
            elites.append(muts)

    new_population = set(elites)
    attempts = 0

    while len(new_population) < GA_POP_SIZE and elites and attempts < GA_POP_SIZE * 100:
        attempts += 1
        p1 = elites[int(rng.integers(0, len(elites)))]
        p2 = elites[int(rng.integers(0, len(elites)))]
        child = crossover(p1, p2)
        if child is not None:
            child = mutate_combo(child)
        if child is not None:
            new_population.add(child)

    # 补足随机组合，避免种群塌缩。
    while len(new_population) < GA_POP_SIZE and attempts < GA_POP_SIZE * 160:
        attempts += 1
        combo = random_combo_from_two_pools()
        if combo is not None:
            new_population.add(combo)

    population = new_population

# ==========================================
# 6. 汇总 GA 候选
# ==========================================

ga_all = pd.concat(all_scored, ignore_index=True)
ga_all = ga_all.sort_values("GA_Fitness", ascending=False)
ga_all = ga_all.drop_duplicates("Sequence").reset_index(drop=True)

if TOP_N_REEMBED is not None:
    cand = ga_all.head(TOP_N_REEMBED).copy()
else:
    cand = ga_all.copy()

cand["Seq_ID"] = [f"ga_sfgfp_{i + 1:05d}" for i in range(len(cand))]
cand["Candidate_Source"] = "ga_pocket_scaffold_wt_anchor"
cand.to_csv(CANDIDATE_POOL_OUT, index=False)

print("GA final candidates:", cand.shape)
print("候选池已保存:", CANDIDATE_POOL_OUT)

display(cand[[
    "Seq_ID",
    "Mutations",
    "Mutation_Count",
    "Pocket_Mutation_Count",
    "Scaffold_Mutation_Count",
    "XGBoost_Brightness",
    "Brightness_Rescore",
    "Brightness_Uncertainty",
    "GA_Fitness",
    "Pre_Thermo_ddG_sum",
]].head(30))

# ==========================================
# 7. 为第 10 单元准备 cand_esm / cand_saprot / cand_pocket_features
# ==========================================

cand["saprot_input"] = cand["Sequence"].map(
    lambda s: stitch_saprot(s, WT_3DI["sfGFP"], strict=False)
)

cand_mut_lists = [
    split_candidate_mutations(muts.replace(";", ":"))
    for muts in cand["Mutations"].fillna("").astype(str)
]

cand_pocket_features = combined_pocket_feature_matrix(
    cand["Sequence"],
    ["sfGFP"] * len(cand),
    cand_mut_lists,
)

np.save(
    RUN_OUTPUT_DIR / "recomputed_candidate_pymol_pocket_features.npy",
    cand_pocket_features,
)

cand_esm = compute_esm_embeddings(
    cand["Sequence"].tolist(),
    CANDIDATE_ESM_OUT,
    ESM_BATCH_SIZE,
    force=RECOMPUTE_CANDIDATE_EMBEDDINGS,
)

cand_saprot = compute_saprot_embeddings(
    cand["saprot_input"].tolist(),
    CANDIDATE_SAPROT_OUT,
    SAPROT_BATCH_SIZE,
    force=RECOMPUTE_CANDIDATE_EMBEDDINGS,
)

print("cand_esm:", cand_esm.shape)
print("cand_saprot:", cand_saprot.shape)
print("cand_pocket_features:", cand_pocket_features.shape)


使用 embedding 缓存: /home/liuchang/PythonProject2/server_full_recompute_outputs/pseudo_sfgfp_pocket_single_scan_esm2_embeddings.npy shape=(324, 1280)
使用 embedding 缓存: /home/liuchang/PythonProject2/server_full_recompute_outputs/pseudo_sfgfp_pocket_single_scan_saprot_embeddings.npy shape=(324, 1280)
pseudo-sfGFP pocket single mutants: (324, 23)


,Mutations,XGBoost_Brightness,XGBoost_RankScore,GatingNN_RankScore,Brightness_Rescore,Brightness_Uncertainty,Pre_Thermo_ddG_sum
0,F223Y,3.428563,0.567963,1.000000,0.978755,0.156756,0.025208
1,F223V,3.566262,0.620566,1.000000,0.941452,0.134350,0.633911
2,F223T,3.309962,0.508905,1.000000,0.941410,0.174531,0.675733
3,F223W,3.454946,0.522747,1.000000,0.940901,0.170776,-0.023940
4,V68M,3.210064,0.367968,0.850276,0.903715,0.217172,0.421489
5,T63S,3.758860,0.711420,0.864189,0.899386,0.066940,0.546280
6,Q204L,3.440953,0.683290,0.929435,0.891961,0.089116,0.266000
7,T62S,3.294776,0.595358,0.848065,0.886576,0.106165,0.502482
8,V206A,3.054402,0.425471,0.830922,0.878107,0.171205,0.740880
9,L44M,3.296945,0.460385,0.822797,0.877240,0.168071,0.571591


pocket brightness pool: 60
scaffold thermo pool: 27
pocket examples: ['F223Y', 'F223V', 'F223T', 'F223W', 'V68M', 'T63S', 'Q204L', 'T62S', 'V206A', 'L44M', 'S205Q', 'L221E', 'L221V', 'S147E', 'L42W']
scaffold examples: ['Q177M', 'K166V', 'H169Y', 'N185F', 'K166I', 'N185Y', 'K45I', 'K166T', 'T203I', 'E172I', 'E172L', 'Q183M', 'Q177L', 'H231P', 'T203V']
initial GA population: 220
缓存 manifest 不匹配，将重算: /home/liuchang/PythonProject2/server_full_recompute_outputs/ga_gen_1_pocket_scaffold_esm2_embeddings.npy


Some weights of EsmModel were not initialized from the model checkpoint at /home/liuchang/esm2_t33_650M and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
ESM -> ga_gen_1_pocket_scaffold_esm2_embeddings.npy: 100%|██████████| 14/14 [00:02<00:00,  6.94it/s]


缓存 manifest 不匹配，将重算: /home/liuchang/PythonProject2/server_full_recompute_outputs/ga_gen_1_pocket_scaffold_saprot_embeddings.npy


...

缓存 manifest 不匹配，将重算: /home/liuchang/PythonProject2/server_full_recompute_outputs/recomputed_candidate_esm2_embeddings.npy


Some weights of EsmModel were not initialized from the model checkpoint at /home/liuchang/esm2_t33_650M and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
ESM -> recomputed_candidate_esm2_embeddings.npy: 100%|██████████| 63/63 [00:09<00:00,  6.91it/s]


缓存 manifest 不匹配，将重算: /home/liuchang/PythonProject2/server_full_recompute_outputs/recomputed_candidate_saprot_embeddings.npy


Some weights of EsmModel were not initialized from the model checkpoint at /home/liuchang/saprot_650M and are newly initialized: ['esm.contact_head.regression.bias', 'esm.contact_head.regression.weight', 'esm.embeddings.position_embeddings.weight', 'esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
SaProt -> recomputed_candidate_saprot_embeddings.npy: 100%|██████████| 63/63 [00:02<00:00, 28.93it/s]

cand_esm: (1000, 1280)
cand_saprot: (1000, 1280)
cand_pocket_features: (1000, 1432)


In [12]:
# ==========================================
# 9. 候选重评分、ThermoMPNN hard filter、分策略 top 6 输出
# ==========================================

def parse_candidate_mutations(label):
    """解析 A12V;B13C 这类突变标签。无法解析的 token 会自动跳过。"""
    if pd.isna(label):
        return []
    muts = []
    for tok in str(label).replace(":", ";").replace("-", ";").split(";"):
        tok = tok.strip()
        m = re.fullmatch(r"([A-Z])(\d+)([A-Z])", tok)
        if m:
            muts.append((m.group(1), int(m.group(2)) - 1, m.group(3)))
    return muts

def thermompnn_candidate_score(mutations, ddg_map):
    """把候选中的单点 ddG 聚合成稳定性分数。"""
    if not ddg_map or not mutations:
        return np.nan, 1.0
    vals = []
    for _wt, pos0, mut in mutations:
        if (pos0, mut) in ddg_map:
            vals.append(ddg_map[(pos0, mut)])
    if not vals:
        return np.nan, 1.0
    ddg_sum = float(np.sum(vals))
    score = float(1.0 + 0.5 * np.tanh((-ddg_sum) / 4.0))
    return ddg_sum, score

def load_exclusion_set(path):
    if not Path(path).exists():
        return set()
    seqs = set()
    for chunk in pd.read_csv(path, usecols=["Sequence"], chunksize=100000):
        seqs.update(chunk["Sequence"].dropna().astype(str).str.strip().str.upper())
    return seqs

def check_sequence(seq):
    errors = []
    if not (220 <= len(seq) <= 250):
        errors.append("length_not_220_250")
    if not seq.startswith("M"):
        errors.append("not_start_with_M")
    if not set(seq).issubset(STANDARD_AA):
        errors.append("nonstandard_aa")
    for pos0 in PROTECTED_POSITIONS_0IDX:
        if pos0 < len(seq) and seq[pos0] != SF_GFP[pos0]:
            errors.append(f"protected_pos_{pos0 + 1}")
    return errors

def hamming(a, b):
    return sum(x != y for x, y in zip(a, b)) + abs(len(a) - len(b))

def has_mutation(label, mutation_prefix):
    return any(tok.startswith(mutation_prefix) for tok in str(label).replace(":", ";").split(";"))

def pocket_pattern(label):
    muts = parse_candidate_mutations(label)
    pocket_labels = [
        f"{wt}{pos0 + 1}{mut}"
        for wt, pos0, mut in muts
        if pos0 in SF_GFP_POCKET_POSITIONS_0IDX
    ]
    return ";".join(sorted(pocket_labels)) if pocket_labels else "no_pocket"

def diversity_select(df, n=6, min_hamming=2, already=None, already_rows=None):
    selected_rows = []
    selected_seqs = list(already or [])
    selected_meta = list(already_rows or [])

    for _, row in df.iterrows():
        seq = row["Sequence"]
        mut_label = row.get("Mutations", "")
        next_meta = selected_meta + [row]

        h148y_count = sum(has_mutation(r.get("Mutations", ""), "H148Y") for r in next_meta)
        if h148y_count > MAX_H148Y_IN_TOP6:
            continue

        patterns = [pocket_pattern(r.get("Mutations", "")) for r in next_meta]
        if patterns.count(pocket_pattern(mut_label)) > MAX_SAME_POCKET_PATTERN_IN_TOP6:
            continue

        if all(hamming(seq, s) >= min_hamming for s in selected_seqs):
            selected_rows.append(row)
            selected_seqs.append(seq)
            selected_meta.append(row)
            if len(selected_rows) >= n:
                break

    return pd.DataFrame(selected_rows)



def select_score_col(df):
    """选择最终排序使用的分数列，避免某些诊断列未生成时 NameError。"""
    priority = [
        "CrossFamily_EnsembleScore",
        "Final_Learned_Score",
        "Data_Driven_Brightness_Score",
        "Brightness_Rescore",
        "Heavy_Rank_Ensemble",
        "XGBoost_Raw_Relative_to_WT",
        "XGBoost_Brightness",
    ]
    for col in priority:
        if col in df.columns:
            return col
    raise ValueError(f"候选表里找不到可用于排序的分数列，当前列名: {list(df.columns)}")

def pick_strategy_top6(valid):
    """最终 top6 不再强塞 stability_first；优先选择 WT-anchored 综合分高且多样的候选。"""
    chosen = []
    chosen_seqs = []
    chosen_rows_meta = []

    score_col = select_score_col(valid)
    non_h148y = valid[~valid["Mutations"].astype(str).str.contains("H148Y", regex=False)].copy()
    strategies = [
        ("learned_score_best", valid.sort_values([score_col, "XGBoost_Raw_Relative_to_WT"], ascending=[False, False]), 2),
        ("raw_brightness_first", valid.sort_values(["XGBoost_Raw_Relative_to_WT", score_col], ascending=[False, False]), 2),
        ("non_H148Y_diversity", non_h148y.sort_values(score_col, ascending=False), 1),
        ("low_uncertainty_balance", valid.sort_values(["Relative_Uncertainty", score_col], ascending=[True, False]), 1),
    ]

    for label, frame, quota in strategies:
        picked = diversity_select(
            frame,
            n=quota,
            min_hamming=TOP6_MIN_HAMMING,
            already=chosen_seqs,
            already_rows=chosen_rows_meta,
        )
        if not picked.empty:
            picked = picked.copy()
            picked["Design_Strategy"] = label
            chosen.append(picked)
            chosen_seqs.extend(picked["Sequence"].tolist())
            chosen_rows_meta.extend([row for _, row in picked.iterrows()])

    out = pd.concat(chosen, ignore_index=True) if chosen else pd.DataFrame()

    if len(out) < 6:
        score_col = select_score_col(valid)
        fill = valid.sort_values(score_col, ascending=False)
        picked = diversity_select(
            fill,
            n=6 - len(out),
            min_hamming=TOP6_MIN_HAMMING,
            already=chosen_seqs,
            already_rows=chosen_rows_meta,
        )
        if not picked.empty:
            picked = picked.copy()
            picked["Design_Strategy"] = "fallback_best"
            out = pd.concat([out, picked], ignore_index=True)

    return out.drop_duplicates("Sequence").head(6).copy()

# ==========================================
# 预测 sfGFP WT 亮度，用于计算相对 WT 亮度比值
# ==========================================

def predict_brightness_for_sequences(seq_df, tag):
    seq_df = seq_df.copy().reset_index(drop=True)

    seq_df["saprot_input"] = seq_df["Sequence"].map(
        lambda s: stitch_saprot(s, WT_3DI["sfGFP"], strict=False)
    )

    mut_lists = [
        split_mutation_string(str(muts).replace(";", ":"))
        if "Mutations" in seq_df.columns else []
        for muts in seq_df.get("Mutations", pd.Series([""] * len(seq_df))).fillna("").astype(str)
    ]

    pocket_features = combined_pocket_feature_matrix(
        seq_df["Sequence"],
        ["sfGFP"] * len(seq_df),
        mut_lists,
    )

    esm = compute_esm_embeddings(
        seq_df["Sequence"].tolist(),
        RUN_OUTPUT_DIR / f"{tag}_esm2_embeddings.npy",
        ESM_BATCH_SIZE,
        force=RECOMPUTE_CANDIDATE_EMBEDDINGS,
    )

    sap = compute_saprot_embeddings(
        seq_df["saprot_input"].tolist(),
        RUN_OUTPUT_DIR / f"{tag}_saprot_embeddings.npy",
        SAPROT_BATCH_SIZE,
        force=RECOMPUTE_CANDIDATE_EMBEDDINGS,
    )

    X = build_model_feature_matrix(esm, sap, pocket_features, ["sfGFP"] * len(seq_df))

    seq_df = add_brightness_predictions(seq_df, X, esm, sap, pocket_features, families=["sfGFP"] * len(seq_df))

    return seq_df

wt_pred = predict_brightness_for_sequences(
    pd.DataFrame({
        "Seq_ID": ["sfGFP_WT"],
        "Sequence": [SF_GFP],
        "Mutations": ["WT"],
    }),
    "sfgfp_wt_for_ratio",
)

WT_XGB_BRIGHTNESS = float(wt_pred["XGBoost_Brightness"].iloc[0]) if "XGBoost_Brightness" in wt_pred.columns else np.nan
WT_XGB_RANK = float(wt_pred["XGBoost_RankScore"].iloc[0]) if "XGBoost_RankScore" in wt_pred.columns else np.nan
WT_GNN_RANK = float(wt_pred["GatingNN_RankScore"].iloc[0]) if "GatingNN_RankScore" in wt_pred.columns else np.nan
WT_BRIGHTNESS_RESCORE = float(wt_pred["Brightness_Rescore"].iloc[0])

print("sfGFP WT brightness:")
display(wt_pred[[
    c for c in [
        "Seq_ID",
        "XGBoost_Brightness",
        "XGBoost_RankScore",
        "GatingNN_RankScore",
        "Brightness_Rescore",
        "Brightness_Uncertainty",
        "Relative_Uncertainty",
    ]
    if c in wt_pred.columns
]])

X_cand = build_model_feature_matrix(cand_esm, cand_saprot, cand_pocket_features, ["sfGFP"] * len(cand))
cand = add_brightness_predictions(cand, X_cand, cand_esm, cand_saprot, cand_pocket_features, families=["sfGFP"] * len(cand))

# ==========================================
# 相对 sfGFP WT 的预测亮度比值
# ==========================================

if "XGBoost_Brightness" in cand.columns and not np.isnan(WT_XGB_BRIGHTNESS):
    cand["XGBoost_Raw_Relative_to_WT"] = cand["XGBoost_Brightness"] / max(WT_XGB_BRIGHTNESS, 1e-6)

if "XGBoost_RankScore" in cand.columns and not np.isnan(WT_XGB_RANK):
    cand["XGBoost_Rank_Relative_to_WT"] = cand["XGBoost_RankScore"] / max(WT_XGB_RANK, 1e-6)

if "GatingNN_RankScore" in cand.columns and not np.isnan(WT_GNN_RANK):
    cand["GatingNN_Rank_Relative_to_WT"] = cand["GatingNN_RankScore"] / max(WT_GNN_RANK, 1e-6)

cand["RankScore_Relative_to_WT"] = (
    cand["Brightness_Rescore"] / max(WT_BRIGHTNESS_RESCORE, 1e-6)
)

cand = apply_crossfamily_ensemble_scores(cand)

if "Stability_Proxy" not in cand.columns:
    cand["Stability_Proxy"] = 1.0

cand["_parsed_mutations"] = cand["Mutations"].map(parse_candidate_mutations)
thermo_pairs = cand["_parsed_mutations"].map(lambda muts: thermompnn_candidate_score(muts, thermo_ddg_map))
cand["ThermoMPNN_ddG_sum"] = thermo_pairs.map(lambda x: x[0])
cand["ThermoMPNN_Score"] = thermo_pairs.map(lambda x: x[1])

cand["Pass_RawRelative"] = cand["XGBoost_Raw_Relative_to_WT"].fillna(0.0) >= CONSENSUS_RAW_RELATIVE_MIN
cand["Pass_HeavyRank"] = cand["Brightness_Rescore"].fillna(0.0) >= CONSENSUS_RANK_SCORE_MIN
cand["Pass_LightModels"] = cand.get("Lightweight_Consensus", pd.Series(np.nan, index=cand.index)).fillna(0.0) >= CONSENSUS_LIGHT_SCORE_MIN
cand["Pass_ThermoMPNN"] = cand["ThermoMPNN_ddG_sum"].isna() | (cand["ThermoMPNN_ddG_sum"] <= CONSENSUS_THERMO_DDG_MAX)
cand["Pass_Uncertainty"] = cand["Relative_Uncertainty"].fillna(999.0) <= CONSENSUS_UNCERTAINTY_MAX
cand["Model_Consensus_Count"] = cand[[
    "Pass_RawRelative",
    "Pass_HeavyRank",
    "Pass_LightModels",
    "Pass_ThermoMPNN",
    "Pass_Uncertainty",
]].sum(axis=1)

# 最终排序分数：由验证集数据学习亮度排序；稳定性、WT-relative、突变数作为硬约束/解释列。
cand["Pocket_Overload"] = np.maximum(
    cand["Pocket_Mutation_Count"].fillna(0) - MAX_POCKET_MUTATIONS_PER_CANDIDATE,
    0,
)


def historical_pattern_score(mutation_string):
    """给候选序列一个历史优秀序列模式参考分。

    这个分数不是最终亮度模型的核心监督信号，只用于解释候选是否接近往年优秀序列的突变模式。
    为了保证 notebook 从任意位置重跑都不会报错：
    - 如果前面已经构建了 HISTORICAL_MUTATION_COUNTER / HISTORICAL_MUTATION_SET，就按重合度打分；
    - 如果没有历史模式库，就返回 0，表示不使用该项信息。
    """
    muts = parse_candidate_mutations(mutation_string) if isinstance(mutation_string, str) else []
    if not muts:
        return 0.0

    labels = [f"{wt}{pos0 + 1}{mut}" for wt, pos0, mut in muts]

    if "HISTORICAL_MUTATION_COUNTER" in globals():
        counter = HISTORICAL_MUTATION_COUNTER
        raw = sum(float(counter.get(label, 0.0)) for label in labels)
        norm = max(1.0, float(max(counter.values())) if len(counter) else 1.0)
        return float(raw / (len(labels) * norm))

    if "HISTORICAL_MUTATION_SET" in globals():
        hist_set = set(HISTORICAL_MUTATION_SET)
        return float(sum(label in hist_set for label in labels) / max(1, len(labels)))

    return 0.0

cand["HistoricalPatternScore"] = cand["Mutations"].map(historical_pattern_score)

# Data_Driven_Brightness_Score 在 add_brightness_predictions() 中由 final_brightness_meta_model 生成。
# ThermoMPNN 不再用任意系数混进分数，而是作为 hard filter 和 tie-break 进入最终选择。
cand["Final_Learned_Score"] = cand["Data_Driven_Brightness_Score"].fillna(cand["Brightness_Rescore"])
cand["Final_Rescore"] = cand["Final_Learned_Score"]  # 兼容旧输出列名
if "CrossFamily_EnsembleScore" in cand.columns:
    cand["Final_Learned_Score_Original"] = cand["Final_Learned_Score"]
    cand["Final_Learned_Score"] = cand["CrossFamily_EnsembleScore"].fillna(cand["Final_Learned_Score"])
    cand["Final_Rescore"] = cand["Final_Learned_Score"]
cand["Check_Errors"] = cand["Sequence"].map(lambda s: ";".join(check_sequence(s)))

exclusion_set = load_exclusion_set(EXCLUSION_CSV)
cand["In_Exclusion"] = cand["Sequence"].isin(exclusion_set)

# 输出 pocket 单点饱和扫描结果，方便人工查看哪些邻域单点突变最值得关注。
# 对第 9 单元生成的所有 pocket 单点扫描补充 WT-relative 指标，作为 pseudo-sfGFP 局部扫描表。
if "single_df" in globals() and not single_df.empty:
    pseudo_single_scan = single_df.copy()
    if "XGBoost_Brightness" in pseudo_single_scan.columns and not np.isnan(WT_XGB_BRIGHTNESS):
        pseudo_single_scan["XGBoost_Raw_Relative_to_WT"] = (
            pseudo_single_scan["XGBoost_Brightness"] / max(WT_XGB_BRIGHTNESS, 1e-6)
        )
    if "Brightness_Rescore" in pseudo_single_scan.columns:
        pseudo_single_scan["RankScore_Relative_to_WT"] = (
            pseudo_single_scan["Brightness_Rescore"] / max(WT_BRIGHTNESS_RESCORE, 1e-6)
        )
    pseudo_single_scan = pseudo_single_scan.sort_values(
        ["XGBoost_Raw_Relative_to_WT", "Brightness_Rescore"],
        ascending=False,
    )
    pseudo_single_scan.to_csv(SINGLE_MUTANT_SCAN_OUT, index=False)
    print("pseudo-sfGFP single mutant scan saved:", SINGLE_MUTANT_SCAN_OUT, pseudo_single_scan.shape)
    display(pseudo_single_scan[[
        "Mutations",
        "XGBoost_Brightness",
        "XGBoost_Raw_Relative_to_WT",
        "XGBoost_RankScore",
        "GatingNN_RankScore",
        "Brightness_Rescore",
        "RankScore_Relative_to_WT",
        "Brightness_Uncertainty",
        "Pre_Thermo_ddG_sum",
    ]].head(30))
else:
    print("single_df 不存在，跳过 pseudo-sfGFP 单点扫描相对 WT 输出。")

valid = cand[(cand["Check_Errors"] == "") & (~cand["In_Exclusion"])].copy()

# ThermoMPNN hard filter：明显不稳定的组合不进入最终 top6。
# 如果某条候选没有可映射 ddG，则不因为缺失值直接丢弃。
before_filter = len(valid)
valid = valid[
    valid["ThermoMPNN_ddG_sum"].isna()
    | (valid["ThermoMPNN_ddG_sum"] <= THERMO_DDG_HARD_MAX)
].copy()
print("ThermoMPNN hard filter:", before_filter, "->", len(valid), "threshold:", THERMO_DDG_HARD_MAX)

# 严格使用 sfGFP WT 作锚点：优先要求 raw brightness 超过 WT，同时限制不确定性和 rank 太低的候选。
strict_valid = valid[
    (valid["XGBoost_Raw_Relative_to_WT"].fillna(0.0) >= MIN_XGB_RAW_RELATIVE_TO_WT)
    & (valid["Brightness_Rescore"].fillna(0.0) >= MIN_BRIGHTNESS_RESCORE_FOR_TOP)
    & (valid["Relative_Uncertainty"].fillna(999.0) <= MAX_RELATIVE_UNCERTAINTY_FOR_TOP)
    & (valid["Model_Consensus_Count"].fillna(0) >= MIN_MODEL_CONSENSUS_COUNT)
].copy()

if len(strict_valid) >= 6:
    valid = strict_valid
    print("WT-anchor + model consensus filter kept:", len(valid))
else:
    consensus_valid = valid[valid["Model_Consensus_Count"].fillna(0) >= MIN_MODEL_CONSENSUS_COUNT].copy()
    if len(consensus_valid) >= 6:
        valid = consensus_valid
        print("strict filter too few; using model consensus filter:", len(valid))
    else:
        print(
            "strict/consensus filter candidates too few:",
            len(strict_valid),
            len(consensus_valid),
            "-> 放宽为 Final_Rescore 排序，但保留共识列供人工判断。",
        )

valid = valid.sort_values(
    [select_score_col(valid), "XGBoost_Raw_Relative_to_WT", "ThermoMPNN_ddG_sum", "Relative_Uncertainty"],
    ascending=[False, False, True, True],
)

# 多目标 Pareto 诊断：不直接替代排序，但标出同时兼顾亮度、相对 WT、稳定性和低不确定性的候选。
def mark_pareto_efficient(frame):
    if frame.empty:
        return pd.Series([], dtype=bool, index=frame.index)
    score = frame[select_score_col(frame)].fillna(0.0).values
    raw = frame["XGBoost_Raw_Relative_to_WT"].fillna(0.0).values
    ddg = frame["ThermoMPNN_ddG_sum"].fillna(0.0).values
    uncertainty = frame["Relative_Uncertainty"].fillna(999.0).values

    efficient = np.ones(len(frame), dtype=bool)
    for i in range(len(frame)):
        dominated = (
            (score >= score[i])
            & (raw >= raw[i])
            & (ddg <= ddg[i])
            & (uncertainty <= uncertainty[i])
            & ((score > score[i]) | (raw > raw[i]) | (ddg < ddg[i]) | (uncertainty < uncertainty[i]))
        )
        dominated[i] = False
        if dominated.any():
            efficient[i] = False
    return pd.Series(efficient, index=frame.index)

valid["Pareto_Efficient"] = mark_pareto_efficient(valid)
valid.to_csv(CANDIDATE_SCORED_OUT, index=False)
print("valid candidates:", valid.shape)
print("scored candidates saved:", CANDIDATE_SCORED_OUT)

top6 = pick_strategy_top6(valid)
top6 = top6.sort_values(
    [select_score_col(top6), "XGBoost_Raw_Relative_to_WT", "ThermoMPNN_ddG_sum"],
    ascending=[False, False, True],
).reset_index(drop=True)
top6["Team_Name"] = "GFP_design_team"
top6["Seq_ID"] = [str(i + 1) for i in range(len(top6))]

detailed_cols = [
    "Team_Name", "Seq_ID", "Design_Strategy", "Sequence", "Mutations", "Candidate_Source",
    "Pocket_Mutation_Count", "Scaffold_Mutation_Count",
    "XGBoost_Brightness",
    "XGBoost_RankScore", "GatingNN_RankScore", "Kmer_RankScore", "Position_RankScore",
    "Heavy_Rank_Ensemble", "Lightweight_Consensus",
    "Raw_Brightness_Percentile", "Model_Agreement", "Data_Driven_Brightness_Score",
    "Brightness_Rescore", "RankScore_Relative_to_WT",
    "XGBoost_Raw_Relative_to_WT",
    "XGBoost_Rank_Relative_to_WT", "GatingNN_Rank_Relative_to_WT",
    "Brightness_Uncertainty", "Relative_Uncertainty",
    "Stability_Proxy", "ThermoMPNN_ddG_sum", "ThermoMPNN_Score", "Pocket_Overload",
    "Pass_RawRelative", "Pass_HeavyRank", "Pass_LightModels", "Pass_ThermoMPNN", "Pass_Uncertainty",
    "Model_Consensus_Count", "Pareto_Efficient",
    "HistoricalPatternScore", "Final_Learned_Score", "Final_Rescore", "Check_Errors",
]
top6[detailed_cols].to_csv(DETAILED_TOP6_OUT, index=False)

# 比赛提交文件严格只保留三列：Team_Name, Seq_ID, Sequence。
submission_cols = ["Team_Name", "Seq_ID", "Sequence"]
top6[submission_cols].to_csv(SUBMISSION_TOP6_OUT, index=False)

display(top6[detailed_cols])
print("detailed top6 saved:", DETAILED_TOP6_OUT)
print("competition submission saved:", SUBMISSION_TOP6_OUT)


使用 embedding 缓存: /home/liuchang/PythonProject2/server_full_recompute_outputs/sfgfp_wt_for_ratio_esm2_embeddings.npy shape=(1, 1280)
使用 embedding 缓存: /home/liuchang/PythonProject2/server_full_recompute_outputs/sfgfp_wt_for_ratio_saprot_embeddings.npy shape=(1, 1280)
sfGFP WT brightness:


,Seq_ID,XGBoost_Brightness,XGBoost_RankScore,GatingNN_RankScore,Brightness_Rescore,Brightness_Uncertainty,Relative_Uncertainty
0,sfGFP_WT,3.717099,0.757031,1.0,0.96605,0.0882,0.0913


pseudo-sfGFP single mutant scan saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/sfgfp_pocket_single_mutant_scan.csv (324, 25)


,Mutations,XGBoost_Brightness,XGBoost_Raw_Relative_to_WT,XGBoost_RankScore,GatingNN_RankScore,Brightness_Rescore,RankScore_Relative_to_WT,Brightness_Uncertainty,Pre_Thermo_ddG_sum
68,L44I,3.767434,1.013541,0.665945,0.642676,0.753399,0.779875,0.057359,0.546412
5,T63S,3.758860,1.011235,0.711420,0.864189,0.899386,0.930993,0.066940,0.546280
32,V150I,3.736005,1.005086,0.666887,0.770197,0.815124,0.843769,0.047743,-0.226828
97,S147Q,3.731287,1.003817,0.649136,0.626220,0.723868,0.749307,0.078862,0.430033
109,V150A,3.725562,1.002277,0.574935,0.646188,0.708896,0.733808,0.066507,1.654604
100,L42I,3.711435,0.998476,0.630430,0.642695,0.717589,0.742807,0.048185,-0.402409
59,L221M,3.676313,0.989027,0.657694,0.673860,0.771751,0.798873,0.062045,0.108028
89,Q204N,3.636245,0.978248,0.616211,0.643843,0.733522,0.759300,0.055781,0.450506
246,S147D,3.623671,0.974865,0.604406,0.256547,0.496475,0.513922,0.198614,0.419195
25,L64M,3.615014,0.972536,0.648284,0.735279,0.841478,0.871050,0.081018,0.696416


ThermoMPNN hard filter: 1000 -> 955 threshold: 0.0
WT-anchor + model consensus filter kept: 23
valid candidates: (23, 48)
scored candidates saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/recomputed_candidate_scores.csv


,Team_Name,Seq_ID,Design_Strategy,Sequence,Mutations,Candidate_Source,Pocket_Mutation_Count,Scaffold_Mutation_Count,XGBoost_Brightness,XGBoost_RankScore,...,Pass_HeavyRank,Pass_LightModels,Pass_ThermoMPNN,Pass_Uncertainty,Model_Consensus_Count,Pareto_Efficient,HistoricalPatternScore,Final_Learned_Score,Final_Rescore,Check_Errors
0,GFP_design_team,1,learned_score_best,MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKF...,T63S;K166T;E172L;S205T,ga_pocket_scaffold_wt_anchor,2,2,3.754096,0.759917,...,True,True,True,True,5,True,0.0,0.894219,0.894219,
1,GFP_design_team,2,learned_score_best,MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKF...,T62S;K166T;E172I;Q204R,ga_pocket_scaffold_wt_anchor,2,2,3.974350,0.736295,...,True,True,True,True,5,True,0.0,0.886349,0.886349,
2,GFP_design_team,3,raw_brightness_first,MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKF...,T62S;T63S;E115V;E172I,ga_pocket_scaffold_wt_anchor,2,2,3.811966,0.778029,...,True,True,True,True,5,False,0.0,0.882500,0.882500,
3,GFP_design_team,4,non_H148Y_diversity,MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKF...,T62S;T63S;K166T;E172I;S202T,ga_pocket_scaffold_wt_anchor,2,3,3.742553,0.683101,...,True,True,True,True,5,False,0.0,0.877576,0.877576,
4,GFP_design_team,5,raw_brightness_first,MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKF...,E172I;T203V;Q204L;F223Y,ga_pocket_scaffold_wt_anchor,2,2,3.876508,0.652510,...,True,True,True,True,5,True,0.0,0.810230,0.810230,
5,GFP_design_team,6,low_uncertainty_balance,MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKF...,T62S;E115V;E172I;S205T,ga_pocket_scaffold_wt_anchor,2,2,3.742622,0.740123,...,True,True,True,True,5,False,0.0,0.806854,0.806854,


detailed top6 saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/recomputed_top6_detailed_report.csv
competition submission saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/recomputed_submission_top6.csv


In [13]:
# 导出严格使用的 sfGFP 参考序列
STRICT_SFGFP_FASTA = PROJECT_DIR / "strict_sfgfp_238.fasta"

with open(STRICT_SFGFP_FASTA, "w") as f:
    f.write(">strict_sfGFP_238\n")
    f.write(SF_GFP + "\n")

print("SF_GFP length:", len(SF_GFP))
print("saved:", STRICT_SFGFP_FASTA)
print(SF_GFP)

SF_GFP length: 238
saved: /home/liuchang/PythonProject2/strict_sfgfp_238.fasta
MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKRHDFFKSAMPEGYVQERTISFKDDGTYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNFNSHNVYITADKQKNGIKANFKIRHNVEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSVLSKDPNEKRDHMVLLEFVTAAGITHGMDELYK


In [14]:
# ==========================================
# 往年优秀 avGFP-like 序列亮度预测表
# ==========================================

def closest_gfp_family(seq, refs):
    seq = str(seq).strip().upper()
    best_family = None
    best_dist = None

    for fam, ref_seq in refs.items():
        fam = normalize_family_name(fam)
        n = min(len(seq), len(ref_seq))
        mismatches = sum(a != b for a, b in zip(seq[:n], ref_seq[:n]))
        dist = mismatches + abs(len(seq) - len(ref_seq))

        if best_dist is None or dist < best_dist:
            best_dist = dist
            best_family = fam

    return best_family, best_dist

def mutation_labels_vs_ref(seq, ref_seq):
    muts = []
    for i, (wt, aa) in enumerate(zip(ref_seq, seq), start=1):
        if wt != aa:
            muts.append(f"{wt}{i}{aa}")
    return ";".join(muts) if muts else "WT"

def predict_brightness_table(seq_df, family, wt_seq, tag):
    seq_df = seq_df.copy().reset_index(drop=True)

    seq_df["saprot_input"] = seq_df["Sequence"].map(
        lambda s: stitch_saprot(s, WT_3DI[family], strict=False)
    )

    mut_lists = [
        split_mutation_string(m.replace(";", ":")) if m != "WT" else []
        for m in seq_df["Mutations_vs_WT"].fillna("WT").astype(str)
    ]

    pocket_features = combined_pocket_feature_matrix(
        seq_df["Sequence"],
        [family] * len(seq_df),
        mut_lists,
    )

    esm = compute_esm_embeddings(
        seq_df["Sequence"].tolist(),
        RUN_OUTPUT_DIR / f"{tag}_esm2_embeddings.npy",
        ESM_BATCH_SIZE,
        force=RECOMPUTE_CANDIDATE_EMBEDDINGS,
    )

    sap = compute_saprot_embeddings(
        seq_df["saprot_input"].tolist(),
        RUN_OUTPUT_DIR / f"{tag}_saprot_embeddings.npy",
        SAPROT_BATCH_SIZE,
        force=RECOMPUTE_CANDIDATE_EMBEDDINGS,
    )

    X = build_model_feature_matrix(esm, sap, pocket_features, [family] * len(seq_df))

    seq_df = add_brightness_predictions(seq_df, X, esm, sap, pocket_features, families=[family] * len(seq_df))

    return seq_df

# 1. 读取 beforetopseqs
before_top = pd.read_excel(GFP_XLSX, sheet_name="beforetopseqs")

seq_col = None
for c in ["sequence", "Sequence", "seq", "AA_sequence", "aa_sequence"]:
    if c in before_top.columns:
        seq_col = c
        break

if seq_col is None:
    raise ValueError(f"找不到序列列，当前列名: {list(before_top.columns)}")

before_top = before_top.copy()
before_top["Sequence"] = before_top[seq_col].astype(str).str.strip().str.upper()
before_top = before_top[before_top["Sequence"].map(lambda s: set(s).issubset(STANDARD_AA))]
before_top = before_top[before_top["Sequence"].map(lambda s: 220 <= len(s) <= 250)].reset_index(drop=True)

# 2. 自动判断最接近哪个 GFP family，只保留 avGFP-like
family_info = before_top["Sequence"].map(lambda s: closest_gfp_family(s, refs))
before_top["Closest_Family"] = family_info.map(lambda x: x[0])
before_top["Family_Distance"] = family_info.map(lambda x: x[1])

avgfp_top = before_top[before_top["Closest_Family"] == "avGFP"].copy().reset_index(drop=True)

print("avGFP-like beforetopseqs:", avgfp_top.shape)

# 3. 计算相对 avGFP WT 的突变
AV_GFP = refs["avGFP"]

avgfp_top["Mutations_vs_WT"] = avgfp_top["Sequence"].map(
    lambda s: mutation_labels_vs_ref(s, AV_GFP)
)

avgfp_top["Mutation_Count"] = avgfp_top["Mutations_vs_WT"].map(
    lambda x: 0 if x == "WT" else len(x.split(";"))
)

# 4. 预测 avGFP WT 亮度
avgfp_wt_df = pd.DataFrame({
    "year": ["WT"],
    "Sequence": [AV_GFP],
    "Closest_Family": ["avGFP"],
    "Family_Distance": [0],
    "Mutations_vs_WT": ["WT"],
    "Mutation_Count": [0],
})

avgfp_wt_pred = predict_brightness_table(
    avgfp_wt_df,
    family="avGFP",
    wt_seq=AV_GFP,
    tag="avgfp_wt_for_ratio",
)

WT_AVGFP_BRIGHTNESS = float(avgfp_wt_pred["Brightness_Rescore"].iloc[0])
WT_AVGFP_XGB = float(avgfp_wt_pred["XGBoost_Brightness"].iloc[0])
WT_AVGFP_XGB_RANK = float(avgfp_wt_pred["XGBoost_RankScore"].iloc[0]) if "XGBoost_RankScore" in avgfp_wt_pred.columns else np.nan
WT_AVGFP_GNN_RANK = float(avgfp_wt_pred["GatingNN_RankScore"].iloc[0]) if "GatingNN_RankScore" in avgfp_wt_pred.columns else np.nan

print("avGFP WT prediction:")
display(avgfp_wt_pred[[
    c for c in [
        "XGBoost_Brightness",
        "XGBoost_RankScore",
        "GatingNN_RankScore",
        "Kmer_RankScore",
        "Position_RankScore",
        "Heavy_Rank_Ensemble",
        "Lightweight_Consensus",
        "Brightness_Rescore",
        "Data_Driven_Brightness_Score",
        "Brightness_Uncertainty",
        "Relative_Uncertainty",
    ]
    if c in avgfp_wt_pred.columns
]])

# 5. 预测 avGFP-like 往年 Top 序列
avgfp_result = predict_brightness_table(
    avgfp_top,
    family="avGFP",
    wt_seq=AV_GFP,
    tag="beforetopseqs_avgfp_like",
)

# 6. 计算相对 avGFP WT 的亮度比值
avgfp_result["RankScore_Relative_to_avGFP_WT"] = (
    avgfp_result["Brightness_Rescore"] / max(WT_AVGFP_BRIGHTNESS, 1e-6)
)

avgfp_result["XGBoost_Raw_Relative_to_avGFP_WT"] = (
    avgfp_result["XGBoost_Brightness"] / max(WT_AVGFP_XGB, 1e-6)
)
if "XGBoost_RankScore" in avgfp_result.columns and not np.isnan(WT_AVGFP_XGB_RANK):
    avgfp_result["XGBoost_Rank_Relative_to_avGFP_WT"] = (
        avgfp_result["XGBoost_RankScore"] / max(WT_AVGFP_XGB_RANK, 1e-6)
    )

if "GatingNN_RankScore" in avgfp_result.columns and not np.isnan(WT_AVGFP_GNN_RANK):
    avgfp_result["GatingNN_Rank_Relative_to_avGFP_WT"] = (
        avgfp_result["GatingNN_RankScore"] / max(WT_AVGFP_GNN_RANK, 1e-6)
    )

# 7. 整理输出表
out_cols = [
    c for c in [
        "year",
        "Closest_Family",
        "Family_Distance",
        "Sequence",
        "Mutations_vs_WT",
        "Mutation_Count",
        "XGBoost_Brightness",
        "XGBoost_RankScore",
        "GatingNN_RankScore",
        "Kmer_RankScore",
        "Position_RankScore",
        "Heavy_Rank_Ensemble",
        "Lightweight_Consensus",
        "Brightness_Rescore",
        "Data_Driven_Brightness_Score",
        "Final_Learned_Score",
        "Raw_Brightness_Percentile",
        "Model_Agreement",
        "RankScore_Relative_to_avGFP_WT",
        "XGBoost_Raw_Relative_to_avGFP_WT",
        "GatingNN_Rank_Relative_to_avGFP_WT",
        "Brightness_Uncertainty",
        "Relative_Uncertainty",
    ]
    if c in avgfp_result.columns
]

# 兼容 final meta model 没有生成 Final_Learned_Score 的情况。
# 先补齐排序列，再排序，最后裁剪输出列，避免 KeyError。
if "Final_Learned_Score" not in avgfp_result.columns:
    if "Data_Driven_Brightness_Score" in avgfp_result.columns:
        avgfp_result["Final_Learned_Score"] = avgfp_result["Data_Driven_Brightness_Score"]
    elif "Brightness_Rescore" in avgfp_result.columns:
        avgfp_result["Final_Learned_Score"] = avgfp_result["Brightness_Rescore"]
    elif "Heavy_Rank_Ensemble" in avgfp_result.columns:
        avgfp_result["Final_Learned_Score"] = avgfp_result["Heavy_Rank_Ensemble"]
    else:
        avgfp_result["Final_Learned_Score"] = 0.0

if "Final_Rescore" not in avgfp_result.columns:
    avgfp_result["Final_Rescore"] = avgfp_result["Final_Learned_Score"]

sort_col = select_score_col(avgfp_result) if "select_score_col" in globals() else "Final_Learned_Score"
if sort_col not in out_cols:
    out_cols.append(sort_col)
if "Final_Rescore" in avgfp_result.columns and "Final_Rescore" not in out_cols:
    out_cols.append("Final_Rescore")

avgfp_result = avgfp_result.sort_values(
    sort_col,
    ascending=False,
).reset_index(drop=True)
avgfp_result = avgfp_result[out_cols]

out_path = RUN_OUTPUT_DIR / "beforetopseqs_avGFP_like_brightness_predictions.csv"
avgfp_result.to_csv(out_path, index=False)

display(avgfp_result)
print("saved:", out_path)

avGFP-like beforetopseqs: (20, 5)
使用 embedding 缓存: /home/liuchang/PythonProject2/server_full_recompute_outputs/avgfp_wt_for_ratio_esm2_embeddings.npy shape=(1, 1280)
使用 embedding 缓存: /home/liuchang/PythonProject2/server_full_recompute_outputs/avgfp_wt_for_ratio_saprot_embeddings.npy shape=(1, 1280)
avGFP WT prediction:


,XGBoost_Brightness,XGBoost_RankScore,GatingNN_RankScore,Kmer_RankScore,Position_RankScore,Heavy_Rank_Ensemble,Lightweight_Consensus,Brightness_Rescore,Data_Driven_Brightness_Score,Brightness_Uncertainty,Relative_Uncertainty
0,3.682683,0.841086,1.0,0.978388,0.747185,1.0,0.862786,1.0,0.969554,0.103308,0.103308


使用 embedding 缓存: /home/liuchang/PythonProject2/server_full_recompute_outputs/beforetopseqs_avgfp_like_esm2_embeddings.npy shape=(20, 1280)
使用 embedding 缓存: /home/liuchang/PythonProject2/server_full_recompute_outputs/beforetopseqs_avgfp_like_saprot_embeddings.npy shape=(20, 1280)


,year,Closest_Family,Family_Distance,Sequence,Mutations_vs_WT,Mutation_Count,XGBoost_Brightness,XGBoost_RankScore,GatingNN_RankScore,Kmer_RankScore,...,Data_Driven_Brightness_Score,Final_Learned_Score,Raw_Brightness_Percentile,Model_Agreement,RankScore_Relative_to_avGFP_WT,XGBoost_Raw_Relative_to_avGFP_WT,GatingNN_Rank_Relative_to_avGFP_WT,Brightness_Uncertainty,Relative_Uncertainty,Final_Rescore
0,2024,avGFP,2,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,V68I;T203I,2,3.641551,0.888561,0.879386,0.932280,...,0.904840,0.904840,0.491179,0.819868,0.962633,0.988831,0.879386,0.078000,0.081027,0.904840
1,2025,avGFP,4,MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKF...,S30R;Y39N;S65T;I171V,4,3.231821,0.655912,1.000000,0.746792,...,0.876581,0.876581,0.332434,0.690541,0.913205,0.877572,1.000000,0.134000,0.146736,0.876581
2,2025,avGFP,5,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDASYGKLTLKF...,T38S;S65T;R73H;Q80R;N105T,5,2.710151,0.502315,1.000000,0.619706,...,0.806562,0.806562,0.177223,0.572413,0.836045,0.735918,1.000000,0.185151,0.221460,0.806562
3,2025,avGFP,2,MSKGEELFTGVVPILVELDGDVNGHKFSVQGEGEGDATYGKLTLKF...,S30Q;S65T,2,3.601747,0.724608,1.000000,0.599476,...,0.800848,0.800848,0.465000,0.629436,0.823763,0.978023,1.000000,0.160459,0.194788,0.800848
4,2024,avGFP,5,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,N105S;K158G;V163A;I171V;T203Y,5,3.000244,0.502435,0.799003,0.766705,...,0.792440,0.792440,0.302245,0.726270,0.819993,0.814690,0.799003,0.118529,0.144548,0.792440
5,2024,avGFP,4,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,L64F;S72A;Q80R;T203I,4,2.968066,0.585407,0.712313,0.773884,...,0.786703,0.786703,0.299134,0.767168,0.778935,0.805952,0.712313,0.100819,0.129432,0.786703
6,2025,avGFP,1,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,Q69M,1,3.256315,0.601098,1.000000,0.759944,...,0.752508,0.752508,0.337450,0.480200,0.921190,0.884224,1.000000,0.225080,0.244336,0.752508
7,2024,avGFP,2,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,S65T;V163A,2,3.621392,0.756637,0.688340,0.711310,...,0.750233,0.750233,0.477215,0.942003,0.728385,0.983357,0.688340,0.025113,0.034478,0.750233
8,2025,avGFP,6,MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATYGKLTLKF...,S30R;S65C;S72A;K79R;N105K;N149K,6,2.539021,0.408556,0.617494,0.750507,...,0.710989,0.710989,0.153290,0.504472,0.715067,0.689449,0.617494,0.214570,0.300069,0.710989
9,2024,avGFP,4,MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKF...,L64F;S65G;S72A;Q80R,4,2.687916,0.498200,0.658725,0.590935,...,0.690904,0.690904,0.172250,0.668978,0.639785,0.729880,0.658725,0.143337,0.224039,0.690904


saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/beforetopseqs_avGFP_like_brightness_predictions.csv


In [15]:
# ==========================================
# 结果可信度审计：LOFO 泛化 + 往年优秀序列 + 最终 top6
# ==========================================

summary_rows = []

if "lofo_diagnostic_df" in globals() and isinstance(lofo_diagnostic_df, pd.DataFrame) and not lofo_diagnostic_df.empty:
    summary_rows.extend([
        {"metric": "LOFO_xgb_raw_r2_mean", "value": float(lofo_diagnostic_df["xgb_raw_r2"].mean())},
        {"metric": "LOFO_xgb_raw_r2_min", "value": float(lofo_diagnostic_df["xgb_raw_r2"].min())},
        {"metric": "LOFO_xgb_rank_r2_mean", "value": float(lofo_diagnostic_df["xgb_rank_r2"].mean())},
        {"metric": "LOFO_xgb_rank_r2_min", "value": float(lofo_diagnostic_df["xgb_rank_r2"].min())},
    ])
else:
    summary_rows.append({"metric": "LOFO_diagnostic_available", "value": 0.0})

if "avgfp_result" in globals() and isinstance(avgfp_result, pd.DataFrame) and not avgfp_result.empty:
    summary_rows.extend([
        {"metric": "historical_top_count", "value": float(len(avgfp_result))},
        {"metric": "historical_final_score_mean", "value": float(avgfp_result["Final_Learned_Score"].mean()) if "Final_Learned_Score" in avgfp_result else np.nan},
        {"metric": "historical_final_score_max", "value": float(avgfp_result["Final_Learned_Score"].max()) if "Final_Learned_Score" in avgfp_result else np.nan},
        {"metric": "historical_raw_relative_gt_1_count", "value": float((avgfp_result.get("XGBoost_Raw_Relative_to_avGFP_WT", pd.Series([], dtype=float)) > 1.0).sum())},
    ])
else:
    summary_rows.append({"metric": "historical_top_prediction_available", "value": 0.0})

if "top6" in globals() and isinstance(top6, pd.DataFrame) and not top6.empty:
    summary_rows.extend([
        {"metric": "top6_count", "value": float(len(top6))},
        {"metric": "top6_raw_relative_mean", "value": float(top6["XGBoost_Raw_Relative_to_WT"].mean())},
        {"metric": "top6_raw_relative_min", "value": float(top6["XGBoost_Raw_Relative_to_WT"].min())},
        {"metric": "top6_ddg_sum_mean", "value": float(top6["ThermoMPNN_ddG_sum"].mean())},
        {"metric": "top6_ddg_sum_max", "value": float(top6["ThermoMPNN_ddG_sum"].max())},
        {"metric": "top6_relative_uncertainty_mean", "value": float(top6["Relative_Uncertainty"].mean())},
        {"metric": "top6_model_consensus_mean", "value": float(top6["Model_Consensus_Count"].mean())},
        {"metric": "top6_pareto_efficient_count", "value": float(top6.get("Pareto_Efficient", pd.Series(False, index=top6.index)).sum())},
    ])
else:
    summary_rows.append({"metric": "top6_available", "value": 0.0})


if "lofo_submodel_summary" in globals() and isinstance(lofo_submodel_summary, pd.DataFrame) and not lofo_submodel_summary.empty:
    best_lofo = lofo_submodel_summary.sort_values("cross_family_score", ascending=False).iloc[0]
    summary_rows.extend([
        {"metric": "LOFO_best_model_by_cross_family_score", "value": best_lofo["model_name"]},
        {"metric": "LOFO_best_cross_family_score", "value": float(best_lofo["cross_family_score"])},
        {"metric": "LOFO_best_mean_spearman", "value": float(best_lofo["mean_spearman"])},
        {"metric": "LOFO_best_mean_top10_recall", "value": float(best_lofo["mean_top10_recall"])},
    ])
if "crossfamily_model_weights" in globals() and isinstance(crossfamily_model_weights, pd.DataFrame) and not crossfamily_model_weights.empty:
    for _, row in crossfamily_model_weights.iterrows():
        summary_rows.append({
            "metric": "CrossFamily_weight_" + str(row["model_name"]),
            "value": float(row["weight"]),
        })

model_trust_audit = pd.DataFrame(summary_rows)
model_trust_audit.to_csv(MODEL_TRUST_AUDIT_OUT, index=False)
display(model_trust_audit)
print("model trust audit saved:", MODEL_TRUST_AUDIT_OUT)


,metric,value
0,LOFO_xgb_raw_r2_mean,-2.447922
1,LOFO_xgb_raw_r2_min,-6.873715
2,LOFO_xgb_rank_r2_mean,-0.1299
3,LOFO_xgb_rank_r2_min,-0.745411
4,historical_top_count,20.0
5,historical_final_score_mean,0.653718
6,historical_final_score_max,0.90484
7,historical_raw_relative_gt_1_count,0.0
8,top6_count,6.0
9,top6_raw_relative_mean,1.02688


model trust audit saved: /home/liuchang/PythonProject2/server_full_recompute_outputs/model_trust_audit_summary.csv
